# HybridRAG OpenAI: PDF-to-LLM Extraction + VectorRAG + GraphRAG for Open WebUI

This notebook builds per-paper **VectorRAG** and **GraphRAG** bundles from PDFs and prepares corpus-level retrieval artifacts using **OpenAI models** for LLM generation and embeddings.

Key features in this OpenAI version:

- local/open-source PDF extraction via `pdf_to_llm_open_source.py`;
- LLM-friendly extraction packs with equation/table/figure candidates and false-positive/false-negative audit tasks;
- OpenAI-powered profile extraction, knowledge graph extraction, optional extraction audit, visual figure descriptions, answer synthesis, and embeddings;
- default high-quality models: `gpt-5.2` for generation/answering/auditing and `text-embedding-3-large` for vector embeddings;
- optimized rebuild behavior with extraction reuse, embedding cache, resumable KG cache, and Chroma reuse;
- per-paper VectorRAG folders, per-paper GraphRAG folders, and corpus-level comparison artifacts;
- production-style retrieval: route to papers first, retrieve per-paper evidence, then add corpus-level comparison context.

Set your key before running:

```bash
export OPENAI_API_KEY="sk-..."
```

or set it in your notebook environment before executing the OpenAI helper cell.


## 1. Install dependencies

In [1]:
# Uncomment and run once if needed.
# %pip install -U openai pymupdf pdfplumber pdfminer.six chromadb networkx pyvis pandas requests tqdm rapidfuzz pillow numpy pypdf pypdfium2 lxml

# Optional OCR support for scanned PDFs. You also need the Tesseract executable installed on the OS.
# %pip install -U pytesseract

# Optional local-only equation OCR if you decide to use pix2tex later. It is not required for normal RAG generation.
# %pip install -U pix2tex

# OpenAI API key setup:
# In a terminal before launching Jupyter:
#   export OPENAI_API_KEY="sk-..."
# Or, inside the notebook before the OpenAI helper cell:
#   import os
#   os.environ["OPENAI_API_KEY"] = "sk-..."


## 2. Configuration

In [ ]:
from pathlib import Path
import os

# ----------------------------
# User paths
# ----------------------------
PDF_DIR = Path(os.environ.get("PDF_DIR", "myPDFs"))
# Use a separate default output folder so OpenAI embeddings do not mix with older Ollama/local vector stores.
OUT_DIR = Path(os.environ.get("OUT_DIR", "rag_output_openai"))

# ----------------------------
# OpenAI API configuration
# ----------------------------
# Recommended: set OPENAI_API_KEY in your shell before starting Jupyter.
# Example:
#   export OPENAI_API_KEY="sk-..."
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL") or None

# GPT-5.2 is the default high-quality LLM. You can override with environment variables.
GENERATION_MODEL = os.environ.get("OPENAI_GENERATION_MODEL", "gpt-5.2")
ANSWER_MODEL = os.environ.get("OPENAI_ANSWER_MODEL", GENERATION_MODEL)
EXTRACTION_AUDIT_MODEL = os.environ.get("OPENAI_EXTRACTION_AUDIT_MODEL", GENERATION_MODEL)
VISION_MODEL = os.environ.get("OPENAI_VISION_MODEL", GENERATION_MODEL)

# text-embedding-3-large is the default high-quality embedding model.
EMBEDDING_MODEL = os.environ.get("OPENAI_EMBEDDING_MODEL", "text-embedding-3-large")
# Leave unset/0 for the model default dimensions. For text-embedding-3-large, you may optionally set a smaller dimension.
_OPENAI_EMBEDDING_DIMENSIONS_RAW = os.environ.get("OPENAI_EMBEDDING_DIMENSIONS", "").strip()
EMBEDDING_DIMENSIONS = int(_OPENAI_EMBEDDING_DIMENSIONS_RAW) if _OPENAI_EMBEDDING_DIMENSIONS_RAW else None

# Reasoning and verbosity controls for GPT-5.2 Responses API.
# For speed, use OPENAI_REASONING_EFFORT=none or low.
# For higher-quality KG/profile extraction, use medium/high/xhigh.
OPENAI_REASONING_EFFORT = os.environ.get("OPENAI_REASONING_EFFORT", "high").strip().lower()
if OPENAI_REASONING_EFFORT not in {"none", "low", "medium", "high", "xhigh"}:
    raise ValueError("OPENAI_REASONING_EFFORT must be one of: none, low, medium, high, xhigh")
OPENAI_TEXT_VERBOSITY = os.environ.get("OPENAI_TEXT_VERBOSITY", "low").strip().lower()
if OPENAI_TEXT_VERBOSITY not in {"low", "medium", "high"}:
    raise ValueError("OPENAI_TEXT_VERBOSITY must be one of: low, medium, high")

OPENAI_MAX_RETRIES = int(os.environ.get("OPENAI_MAX_RETRIES", "5"))
OPENAI_RETRY_BASE_SECONDS = float(os.environ.get("OPENAI_RETRY_BASE_SECONDS", "2.0"))
OPENAI_REQUEST_TIMEOUT = float(os.environ.get("OPENAI_REQUEST_TIMEOUT", "300"))

# In the OpenAI API version, local CPU/GPU selection does not control model inference.
# Keep this field for manifests and compatibility with the previous notebook.
DEVICE_MODE = "openai_api"

# Reduce noisy telemetry and improve tokenizer behavior where supported.
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return str(raw).strip().lower() in {"1", "true", "yes", "y", "on"}


def _env_int(name: str, default=None):
    raw = os.environ.get(name)
    if raw is None or str(raw).strip() == "":
        return default
    return int(raw)

# ----------------------------
# Performance/resume controls
# ----------------------------
# These are the main speed controls. Turn FORCE_* on only when you want a clean rebuild.
FORCE_REBUILD_ALL = _env_bool("FORCE_REBUILD_ALL", True)
FORCE_REBUILD_CHROMA = _env_bool("FORCE_REBUILD_CHROMA", FORCE_REBUILD_ALL)
FORCE_REBUILD_GRAPHS = _env_bool("FORCE_REBUILD_GRAPHS", FORCE_REBUILD_ALL)
REUSE_EXISTING_EXTRACTION = _env_bool("REUSE_EXISTING_EXTRACTION", True)
REUSE_EXISTING_UNITS_AND_CHUNKS = _env_bool("REUSE_EXISTING_UNITS_AND_CHUNKS", True)
REUSE_EXISTING_PROFILES = _env_bool("REUSE_EXISTING_PROFILES", True)
REUSE_EXISTING_VECTOR_STORES = _env_bool("REUSE_EXISTING_VECTOR_STORES", True)
REUSE_EXISTING_GRAPHS = _env_bool("REUSE_EXISTING_GRAPHS", True)
EMBEDDING_CACHE_ENABLED = _env_bool("EMBEDDING_CACHE_ENABLED", True)

# Parallelize PDF extraction across files. Keep this modest; each PDF may render/crop pages.
PDF_EXTRACTION_WORKERS = _env_int("PDF_EXTRACTION_WORKERS", max(1, min(4, (os.cpu_count() or 2) // 2 or 1)))

# OpenAI embedding batching. Larger batches are faster but may hit rate limits or token limits on huge chunks.
EMBEDDING_BATCH_SIZE = _env_int("EMBEDDING_BATCH_SIZE", 128)
CHROMA_UPSERT_BATCH_SIZE = _env_int("CHROMA_UPSERT_BATCH_SIZE", 512)

# ----------------------------
# OCR and extraction settings
# ----------------------------
# OCR is useful only for scanned PDFs. Keeping it off makes native PDFs much faster.
OCR_ENABLED = _env_bool("OCR_ENABLED", False)
OCR_LANGUAGE = os.environ.get("OCR_LANGUAGE", "eng")

# Extraction settings adapted from the local open-source PDF pipeline.
SAVE_PAGE_RENDERS = _env_bool("SAVE_PAGE_RENDERS", False)
SAVE_EXTRACTION_CROPS = _env_bool("SAVE_EXTRACTION_CROPS", True)
EXTRACT_EMBEDDED_IMAGES = _env_bool("EXTRACT_EMBEDDED_IMAGES", True)
EXPORT_TABLE_ASSETS = True
EXPORT_LINK_UNITS = True
PDF_RENDER_ZOOM = float(os.environ.get("PDF_RENDER_ZOOM", "2.0"))
PDF_RENDER_DPI = _env_int("PDF_RENDER_DPI", 200)
FIGURE_CROP_PADDING_POINTS = float(os.environ.get("FIGURE_CROP_PADDING_POINTS", "8.0"))
EQUATION_CROP_PADDING_POINTS = float(os.environ.get("EQUATION_CROP_PADDING_POINTS", "10.0"))
MIN_NATIVE_WORDS_FOR_NO_OCR = _env_int("MIN_NATIVE_WORDS_FOR_NO_OCR", 25)
SCANNED_IMAGE_COVERAGE_THRESHOLD = float(os.environ.get("SCANNED_IMAGE_COVERAGE_THRESHOLD", "0.55"))
MIN_FIGURE_AREA_FRACTION = float(os.environ.get("MIN_FIGURE_AREA_FRACTION", "0.015"))
MERGE_GRAPHICS_MARGIN_POINTS = float(os.environ.get("MERGE_GRAPHICS_MARGIN_POINTS", "12.0"))
FAST_FIGURE_MODE = _env_bool("FAST_FIGURE_MODE", True)
RENDER_ONLY_FIGURE_PAGES = _env_bool("RENDER_ONLY_FIGURE_PAGES", True)
FIGURE_MIN_IMAGE_BYTES = _env_int("FIGURE_MIN_IMAGE_BYTES", 4096)
MAX_EMBEDDED_IMAGES_PER_PAGE = _env_int("MAX_EMBEDDED_IMAGES_PER_PAGE", None)
IMAGE_DEDUP_BY_HASH = _env_bool("IMAGE_DEDUP_BY_HASH", True)

# Optional OpenAI vision descriptions for figure crops. This can add cost/time, so it is off by default.
USE_VISION_FOR_FIGURES = _env_bool("USE_VISION_FOR_FIGURES", False)

# Chunking. Larger chunks reduce embedding/KG calls and usually speed up builds.
CHUNK_SIZE_CHARS = _env_int("CHUNK_SIZE_CHARS", 2400)
CHUNK_OVERLAP_CHARS = _env_int("CHUNK_OVERLAP_CHARS", 200)
PROFILE_CONTEXT_MAX_CHARS = _env_int("PROFILE_CONTEXT_MAX_CHARS", 14000)

# ----------------------------
# Per-paper GraphRAG controls
# ----------------------------
RUN_KG_EXTRACTION = _env_bool("RUN_KG_EXTRACTION", True)
# Keep this limited for speed. Set MAX_CHUNKS_PER_PDF_FOR_KG=0 to disable LLM KG extraction.
_mkg = _env_int("MAX_CHUNKS_PER_PDF_FOR_KG", 8)
MAX_CHUNKS_PER_PDF_FOR_KG = None if _mkg is None or _mkg < 0 else _mkg
if MAX_CHUNKS_PER_PDF_FOR_KG == 0:
    RUN_KG_EXTRACTION = False
KG_ONLY_HIGH_VALUE_CHUNKS = _env_bool("KG_ONLY_HIGH_VALUE_CHUNKS", True)
KG_SORT_BY_SCORE = _env_bool("KG_SORT_BY_SCORE", True)
KG_MIN_CHARS = _env_int("KG_MIN_CHARS", 500)
KG_MIN_SCORE = float(os.environ.get("KG_MIN_SCORE", "2.5"))
KG_MAX_CHARS_PER_CHUNK = _env_int("KG_MAX_CHARS_PER_CHUNK", 4500)
KG_BATCH_SLEEP_SECONDS = float(os.environ.get("KG_BATCH_SLEEP_SECONDS", "0.0"))
KG_RESUME_FROM_CACHE = _env_bool("KG_RESUME_FROM_CACHE", True)

# Retrieval
VECTOR_TOP_K = _env_int("VECTOR_TOP_K", 8)
GRAPH_TOP_NODES = _env_int("GRAPH_TOP_NODES", 8)
GRAPH_HOPS = _env_int("GRAPH_HOPS", 1)
MAX_CONTEXT_CHUNKS = _env_int("MAX_CONTEXT_CHUNKS", 14)
CROSS_PAPER_TOP_PAPERS = _env_int("CROSS_PAPER_TOP_PAPERS", 4)
CROSS_PAPER_PER_PAPER_HITS = _env_int("CROSS_PAPER_PER_PAPER_HITS", 4)

# Production inference flow
PRODUCTION_USE_PER_PAPER_FIRST = True
PRODUCTION_USE_CORPUS_SECOND = True
PRODUCTION_AUTO_COMPARISON = True
ROUTER_TOP_PAPERS = _env_int("ROUTER_TOP_PAPERS", 6)
CORPUS_ROUTER_TOP_K = _env_int("CORPUS_ROUTER_TOP_K", 12)
MAX_SYNTHESIS_CONTEXT_HITS = _env_int("MAX_SYNTHESIS_CONTEXT_HITS", 18)
MIN_ROUTER_SCORE = float(os.environ.get("MIN_ROUTER_SCORE", "0.15"))

# Corpus comparison artifacts
BUILD_CORPUS_COMPARISON = _env_bool("BUILD_CORPUS_COMPARISON", True)
PAIRWISE_TOP_K = _env_int("PAIRWISE_TOP_K", 3)
EXPORT_FINETUNE_FILES = _env_bool("EXPORT_FINETUNE_FILES", True)

# Storage naming
PER_PDF_VECTOR_SUFFIX = "_vector"
PER_PDF_GRAPH_SUFFIX = "_graph"

MANIFEST_DIR = OUT_DIR / "manifests"
CORPUS_VECTOR_DIR = OUT_DIR / "corpus_vector"
CORPUS_GRAPH_DIR = OUT_DIR / "corpus_graph"
FINETUNE_EXPORT_DIR = OUT_DIR / "finetune_exports"
EMBEDDING_CACHE_DIR = OUT_DIR / "embedding_cache"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_VECTOR_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_GRAPH_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# PDF-to-LLM extraction integration
USE_PDF_TO_LLM_OPEN_SOURCE = True
PDF_TO_LLM_EXTRACTION_SUBDIR = "llm_extraction"
BUILD_LLM_LEARNING_PACK = True

# Audit-ready extraction controls. These use OpenAI only when RUN_OPENAI_EXTRACTION_AUDIT=True.
# The notebook creates deterministic reviewable candidate inventories by default.
RUN_OPENAI_EXTRACTION_AUDIT = _env_bool("RUN_OPENAI_EXTRACTION_AUDIT", False)
# Backward-compatible alias for notebooks/scripts that still reference the old name.
RUN_LOCAL_LLM_EXTRACTION_AUDIT = RUN_OPENAI_EXTRACTION_AUDIT

INCLUDE_EXTRACTION_AUDIT_TASKS_IN_RAG = _env_bool("INCLUDE_EXTRACTION_AUDIT_TASKS_IN_RAG", False)
DROP_LLM_AUDIT_FALSE_POSITIVES_FROM_RAG = _env_bool("DROP_LLM_AUDIT_FALSE_POSITIVES_FROM_RAG", False)
ADD_LLM_AUDIT_FALSE_NEGATIVES_TO_RAG = _env_bool("ADD_LLM_AUDIT_FALSE_NEGATIVES_TO_RAG", False)
MAX_AUDIT_PAGE_TEXT_CHARS = _env_int("MAX_AUDIT_PAGE_TEXT_CHARS", 9000)
MAX_AUDIT_CONTEXT_LINES = _env_int("MAX_AUDIT_CONTEXT_LINES", 8)

# Equation-to-LaTeX controls. These use OpenAI only when RUN_LLM_EQUATION_LATEX_AUDIT=True.
ENABLE_EQUATION_LATEX_LAYER = _env_bool("ENABLE_EQUATION_LATEX_LAYER", True)
RUN_LLM_EQUATION_LATEX_AUDIT = _env_bool("RUN_LLM_EQUATION_LATEX_AUDIT", True)
EQUATION_LATEX_MODEL = os.environ.get("OPENAI_EQUATION_LATEX_MODEL", EXTRACTION_AUDIT_MODEL)
EQUATION_LATEX_USE_CROPS = _env_bool("EQUATION_LATEX_USE_CROPS", True)
EQUATION_LATEX_MAX_CANDIDATES_PER_PDF = _env_int("EQUATION_LATEX_MAX_CANDIDATES_PER_PDF", 250)
DROP_EQUATION_FALSE_POSITIVES_FROM_RAG = _env_bool("DROP_EQUATION_FALSE_POSITIVES_FROM_RAG", True)
ADD_EQUATION_FALSE_NEGATIVES_TO_RAG = _env_bool("ADD_EQUATION_FALSE_NEGATIVES_TO_RAG", True)
EQUATION_LATEX_CONFIDENCE_KEEP_THRESHOLD = float(os.environ.get("EQUATION_LATEX_CONFIDENCE_KEEP_THRESHOLD", "0.55"))
EQUATION_LATEX_STRONG_FP_SKIP_LLM = _env_bool("EQUATION_LATEX_STRONG_FP_SKIP_LLM", True)

print("PDF_DIR:", PDF_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())
print("Provider:", DEVICE_MODE, "| EMBEDDING_BATCH_SIZE:", EMBEDDING_BATCH_SIZE)
print("Generation model:", GENERATION_MODEL)
print("Embedding model:", EMBEDDING_MODEL, "| dimensions:", EMBEDDING_DIMENSIONS or "model default")
print("Answer model:", ANSWER_MODEL)
print("Reasoning effort:", OPENAI_REASONING_EFFORT, "| verbosity:", OPENAI_TEXT_VERBOSITY)
print("PDF extraction workers:", PDF_EXTRACTION_WORKERS)
print("Equation LaTeX layer:", ENABLE_EQUATION_LATEX_LAYER, "| LLM audit:", RUN_LLM_EQUATION_LATEX_AUDIT, "| model:", EQUATION_LATEX_MODEL)


PDF_DIR: /Users/rahulmitra/Downloads/RAG_webui_OpenAI/myPDFs
OUT_DIR: /Users/rahulmitra/Downloads/RAG_webui_OpenAI/rag_output_openai
Provider: openai_api | EMBEDDING_BATCH_SIZE: 128
Generation model: gpt-5.2
Embedding model: text-embedding-3-large | dimensions: model default
Answer model: gpt-5.2
Reasoning effort: high | verbosity: low
PDF extraction workers: 4
Equation LaTeX layer: True | LLM audit: True | model: gpt-5.2


## 3. Imports and utility helpers

In [3]:

import base64
import shutil
import gc
import hashlib
import json
import math
import re
import time
import statistics
from collections import defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from openai import OpenAI
from tqdm.auto import tqdm

import fitz  # PyMuPDF
import pdfplumber
import networkx as nx
import chromadb

try:
    from rapidfuzz import fuzz
except Exception:
    fuzz = None

try:
    from PIL import Image
except Exception:
    Image = None


def stable_id(*parts: Any, length: int = 16) -> str:
    raw = "|".join(str(p) for p in parts)
    return hashlib.sha1(raw.encode("utf-8", errors="ignore")).hexdigest()[:length]


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"([A-Za-z])[-‐‑‒–—]\n([A-Za-z])", r"\1\2", text)
    return text.strip()


def safe_filename(name: str, max_len: int = 80) -> str:
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", str(name)).strip("._")
    return name[:max_len] or "file"


def parse_json_loose(raw: str) -> Dict[str, Any]:
    if raw is None:
        return {}
    text = str(raw).strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {}
    return {}


def list_pdf_files(pdf_dir: Path) -> List[Path]:
    pdfs = sorted(pdf_dir.glob("*.pdf"))
    if not pdfs:
        raise FileNotFoundError(f"No PDFs found in {pdf_dir.resolve()}. Put your PDFs in that folder first.")
    return pdfs


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def append_jsonl(path: Path, row: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def image_to_base64(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode("utf-8")


def ensure_list_of_strings(value: Any) -> List[str]:
    if value is None:
        return []
    if isinstance(value, str):
        value = [value]
    if not isinstance(value, (list, tuple, set)):
        return [clean_text(str(value))]
    out = []
    for item in value:
        item = clean_text(str(item))
        if item:
            out.append(item)
    return out


def ensure_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return "; ".join(clean_text(str(x)) for x in value if clean_text(str(x)))
    return clean_text(str(value))


def cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    if len(vectors) == 0:
        return np.zeros((0, 0), dtype=float)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    normalized = vectors / norms
    return normalized @ normalized.T

try:
    import pypdfium2 as pdfium
except Exception:
    pdfium = None

try:
    from pypdf import PdfReader
except Exception:
    PdfReader = None


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. OpenAI helpers


In [4]:
_OPENAI_CLIENT = None


def _openai_client() -> OpenAI:
    """Create the OpenAI client lazily so the notebook can load before the key is set."""
    global _OPENAI_CLIENT
    if _OPENAI_CLIENT is not None:
        return _OPENAI_CLIENT
    api_key = os.environ.get("OPENAI_API_KEY") or OPENAI_API_KEY
    if not api_key:
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Set it before running model calls, e.g. "
            "export OPENAI_API_KEY='sk-...' or os.environ['OPENAI_API_KEY']='sk-...'."
        )
    kwargs = {"api_key": api_key}
    if OPENAI_BASE_URL:
        kwargs["base_url"] = OPENAI_BASE_URL
    _OPENAI_CLIENT = OpenAI(**kwargs)
    return _OPENAI_CLIENT


def openai_available() -> bool:
    return bool(os.environ.get("OPENAI_API_KEY") or OPENAI_API_KEY)


def _retry_openai_call(fn, description: str):
    last_exc = None
    for attempt in range(int(OPENAI_MAX_RETRIES)):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            wait_s = min(60.0, float(OPENAI_RETRY_BASE_SECONDS) * (2 ** attempt))
            print(f"{description} failed on attempt {attempt + 1}/{OPENAI_MAX_RETRIES}: {exc}")
            if attempt < int(OPENAI_MAX_RETRIES) - 1:
                time.sleep(wait_s)
    raise last_exc


def _image_to_data_url(path_or_b64: str) -> str:
    """Accept a path, raw base64 string, or already-formed data URL."""
    s = str(path_or_b64)
    if s.startswith("data:image/"):
        return s
    p = Path(s)
    if p.exists():
        suffix = p.suffix.lower().lstrip(".") or "png"
        if suffix == "jpg":
            suffix = "jpeg"
        return f"data:image/{suffix};base64,{image_to_base64(p)}"
    # Assume caller passed raw base64.
    return f"data:image/png;base64,{s}"


def _extract_openai_response_text(response: Any) -> str:
    """Extract text from a Responses API object across SDK versions."""
    text = getattr(response, "output_text", None)
    if text:
        return str(text)

    try:
        parts = []
        for item in getattr(response, "output", []) or []:
            for content in getattr(item, "content", []) or []:
                value = getattr(content, "text", None)
                if value:
                    parts.append(str(value))
        if parts:
            return "\n".join(parts)
    except Exception:
        pass

    try:
        data = response.model_dump()
        parts = []
        for item in data.get("output", []) or []:
            for content in item.get("content", []) or []:
                if content.get("type") in {"output_text", "text"} and content.get("text"):
                    parts.append(content["text"])
        if parts:
            return "\n".join(parts)
    except Exception:
        pass

    return str(response)


def openai_generate(
    prompt: str,
    model: str = GENERATION_MODEL,
    system: Optional[str] = None,
    format_json: bool = False,
    temperature: float = 0.0,
    timeout: int = 300,
    images: Optional[List[str]] = None,
) -> str:
    """Generate text using the OpenAI Responses API.

    Uses GPT-5.2-style reasoning controls. If format_json=True, the call asks for JSON mode
    and falls back to a plain JSON-instructed prompt if the account/model rejects JSON mode.
    """
    client = _openai_client().with_options(timeout=timeout or OPENAI_REQUEST_TIMEOUT)

    content: List[Dict[str, Any]] = [{"type": "input_text", "text": str(prompt or "")}]
    for img in images or []:
        content.append({"type": "input_image", "image_url": _image_to_data_url(img)})

    request: Dict[str, Any] = {
        "model": model,
        "input": [{"role": "user", "content": content}],
        "reasoning": {"effort": OPENAI_REASONING_EFFORT},
        "text": {"verbosity": OPENAI_TEXT_VERBOSITY},
    }
    if system:
        request["instructions"] = system
    if format_json:
        request["text"] = {
            "verbosity": OPENAI_TEXT_VERBOSITY,
            "format": {"type": "json_object"},
        }
    # GPT-5.2 supports temperature only when reasoning.effort is "none".
    if OPENAI_REASONING_EFFORT == "none":
        request["temperature"] = float(temperature)

    def _call():
        try:
            return client.responses.create(**request)
        except Exception:
            if format_json:
                # Some SDK/account combinations may not accept json_object in Responses.
                fallback_request = dict(request)
                fallback_request.pop("text", None)
                fallback_request["input"] = [{
                    "role": "user",
                    "content": [{
                        "type": "input_text",
                        "text": "Return valid JSON only. Do not include markdown fences.\n\n" + str(prompt or ""),
                    }] + content[1:],
                }]
                return client.responses.create(**fallback_request)
            raise

    response = _retry_openai_call(_call, f"OpenAI generation ({model})")
    return _extract_openai_response_text(response)


_EMBEDDING_CACHE: Dict[str, Dict[str, List[float]]] = {}
_EMBEDDING_CACHE_LOADED: set[str] = set()


def _embedding_cache_namespace(model: str) -> str:
    suffix = f"_{EMBEDDING_DIMENSIONS}d" if EMBEDDING_DIMENSIONS else "_default_dims"
    return f"{model}{suffix}"


def _embedding_cache_path(model: str) -> Path:
    return EMBEDDING_CACHE_DIR / f"{safe_filename(_embedding_cache_namespace(model), 100)}.jsonl"


def _embedding_cache_key(model: str, text: str) -> str:
    raw = (model + "\0" + str(EMBEDDING_DIMENSIONS) + "\0" + str(text or "")).encode("utf-8", errors="ignore")
    return hashlib.sha256(raw).hexdigest()


def _load_embedding_cache(model: str) -> Dict[str, List[float]]:
    if not EMBEDDING_CACHE_ENABLED:
        return {}
    namespace = _embedding_cache_namespace(model)
    if namespace in _EMBEDDING_CACHE_LOADED:
        return _EMBEDDING_CACHE.setdefault(namespace, {})
    cache: Dict[str, List[float]] = {}
    path = _embedding_cache_path(model)
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                    key = row.get("key")
                    emb = row.get("embedding")
                    if key and isinstance(emb, list):
                        cache[key] = emb
                except Exception:
                    continue
    _EMBEDDING_CACHE[namespace] = cache
    _EMBEDDING_CACHE_LOADED.add(namespace)
    return cache


def _append_embedding_cache_rows(model: str, rows: List[Dict[str, Any]]) -> None:
    if not EMBEDDING_CACHE_ENABLED or not rows:
        return
    path = _embedding_cache_path(model)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _openai_embed_batch(batch: List[str], model: str, timeout: int = 300) -> List[List[float]]:
    if not batch:
        return []
    client = _openai_client().with_options(timeout=timeout or OPENAI_REQUEST_TIMEOUT)
    request: Dict[str, Any] = {"model": model, "input": batch}
    if EMBEDDING_DIMENSIONS:
        request["dimensions"] = int(EMBEDDING_DIMENSIONS)

    response = _retry_openai_call(lambda: client.embeddings.create(**request), f"OpenAI embeddings ({model})")
    embeddings = [item.embedding for item in response.data]
    if len(embeddings) != len(batch):
        raise RuntimeError(f"Expected {len(batch)} embeddings, got {len(embeddings)}")
    return embeddings


def openai_embed(texts: List[str], model: str = EMBEDDING_MODEL, batch_size: Optional[int] = None) -> List[List[float]]:
    texts = [str(t or "") for t in texts]
    if not texts:
        return []
    batch_size = int(batch_size or EMBEDDING_BATCH_SIZE)
    results: List[Optional[List[float]]] = [None] * len(texts)
    missing: List[Tuple[int, str, str]] = []

    cache = _load_embedding_cache(model)
    for i, text in enumerate(texts):
        key = _embedding_cache_key(model, text)
        emb = cache.get(key)
        if emb is not None:
            results[i] = emb
        else:
            missing.append((i, key, text))

    if missing:
        new_rows: List[Dict[str, Any]] = []
        for start in tqdm(range(0, len(missing), batch_size), desc="Embedding missing texts with OpenAI", leave=False):
            batch_items = missing[start:start + batch_size]
            batch_texts = [x[2] for x in batch_items]
            embeddings = _openai_embed_batch(batch_texts, model=model)
            for (idx, key, text), emb in zip(batch_items, embeddings):
                results[idx] = emb
                cache[key] = emb
                new_rows.append({
                    "key": key,
                    "model": model,
                    "dimensions": EMBEDDING_DIMENSIONS,
                    "text_sha256": hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest(),
                    "embedding": emb,
                })
        _append_embedding_cache_rows(model, new_rows)

    final = [r for r in results if r is not None]
    if len(final) != len(texts):
        raise RuntimeError("Internal embedding cache error: missing embeddings after OpenAI call")
    return final


# Backward-compatible aliases for any older cells that still use the previous helper names.
ollama_generate = openai_generate
ollama_embed = openai_embed
ollama_available = openai_available
ollama_tags = lambda *args, **kwargs: [GENERATION_MODEL, ANSWER_MODEL, EMBEDDING_MODEL]


if openai_available():
    print("OpenAI API key detected.")
    print("Generation model:", GENERATION_MODEL)
    print("Answer model:", ANSWER_MODEL)
    print("Embedding model:", EMBEDDING_MODEL)
else:
    print("OPENAI_API_KEY is not set yet. Set it before running extraction/profile/KG/embedding/answer cells.")


OpenAI API key detected.
Generation model: gpt-5.2
Answer model: gpt-5.2
Embedding model: text-embedding-3-large


## 5. Extract text, tables, figures, captions, equation candidates, and links

This section now follows the local open-source extraction pipeline style used in `pdf_to_llm_open_source.py`: native text/line extraction with `pdfplumber`, local page rendering with `pypdfium2`, optional OCR for scanned pages, table extraction, figure/graphic region detection, embedded-image export, and equation candidate cropping.

In [5]:
BBox = Tuple[float, float, float, float]  # x0, top, x1, bottom


def _table_settings() -> Dict[str, Any]:
    return {
        "vertical_strategy": "lines",
        "horizontal_strategy": "lines",
        "snap_tolerance": 3,
        "join_tolerance": 3,
        "edge_min_length": 3,
        "min_words_vertical": 3,
        "min_words_horizontal": 1,
        "intersection_tolerance": 3,
        "text_tolerance": 3,
    }


def relpath_str(path: Path, start: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(Path(start).resolve())).replace(os.sep, "/")
    except Exception:
        return str(path)


def bbox_tuple(obj: Dict[str, Any]) -> Optional[BBox]:
    try:
        x0 = float(obj.get("x0", 0.0))
        x1 = float(obj.get("x1", 0.0))
        top = float(obj.get("top", obj.get("y0", 0.0)))
        bottom = float(obj.get("bottom", obj.get("y1", 0.0)))
        if x1 <= x0 or bottom <= top:
            return None
        return (x0, top, x1, bottom)
    except Exception:
        return None


def bbox_to_dict(b: Optional[BBox]) -> Optional[Dict[str, float]]:
    if b is None:
        return None
    x0, top, x1, bottom = b
    return {
        "x0": round(float(x0), 3),
        "top": round(float(top), 3),
        "x1": round(float(x1), 3),
        "bottom": round(float(bottom), 3),
        "width": round(float(x1 - x0), 3),
        "height": round(float(bottom - top), 3),
    }


def bbox_area(b: Optional[BBox]) -> float:
    if not b:
        return 0.0
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def expand_bbox(b: BBox, pad: float, page_width: float, page_height: float) -> BBox:
    return (
        max(0.0, b[0] - pad),
        max(0.0, b[1] - pad),
        min(page_width, b[2] + pad),
        min(page_height, b[3] + pad),
    )


def overlap_area(a: BBox, b: BBox) -> float:
    x0 = max(a[0], b[0])
    top = max(a[1], b[1])
    x1 = min(a[2], b[2])
    bottom = min(a[3], b[3])
    if x1 <= x0 or bottom <= top:
        return 0.0
    return (x1 - x0) * (bottom - top)


def overlap_ratio(a: BBox, b: BBox) -> float:
    denom = min(bbox_area(a), bbox_area(b)) or 1.0
    return overlap_area(a, b) / denom


def boxes_touch_or_overlap(a: BBox, b: BBox, margin: float) -> bool:
    ae = (a[0] - margin, a[1] - margin, a[2] + margin, a[3] + margin)
    return overlap_area(ae, b) > 0


def union_bbox(boxes: Iterable[BBox]) -> BBox:
    box_list = list(boxes)
    return (
        min(b[0] for b in box_list),
        min(b[1] for b in box_list),
        max(b[2] for b in box_list),
        max(b[3] for b in box_list),
    )


def merge_nearby_boxes(boxes: List[BBox], margin: float) -> List[BBox]:
    clusters: List[List[BBox]] = []
    for box in boxes:
        placed = False
        for cluster in clusters:
            if boxes_touch_or_overlap(union_bbox(cluster), box, margin):
                cluster.append(box)
                placed = True
                break
        if not placed:
            clusters.append([box])
    changed = True
    while changed:
        changed = False
        new_clusters: List[List[BBox]] = []
        for cluster in clusters:
            cbox = union_bbox(cluster)
            merged = False
            for existing in new_clusters:
                if boxes_touch_or_overlap(union_bbox(existing), cbox, margin):
                    existing.extend(cluster)
                    changed = True
                    merged = True
                    break
            if not merged:
                new_clusters.append(cluster)
        clusters = new_clusters
    return [union_bbox(c) for c in clusters]


def matrix_to_markdown(matrix: List[List[Any]]) -> str:
    rows = [["" if cell is None else clean_text(str(cell).replace("\n", " ")) for cell in row] for row in matrix if row]
    if not rows:
        return ""
    width = max(len(row) for row in rows)
    rows = [row + [""] * (width - len(row)) for row in rows]
    col_widths = [max(len(row[i]) for row in rows) for i in range(width)]

    def _fmt(row: List[str]) -> str:
        return "| " + " | ".join(row[i].ljust(col_widths[i]) for i in range(width)) + " |"

    header = _fmt(rows[0])
    sep = "| " + " | ".join("-" * max(3, col_widths[i]) for i in range(width)) + " |"
    body = [_fmt(row) for row in rows[1:]]
    return "\n".join([header, sep] + body)


def write_text_file(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def render_page_to_pil(pdf_path: Path, page_index: int, dpi: int = PDF_RENDER_DPI) -> Any:
    if pdfium is None:
        with fitz.open(pdf_path) as doc:
            page = doc[page_index]
            scale = max(1.0, dpi / 72.0)
            pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
            if Image is None:
                raise RuntimeError("Pillow is required for page rendering.")
            return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    pdf = pdfium.PdfDocument(str(pdf_path))
    try:
        page = pdf[page_index]
        bitmap = page.render(scale=dpi / 72.0)
        return bitmap.to_pil()
    finally:
        try:
            pdf.close()
        except Exception:
            pass


def render_page_to_png(doc_or_path: Any, page_index: int, out_path: Path, zoom: float = PDF_RENDER_ZOOM) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path
    if isinstance(doc_or_path, fitz.Document):
        page = doc_or_path[page_index]
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
        pix.save(out_path)
        return out_path
    dpi = int(round(max(1.0, zoom) * 72))
    img = render_page_to_pil(Path(doc_or_path), page_index, dpi=max(PDF_RENDER_DPI, dpi))
    img.save(out_path)
    return out_path


def save_bbox_crop(pdf_path: Path, page_index: int, bbox: BBox, page_width: float, page_height: float,
                   out_path: Path, dpi: int = PDF_RENDER_DPI, pad_points: float = FIGURE_CROP_PADDING_POINTS) -> Optional[str]:
    try:
        b = expand_bbox(bbox, pad_points, page_width, page_height)
        img = render_page_to_pil(pdf_path, page_index, dpi)
        scale = dpi / 72.0
        crop_box = (
            max(0, int(math.floor(b[0] * scale))),
            max(0, int(math.floor(b[1] * scale))),
            min(img.width, int(math.ceil(b[2] * scale))),
            min(img.height, int(math.ceil(b[3] * scale))),
        )
        if crop_box[2] <= crop_box[0] or crop_box[3] <= crop_box[1]:
            return None
        crop = img.crop(crop_box)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        crop.save(out_path)
        return str(out_path)
    except Exception:
        return None


def extract_pdf_metadata(pdf_path: Path) -> Dict[str, Any]:
    try:
        if PdfReader is not None:
            reader = PdfReader(str(pdf_path))
            metadata = {str(k).lstrip("/"): str(v) for k, v in dict(reader.metadata or {}).items()}
            return {"page_count": len(reader.pages), "pdf_metadata": metadata, "is_encrypted": bool(reader.is_encrypted)}
    except Exception as exc:
        return {"page_count": None, "pdf_metadata": {}, "metadata_error": repr(exc)}
    try:
        with fitz.open(pdf_path) as doc:
            return {"page_count": len(doc), "pdf_metadata": dict(doc.metadata or {}), "is_encrypted": False}
    except Exception as exc:
        return {"page_count": None, "pdf_metadata": {}, "metadata_error": repr(exc)}


def extract_embedded_images_with_pypdf(pdf_path: Path, page_index: int, out_dir: Path, rel_root: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    if PdfReader is None:
        return records
    try:
        reader = PdfReader(str(pdf_path))
        images = getattr(reader.pages[page_index], "images", [])
        for i, img in enumerate(images):
            raw_name = getattr(img, "name", f"image_{i}.bin") or f"image_{i}.bin"
            suffix = Path(raw_name).suffix or ".bin"
            out_path = out_dir / f"page_{page_index+1:04d}_embedded_{i:03d}{suffix}"
            data = getattr(img, "data", None)
            if data:
                out_path.parent.mkdir(parents=True, exist_ok=True)
                out_path.write_bytes(data)
                records.append({
                    "id": f"p{page_index+1:04d}_embedded_image_{i:03d}",
                    "source": "pypdf_page_images",
                    "file": relpath_str(out_path, rel_root),
                    "name": raw_name,
                    "bytes": len(data),
                })
    except Exception as exc:
        records.append({"source": "pypdf_page_images", "error": repr(exc)})
    return records


def page_image_coverage(page: Any) -> float:
    page_area = float(page.width * page.height) or 1.0
    total = 0.0
    for img in getattr(page, "images", []) or []:
        b = bbox_tuple(img)
        if b:
            total += bbox_area(b)
    return min(1.0, total / page_area)


def extract_words_and_lines(page: Any) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], str]:
    try:
        words_raw = page.extract_words(
            x_tolerance=1.5,
            y_tolerance=3,
            use_text_flow=True,
            keep_blank_chars=False,
            extra_attrs=["fontname", "size"],
        )
    except TypeError:
        words_raw = page.extract_words(x_tolerance=1.5, y_tolerance=3, use_text_flow=True, keep_blank_chars=False)
    words_raw = words_raw or []

    words: List[Dict[str, Any]] = []
    for i, w in enumerate(words_raw):
        b = bbox_tuple(w)
        text = clean_text(w.get("text", ""))
        if text and b:
            words.append({
                "id": f"w{i:05d}",
                "text": text,
                "bbox": bbox_to_dict(b),
                "fontname": w.get("fontname"),
                "size": float(w["size"]) if w.get("size") is not None else None,
            })

    sorted_words = sorted(words_raw, key=lambda w: (float(w.get("top", 0)), float(w.get("x0", 0))))
    grouped: List[List[Dict[str, Any]]] = []
    for w in sorted_words:
        b = bbox_tuple(w)
        if not b or not clean_text(w.get("text", "")):
            continue
        for group in grouped:
            gb = union_bbox([bbox_tuple(g) for g in group if bbox_tuple(g)])
            if abs(b[1] - gb[1]) <= 3.0 or overlap_area((0, b[1], page.width, b[3]), (0, gb[1], page.width, gb[3])) > 0:
                group.append(w)
                break
        else:
            grouped.append([w])

    sizes_all = [float(w.get("size")) for w in words_raw if w.get("size") is not None]
    median_size = statistics.median(sizes_all) if sizes_all else None
    caption_re = re.compile(r"^\s*(fig(?:ure)?\.?|table|tab\.?|eq(?:uation)?\.?)\s*\(?\d+[\w.-]*\)?", re.I)
    lines: List[Dict[str, Any]] = []
    for i, group in enumerate(grouped):
        group = sorted(group, key=lambda w: float(w.get("x0", 0)))
        boxes = [bbox_tuple(g) for g in group if bbox_tuple(g)]
        if not boxes:
            continue
        b = union_bbox(boxes)
        text = clean_text(" ".join(g.get("text", "") for g in group))
        if not text:
            continue
        sizes = [float(g.get("size")) for g in group if g.get("size") is not None]
        avg_size = statistics.mean(sizes) if sizes else None
        line_type = "text"
        if caption_re.search(text):
            line_type = "caption"
        elif median_size and avg_size and avg_size >= 1.22 * median_size and len(text) <= 160:
            line_type = "heading"
        elif len(text) <= 90 and text.isupper() and sum(c.isalpha() for c in text) > 4:
            line_type = "heading"
        lines.append({
            "id": f"line_{i:04d}",
            "type": line_type,
            "text": text,
            "bbox": bbox_to_dict(b),
            "word_count": len(group),
            "avg_font_size": round(avg_size, 3) if avg_size else None,
        })

    try:
        plain = clean_text(page.extract_text(layout=True) or "")
    except Exception:
        plain = ""
    if not plain:
        plain = clean_text("\n".join(line["text"] for line in lines))
    return words, lines, plain


def should_ocr_page(page: Any, native_word_count: int) -> Tuple[bool, str, float]:
    coverage = page_image_coverage(page)
    if native_word_count == 0:
        return True, "no_native_words", coverage
    if native_word_count < MIN_NATIVE_WORDS_FOR_NO_OCR and coverage >= SCANNED_IMAGE_COVERAGE_THRESHOLD:
        return True, "few_native_words_and_high_image_coverage", coverage
    if native_word_count < max(5, MIN_NATIVE_WORDS_FOR_NO_OCR // 3):
        return True, "very_few_native_words", coverage
    return False, "native_text_sufficient", coverage


def ocr_page_record_if_needed(pdf_path: Path, page_index: int, page: Any, native_word_count: int) -> Dict[str, Any]:
    needed, reason, coverage = should_ocr_page(page, native_word_count)
    out = {
        "needed": needed,
        "applied": False,
        "reason": reason,
        "language": OCR_LANGUAGE,
        "image_coverage": round(coverage, 4),
        "text": "",
        "words": [],
        "lines": [],
        "error": None,
    }
    if not needed or not OCR_ENABLED:
        return out
    try:
        import pytesseract
        from pytesseract import Output
    except Exception as exc:
        out["error"] = f"pytesseract/tesseract not available: {exc!r}"
        return out
    try:
        img = render_page_to_pil(pdf_path, page_index, PDF_RENDER_DPI).convert("RGB")
        data = pytesseract.image_to_data(img, lang=OCR_LANGUAGE, output_type=Output.DICT)
        scale = PDF_RENDER_DPI / 72.0
        words: List[Dict[str, Any]] = []
        groups: Dict[Tuple[int, int, int], List[Dict[str, Any]]] = {}
        for i, txt0 in enumerate(data.get("text", [])):
            txt = clean_text(txt0)
            if not txt:
                continue
            try:
                conf = float(data.get("conf", ["-1"])[i])
            except Exception:
                conf = -1.0
            if conf < 0:
                continue
            x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
            b = (x / scale, y / scale, (x + w) / scale, (y + h) / scale)
            rec = {"id": f"ocr_w{i:05d}", "text": txt, "bbox": bbox_to_dict(b), "confidence": round(conf, 3)}
            words.append(rec)
            key = (
                int(data.get("block_num", [0])[i]),
                int(data.get("par_num", [0])[i]),
                int(data.get("line_num", [0])[i]),
            )
            groups.setdefault(key, []).append({"text": txt, "bbox": b, "confidence": conf})
        lines = []
        for j, (_, group) in enumerate(sorted(groups.items())):
            b = union_bbox([g["bbox"] for g in group])
            confs = [g["confidence"] for g in group if g["confidence"] >= 0]
            lines.append({
                "id": f"ocr_line_{j:04d}",
                "type": "ocr_text",
                "text": clean_text(" ".join(g["text"] for g in group)),
                "bbox": bbox_to_dict(b),
                "avg_confidence": round(statistics.mean(confs), 3) if confs else None,
            })
        out.update({"applied": True, "text": clean_text("\n".join(line["text"] for line in lines)), "words": words, "lines": lines})
    except Exception as exc:
        out["error"] = repr(exc)
    return out


MATH_SYMBOL_RE = re.compile(r"(\\[a-zA-Z]+|[=<>≤≥±≈≠∞∑∫√∂∇πθλμσΩαβγδ]|\^|_|\bfrac\b|\bsum\b|\bint\b)")


def is_equation_like(text: str) -> bool:
    t = clean_text(text)
    if not t or len(t) > 220:
        return False
    symbol_hits = len(MATH_SYMBOL_RE.findall(t))
    digits = sum(ch.isdigit() for ch in t)
    spaces = t.count(" ")
    if symbol_hits >= 2 and len(t) <= 160:
        return True
    if symbol_hits >= 1 and digits >= 1 and spaces <= 12 and len(t) <= 120:
        return True
    if re.search(r"\b[A-Za-z]\s*=\s*", t) and len(t) <= 140:
        return True
    return False


def detect_equations(lines: List[Dict[str, Any]], pdf_path: Path, page_index: int, page_width: float, page_height: float, out_dir: Path, rel_root: Path) -> List[Dict[str, Any]]:
    equations = []
    for i, line in enumerate(lines):
        text = line.get("text", "")
        bdict = line.get("bbox")
        if not bdict or not is_equation_like(text):
            continue
        b = (bdict["x0"], bdict["top"], bdict["x1"], bdict["bottom"])
        crop_file = None
        if SAVE_EXTRACTION_CROPS:
            out_path = out_dir / f"page_{page_index+1:04d}_equation_{i:03d}.png"
            crop_abs = save_bbox_crop(
                pdf_path,
                page_index,
                b,
                page_width,
                page_height,
                out_path,
                dpi=PDF_RENDER_DPI,
                pad_points=EQUATION_CROP_PADDING_POINTS,
            )
            crop_file = relpath_str(Path(crop_abs), rel_root) if crop_abs else None
        equations.append({
            "id": f"p{page_index+1:04d}_eq_{i:03d}",
            "source": "line_math_heuristic",
            "text": text,
            "latex": None,
            "bbox": bbox_to_dict(b),
            "crop_file": crop_file,
            "caption": None,
        })
    return equations


def extract_tables(page: Any, page_index: int, out_dir: Path, rel_root: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    try:
        tables = page.find_tables(table_settings=_table_settings())
    except Exception:
        tables = []
    for i, table in enumerate(tables or []):
        try:
            matrix = table.extract() or []
        except Exception:
            matrix = []
        matrix = [["" if c is None else clean_text(str(c)) for c in row] for row in matrix if row]
        if not matrix:
            continue
        tid = f"p{page_index+1:04d}_table_{i:03d}"
        markdown = matrix_to_markdown(matrix)
        files = {}
        if EXPORT_TABLE_ASSETS:
            csv_path = out_dir / f"page_{page_index+1:04d}_table_{i:03d}.csv"
            md_path = out_dir / f"page_{page_index+1:04d}_table_{i:03d}.md"
            html_path = out_dir / f"page_{page_index+1:04d}_table_{i:03d}.html"
            pd.DataFrame(matrix).to_csv(csv_path, index=False, header=False)
            write_text_file(md_path, markdown)
            try:
                df = pd.DataFrame(matrix[1:], columns=matrix[0]) if len(matrix) >= 2 else pd.DataFrame(matrix)
                write_text_file(html_path, df.to_html(index=False, escape=True))
            except Exception:
                write_text_file(html_path, "<pre>" + markdown.replace("&", "&amp;").replace("<", "&lt;") + "</pre>")
            files = {
                "csv": relpath_str(csv_path, rel_root),
                "markdown": relpath_str(md_path, rel_root),
                "html": relpath_str(html_path, rel_root),
            }
        b = tuple(float(v) for v in table.bbox) if getattr(table, "bbox", None) else None
        records.append({
            "id": tid,
            "source": "pdfplumber.find_tables",
            "bbox": bbox_to_dict(b) if b else None,
            "row_count": len(matrix),
            "col_count": max((len(r) for r in matrix), default=0),
            "matrix": matrix,
            "markdown": markdown,
            "files": files,
            "caption": None,
        })
    return records


def extract_links(page: Any) -> List[Dict[str, Any]]:
    links = []
    for i, link in enumerate(getattr(page, "hyperlinks", []) or []):
        links.append({
            "id": f"link_{i:04d}",
            "uri": link.get("uri"),
            "bbox": bbox_to_dict(bbox_tuple(link)),
        })
    return links


def caption_candidates(lines: List[Dict[str, Any]], kind: Optional[str] = None) -> List[Dict[str, Any]]:
    caps = [ln for ln in lines if ln.get("type") == "caption"]
    if kind == "figure":
        return [c for c in caps if re.match(r"^\s*fig", c.get("text", ""), re.I)] or caps
    if kind == "table":
        return [c for c in caps if re.match(r"^\s*(table|tab\.)", c.get("text", ""), re.I)] or caps
    if kind == "equation":
        return [c for c in caps if re.match(r"^\s*eq", c.get("text", ""), re.I)] or caps
    return caps


def attach_nearest_caption(record: Dict[str, Any], lines: List[Dict[str, Any]], kind: str) -> None:
    bdict = record.get("bbox")
    if not bdict:
        return
    b = (bdict["x0"], bdict["top"], bdict["x1"], bdict["bottom"])
    best = None
    best_score = float("inf")
    for cap in caption_candidates(lines, kind=kind):
        cdict = cap.get("bbox")
        if not cdict:
            continue
        c = (cdict["x0"], cdict["top"], cdict["x1"], cdict["bottom"])
        vertical_gap = min(abs(c[1] - b[3]), abs(b[1] - c[3]))
        horizontal_overlap = max(0.0, min(b[2], c[2]) - max(b[0], c[0]))
        horizontal_bonus = horizontal_overlap / max(1.0, min(b[2] - b[0], c[2] - c[0]))
        score = vertical_gap - 25.0 * horizontal_bonus
        if score < best_score and vertical_gap < 140:
            best, best_score = cap, score
    if best:
        record["caption"] = {"text": best.get("text"), "line_id": best.get("id"), "bbox": best.get("bbox")}


def page_has_figure_hints(page_text: str, lines: Optional[List[Dict[str, Any]]] = None) -> bool:
    lines = lines or []
    if any(ln.get("type") == "caption" for ln in lines):
        return True
    lower = (page_text or "").lower()
    hint_terms = [
        "fig.", "figure ", "plot", "graph", "histogram", "micrograph", "image",
        "spectrum", "distribution", "tomography", "diagram", "schematic", "chart",
    ]
    return any(term in lower for term in hint_terms)


def extract_figures_and_graphics(page: Any, pdf_path: Path, page_index: int, table_records: List[Dict[str, Any]], equation_records: List[Dict[str, Any]], lines: List[Dict[str, Any]], out_dir: Path, rel_root: Path) -> List[Dict[str, Any]]:
    page_area = float(page.width * page.height) or 1.0
    exclusion_boxes: List[BBox] = []
    for rec in table_records + equation_records:
        bdict = rec.get("bbox")
        if bdict:
            exclusion_boxes.append((bdict["x0"], bdict["top"], bdict["x1"], bdict["bottom"]))
    boxes: List[BBox] = []
    for img in getattr(page, "images", []) or []:
        b = bbox_tuple(img)
        if b and bbox_area(b) / page_area >= 0.003:
            boxes.append(b)
    for source_name in ["rects", "curves", "lines"]:
        for obj in getattr(page, source_name, []) or []:
            b = bbox_tuple(obj)
            if not b:
                continue
            if b[2] - b[0] < 1.0:
                b = (b[0] - 0.5, b[1], b[2] + 0.5, b[3])
            if b[3] - b[1] < 1.0:
                b = (b[0], b[1] - 0.5, b[2], b[3] + 0.5)
            if bbox_area(b) / page_area >= 0.00002:
                boxes.append(b)
    clusters = merge_nearby_boxes(boxes, MERGE_GRAPHICS_MARGIN_POINTS) if boxes else []
    records: List[Dict[str, Any]] = []
    min_area = MIN_FIGURE_AREA_FRACTION * page_area
    for i, b in enumerate(clusters):
        if bbox_area(b) < min_area:
            continue
        if any(overlap_ratio(b, ex) > 0.45 for ex in exclusion_boxes):
            continue
        crop_file = None
        if SAVE_EXTRACTION_CROPS:
            crop_path = out_dir / f"page_{page_index+1:04d}_figure_{i:03d}.png"
            crop_abs = save_bbox_crop(
                pdf_path,
                page_index,
                b,
                page.width,
                page.height,
                crop_path,
                dpi=PDF_RENDER_DPI,
                pad_points=FIGURE_CROP_PADDING_POINTS,
            )
            crop_file = relpath_str(Path(crop_abs), rel_root) if crop_abs else None
        rec = {
            "id": f"p{page_index+1:04d}_figure_{i:03d}",
            "source": "pdfplumber_graphic_cluster",
            "bbox": bbox_to_dict(b),
            "crop_file": crop_file,
            "caption": None,
            "note": "Candidate figure/graphic region. May include plots, diagrams, raster images, or vector graphics.",
        }
        attach_nearest_caption(rec, lines, kind="figure")
        records.append(rec)
    return records


def describe_image_with_openai(image_path: Path, surrounding_text: str = "") -> str:
    if not USE_VISION_FOR_FIGURES or not VISION_MODEL:
        return ""
    prompt = f"""
You are describing a scientific figure extracted from a PDF.
Use only visible information in the image.
Return a concise description of the plot/chart/diagram, visible labels if readable, and the main trend or structure.

Nearby text:
{surrounding_text[:1500]}
""".strip()
    try:
        return openai_generate(
            prompt=prompt,
            model=VISION_MODEL,
            images=[str(image_path)],
            temperature=0.0,
            timeout=OPENAI_REQUEST_TIMEOUT if "OPENAI_REQUEST_TIMEOUT" in globals() else 300,
        )
    except Exception as exc:
        print(f"Vision description failed for {image_path.name}: {exc}")
        return ""


def extract_pdf_units(pdf_path: Path, paper_out_dir: Path) -> List[Dict[str, Any]]:
    units: List[Dict[str, Any]] = []
    pdf_id = stable_id(pdf_path.resolve(), pdf_path.stat().st_mtime_ns, pdf_path.stat().st_size)
    file_name = pdf_path.name

    assets_dir = paper_out_dir / "assets"
    page_render_dir = paper_out_dir / "page_renders"
    tables_dir = assets_dir / "tables"
    figures_dir = assets_dir / "figures"
    equations_dir = assets_dir / "equations"
    embedded_dir = assets_dir / "embedded_images"

    for d in [assets_dir, page_render_dir, tables_dir, figures_dir, equations_dir, embedded_dir]:
        d.mkdir(parents=True, exist_ok=True)

    meta = extract_pdf_metadata(pdf_path)
    units.append({
        "unit_id": f"docmeta_{pdf_id}",
        "pdf_id": pdf_id,
        "file_name": file_name,
        "source_path": str(pdf_path),
        "page": None,
        "element_type": "document_metadata",
        "text": json.dumps(meta, ensure_ascii=False),
        "metadata": meta,
    })

    with pdfplumber.open(str(pdf_path)) as pdf:
        seen_embedded_hashes = set()

        for page_index, page in enumerate(tqdm(pdf.pages, desc=f"Extracting {file_name}", leave=False)):
            page_number = page_index + 1
            words, lines, page_text = extract_words_and_lines(page)
            ocr = ocr_page_record_if_needed(pdf_path, page_index, page, len(words))
            if not page_text and ocr.get("text"):
                page_text = ocr["text"]

            figure_hint = page_has_figure_hints(page_text, lines)
            if SAVE_PAGE_RENDERS or (OCR_ENABLED and ocr.get("needed")) or (USE_VISION_FOR_FIGURES and figure_hint):
                page_render_path = page_render_dir / f"p{page_number:04d}.png"
                try:
                    render_page_to_png(pdf_path, page_index, page_render_path, zoom=PDF_RENDER_ZOOM)
                except Exception as exc:
                    print(f"Page render failed for {file_name} page {page_number}: {exc}")

            if page_text:
                units.append({
                    "unit_id": f"text_{stable_id(pdf_id, page_number, page_text[:200])}",
                    "pdf_id": pdf_id,
                    "file_name": file_name,
                    "source_path": str(pdf_path),
                    "page": page_number,
                    "element_type": "page_text",
                    "text": page_text,
                    "metadata": {
                        "char_count": len(page_text),
                        "native_word_count": len(words),
                        "ocr_applied": bool(ocr.get("applied")),
                    },
                })

            for heading in [ln for ln in lines if ln.get("type") == "heading"]:
                units.append({
                    "unit_id": f"heading_{stable_id(pdf_id, page_number, heading.get('id'), heading.get('text'))}",
                    "pdf_id": pdf_id,
                    "file_name": file_name,
                    "source_path": str(pdf_path),
                    "page": page_number,
                    "element_type": "heading",
                    "text": heading.get("text", ""),
                    "metadata": {"bbox": heading.get("bbox"), "avg_font_size": heading.get("avg_font_size")},
                })

            if ocr.get("applied") and ocr.get("text"):
                units.append({
                    "unit_id": f"ocr_{stable_id(pdf_id, page_number, ocr.get('text', '')[:160])}",
                    "pdf_id": pdf_id,
                    "file_name": file_name,
                    "source_path": str(pdf_path),
                    "page": page_number,
                    "element_type": "ocr_text",
                    "text": ocr["text"],
                    "metadata": {
                        "reason": ocr.get("reason"),
                        "image_coverage": ocr.get("image_coverage"),
                    },
                })

            captions = [ln for ln in lines if ln.get("type") == "caption"]
            for cap in captions:
                cap_text = cap.get("text", "")
                if not cap_text:
                    continue
                cap_type = "figure_caption" if re.match(r"^\s*fig", cap_text, re.I) else "table_caption"
                units.append({
                    "unit_id": f"caption_{stable_id(pdf_id, page_number, cap.get('id'), cap_text[:120])}",
                    "pdf_id": pdf_id,
                    "file_name": file_name,
                    "source_path": str(pdf_path),
                    "page": page_number,
                    "element_type": cap_type,
                    "text": cap_text,
                    "metadata": {"bbox": cap.get("bbox")},
                })

            tables = extract_tables(page, page_index, tables_dir, paper_out_dir)
            for table in tables:
                attach_nearest_caption(table, lines, kind="table")
                caption_text = ((table.get("caption") or {}).get("text") or "").strip()
                payload = "\n\n".join(x for x in [caption_text, table.get("markdown", "")] if x).strip()
                if payload:
                    units.append({
                        "unit_id": f"table_{stable_id(pdf_id, page_number, table['id'])}",
                        "pdf_id": pdf_id,
                        "file_name": file_name,
                        "source_path": str(pdf_path),
                        "page": page_number,
                        "element_type": "table",
                        "text": payload,
                        "metadata": {
                            "bbox": table.get("bbox"),
                            "row_count": table.get("row_count"),
                            "col_count": table.get("col_count"),
                            "files": table.get("files", {}),
                        },
                    })

            equations = detect_equations(lines, pdf_path, page_index, page.width, page.height, equations_dir, paper_out_dir)
            for eq in equations:
                attach_nearest_caption(eq, lines, kind="equation")
                payload = "\n\n".join(x for x in [eq.get("text", ""), ((eq.get("caption") or {}).get("text") or "")] if x).strip()
                if payload:
                    units.append({
                        "unit_id": f"equation_{stable_id(pdf_id, page_number, eq['id'])}",
                        "pdf_id": pdf_id,
                        "file_name": file_name,
                        "source_path": str(pdf_path),
                        "page": page_number,
                        "element_type": "equation_candidate",
                        "text": payload,
                        "metadata": {
                            "bbox": eq.get("bbox"),
                            "crop_file": eq.get("crop_file"),
                            "latex": eq.get("latex"),
                        },
                    })

            if not FAST_FIGURE_MODE or figure_hint:
                figures = extract_figures_and_graphics(page, pdf_path, page_index, tables, equations, lines, figures_dir, paper_out_dir)
            else:
                figures = []
            for fig in figures:
                nearby = ((fig.get("caption") or {}).get("text") or "") or page_text[:1200]
                vision_desc = ""
                if USE_VISION_FOR_FIGURES and fig.get("crop_file"):
                    crop_path = paper_out_dir / fig["crop_file"]
                    if crop_path.exists():
                        vision_desc = describe_image_with_openai(crop_path, nearby)
                payload_parts = [
                    ((fig.get("caption") or {}).get("text") or "").strip(),
                    clean_text(vision_desc),
                    fig.get("note", ""),
                ]
                payload = "\n\n".join(x for x in payload_parts if x).strip()
                if payload:
                    units.append({
                        "unit_id": f"figure_{stable_id(pdf_id, page_number, fig['id'])}",
                        "pdf_id": pdf_id,
                        "file_name": file_name,
                        "source_path": str(pdf_path),
                        "page": page_number,
                        "element_type": "figure_crop",
                        "text": payload,
                        "metadata": {
                            "bbox": fig.get("bbox"),
                            "crop_file": fig.get("crop_file"),
                        },
                    })

            if EXTRACT_EMBEDDED_IMAGES:
                embedded_images = extract_embedded_images_with_pypdf(pdf_path, page_index, embedded_dir, paper_out_dir)
                if MAX_EMBEDDED_IMAGES_PER_PAGE:
                    embedded_images = embedded_images[:MAX_EMBEDDED_IMAGES_PER_PAGE]
                for img in embedded_images:
                    file_rel = img.get("file")
                    if not file_rel:
                        continue
                    image_path = paper_out_dir / file_rel
                    try:
                        image_bytes = image_path.read_bytes()
                    except Exception:
                        continue
                    if len(image_bytes) < FIGURE_MIN_IMAGE_BYTES:
                        continue
                    content_hash = hashlib.sha1(image_bytes).hexdigest()[:20]
                    if IMAGE_DEDUP_BY_HASH and content_hash in seen_embedded_hashes:
                        continue
                    seen_embedded_hashes.add(content_hash)
                    payload = "\n\n".join(x for x in [
                        "Embedded image extracted from the PDF.",
                        "\n".join(ln.get("text", "") for ln in captions[:2]),
                    ] if x).strip()
                    units.append({
                        "unit_id": f"embedded_{stable_id(pdf_id, page_number, img.get('id'), content_hash)}",
                        "pdf_id": pdf_id,
                        "file_name": file_name,
                        "source_path": str(pdf_path),
                        "page": page_number,
                        "element_type": "figure_image",
                        "text": payload,
                        "metadata": {"file": file_rel, "bytes": len(image_bytes), "name": img.get("name")},
                    })

            if EXPORT_LINK_UNITS:
                for link in extract_links(page):
                    uri = clean_text(link.get("uri"))
                    if not uri:
                        continue
                    units.append({
                        "unit_id": f"link_{stable_id(pdf_id, page_number, link['id'], uri)}",
                        "pdf_id": pdf_id,
                        "file_name": file_name,
                        "source_path": str(pdf_path),
                        "page": page_number,
                        "element_type": "link",
                        "text": uri,
                        "metadata": {"bbox": link.get("bbox")},
                    })
    return units

## 5b. Open-source PDF-to-LLM extraction layer with audit-ready candidates

This cell overrides `extract_pdf_units(...)` so the HybridRAG pipeline uses the uploaded `pdf_to_llm_open_source.py` functions as the extraction backbone.

It writes an LLM-friendly learning pack for every PDF, including:

- raw page text, OCR status, words/lines, tables, figure/graphic candidates, embedded images, links, and equation candidates;
- candidate-level provenance: page number, bounding boxes, crop paths, captions, nearby context, and source method;
- explicit audit fields so a later local LLM can mark false positives, discover false negatives, and avoid treating uncertain extraction candidates as paper facts;
- optional OpenAI audit functions that can classify candidates and propose missing items using your OpenAI API key.


In [6]:

# -----------------------------------------------------------------------------
# Open-source PDF-to-LLM extraction adapter for HybridRAG
# -----------------------------------------------------------------------------
# This cell intentionally redefines extract_pdf_units(...) after the original
# notebook extraction helpers. The rest of the notebook can stay unchanged.

import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Make the sidecar module importable when the notebook and module are in the same folder.
for _candidate in [Path.cwd(), Path.cwd().parent, Path('/mnt/data')]:
    if str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

try:
    from pdf_to_llm_open_source import (
        PipelineConfig as PDFToLLMConfig,
        process_pdf as pdf_to_llm_process_pdf,
        write_json as pdf_to_llm_write_json,
        write_text as pdf_to_llm_write_text,
        clean_text as pdf_to_llm_clean_text,
    )
except Exception as exc:
    raise ImportError(
        "Could not import pdf_to_llm_open_source.py. Put the file next to this notebook "
        "or add its folder to PYTHONPATH, then rerun this cell. Original error: " + repr(exc)
    )


def _jsonl_write(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def _safe_rel(path_like: Optional[str], root: Path) -> Optional[str]:
    if not path_like:
        return None
    try:
        p = Path(path_like)
        if not p.is_absolute():
            return str(p).replace('\\', '/')
        return str(p.resolve().relative_to(root.resolve())).replace('\\', '/')
    except Exception:
        return str(path_like).replace('\\', '/')




def _sha256_local(path: Path, block_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()


def _find_existing_pdf_to_llm_document(extraction_root: Path, pdf_path: Path) -> Optional[Path]:
    """Return an existing document.json only when it matches this exact PDF hash."""
    if not REUSE_EXISTING_EXTRACTION or FORCE_REBUILD_ALL:
        return None
    try:
        target_hash = _sha256_local(pdf_path)
        target_resolved = str(Path(pdf_path).resolve())
        for candidate in sorted(Path(extraction_root).glob("*/document.json")):
            try:
                doc = json.loads(candidate.read_text(encoding="utf-8"))
                if doc.get("sha256") == target_hash and str(Path(doc.get("source_path", "")).resolve()) == target_resolved:
                    return candidate
            except Exception:
                continue
    except Exception:
        return None
    return None

def _bbox_center_y(bbox: Optional[Dict[str, Any]]) -> Optional[float]:
    if not bbox:
        return None
    try:
        return (float(bbox['top']) + float(bbox['bottom'])) / 2.0
    except Exception:
        return None


def _nearby_line_context(page: Dict[str, Any], bbox: Optional[Dict[str, Any]], max_lines: int = MAX_AUDIT_CONTEXT_LINES) -> Dict[str, Any]:
    lines = page.get('lines', []) or []
    if not lines:
        return {'before': [], 'after': [], 'nearest': []}
    cy = _bbox_center_y(bbox)
    if cy is None:
        nearest = lines[:max_lines]
    else:
        def dist(line: Dict[str, Any]) -> float:
            return abs((_bbox_center_y(line.get('bbox')) or 0.0) - cy)
        nearest = sorted(lines, key=dist)[:max_lines]
        nearest = sorted(nearest, key=lambda ln: (ln.get('bbox') or {}).get('top', 0))
    return {
        'nearest': [
            {
                'line_id': ln.get('id'),
                'type': ln.get('type'),
                'text': ln.get('text'),
                'bbox': ln.get('bbox'),
            }
            for ln in nearest
        ]
    }


_MATH_TOKEN_RE = re.compile(r"(\\[A-Za-z]+|[=<>≤≥±≈≠∞∑∫√∂∇πθλμσΩαβγδ]|\^|_|\bfrac\b|\bsum\b|\bint\b)")
_ONLY_SYMBOLS_RE = re.compile(r"^[\W_]+$", re.UNICODE)


def _candidate_heuristic(kind: str, text: str, record: Dict[str, Any]) -> Dict[str, Any]:
    """Cheap, deterministic triage. This never deletes data; it only flags items for review."""
    t = pdf_to_llm_clean_text(text)
    flags: List[str] = []
    suggested = 'review'
    confidence = 0.5

    if not t:
        flags.append('empty_text')
        suggested = 'drop_candidate'
        confidence = 0.85

    if kind == 'equation':
        math_hits = len(_MATH_TOKEN_RE.findall(t))
        digit_hits = sum(ch.isdigit() for ch in t)
        alpha_words = re.findall(r"[A-Za-z]{4,}", t)
        if len(t) <= 2:
            flags.append('too_short_for_equation')
        if _ONLY_SYMBOLS_RE.match(t) and len(t) < 8:
            flags.append('only_special_characters')
        if math_hits == 0:
            flags.append('no_math_operator_or_symbol')
        if len(alpha_words) >= 7 and math_hits <= 1:
            flags.append('mostly_sentence_text')
        if re.match(r"^\s*(figure|fig\.?|table|tab\.?)\b", t, re.I):
            flags.append('looks_like_caption_not_equation')
        if math_hits >= 2 or ('=' in t and (digit_hits > 0 or len(t) <= 120)):
            suggested = 'keep_candidate'
            confidence = 0.7
        if any(flag in flags for flag in ['only_special_characters', 'no_math_operator_or_symbol', 'looks_like_caption_not_equation']):
            suggested = 'review_or_drop_candidate'
            confidence = max(confidence, 0.7)

    elif kind == 'table':
        rows = record.get('row_count') or 0
        cols = record.get('col_count') or 0
        if rows < 2 or cols < 2:
            flags.append('very_small_table_candidate')
            suggested = 'review_or_drop_candidate'
            confidence = 0.65
        else:
            suggested = 'keep_candidate'
            confidence = 0.7

    elif kind == 'figure':
        if not record.get('caption') and not record.get('crop_file'):
            flags.append('no_caption_or_crop')
            suggested = 'review_or_drop_candidate'
        else:
            suggested = 'keep_candidate'
            confidence = 0.65

    return {
        'heuristic_suggestion': suggested,
        'heuristic_confidence': round(float(confidence), 3),
        'heuristic_flags': flags,
        'important_note': 'Heuristics are not final. Keep/drop decisions should be made by a reviewer or local LLM using the provided page context and crops.',
    }


def _review_schema(kind: str) -> Dict[str, Any]:
    return {
        'allowed_decisions': ['keep', 'drop_false_positive', 'merge_duplicate', 'partial_keep', 'uncertain_needs_visual_review'],
        'false_positive_examples': {
            'equation': [
                'standalone punctuation or special characters',
                'a citation number, page number, or table/figure label',
                'plain prose that contains one mathematical symbol but is not an equation',
                'units or ranges extracted without a mathematical relationship',
            ],
            'table': [
                'layout columns mistaken for a table',
                'a caption or reference list block mistaken for tabular data',
                'a one-row/one-column fragment without table structure',
            ],
            'figure': [
                'decorative lines or page separators',
                'a table border or equation crop mistaken for a figure',
                'a tiny logo or icon that is not scientifically meaningful',
            ],
        }.get(kind, []),
        'false_negative_instruction': 'Also inspect page_text, line_context, captions, and crop/page assets to identify missing equations, tables, figures, or captions that the extractor failed to mark.',
        'decision_json_schema': {
            'candidate_id': 'string',
            'decision': 'keep | drop_false_positive | merge_duplicate | partial_keep | uncertain_needs_visual_review',
            'corrected_type': 'equation | table | figure | text | other | null',
            'corrected_text_or_latex': 'string or null',
            'reason': 'brief evidence-based reason',
            'confidence': 'number from 0 to 1',
        },
    }


def _make_candidate(
    *,
    doc: Dict[str, Any],
    page: Dict[str, Any],
    kind: str,
    record: Dict[str, Any],
    text: str,
    extraction_root: Path,
) -> Dict[str, Any]:
    candidate_id = f"{doc['document_id']}::p{page['page_number']:04d}::{kind}::{record.get('id', stable_id(text))}"
    bbox = record.get('bbox')
    context = _nearby_line_context(page, bbox)
    crop = record.get('crop_file') or record.get('file')
    candidate = {
        'candidate_id': candidate_id,
        'document_id': doc.get('document_id'),
        'file_name': doc.get('file_name'),
        'page': page.get('page_number'),
        'candidate_type': kind,
        'source_record_id': record.get('id'),
        'source_method': record.get('source'),
        'text': pdf_to_llm_clean_text(text),
        'caption': record.get('caption'),
        'bbox': bbox,
        'asset_path': _safe_rel(crop, extraction_root) if crop else None,
        'line_context': context,
        'audit': {
            'status': 'unreviewed',
            'is_candidate_not_final_fact': True,
            'heuristic': _candidate_heuristic(kind, text, record),
            'review_schema': _review_schema(kind),
            'local_llm_decision': None,
        },
    }
    return candidate


def _candidate_unit_text(candidate: Dict[str, Any]) -> str:
    kind = candidate['candidate_type'].upper()
    ctx_lines = candidate.get('line_context', {}).get('nearest', [])
    ctx = '\n'.join(f"- {ln.get('text', '')}" for ln in ctx_lines if ln.get('text'))
    cap = candidate.get('caption') or {}
    cap_text = cap.get('text') if isinstance(cap, dict) else ''
    heur = candidate.get('audit', {}).get('heuristic', {})
    parts = [
        f"UNVERIFIED {kind} CANDIDATE - REVIEW BEFORE USING AS A PAPER FACT",
        f"Candidate ID: {candidate.get('candidate_id')}",
        f"PDF: {candidate.get('file_name')} | page: {candidate.get('page')}",
        f"Extractor text:\n{candidate.get('text') or ''}",
    ]
    if cap_text:
        parts.append(f"Nearest caption:\n{cap_text}")
    if candidate.get('asset_path'):
        parts.append(f"Visual crop/asset path: {candidate.get('asset_path')}")
    if ctx:
        parts.append(f"Nearby page context:\n{ctx}")
    parts.append("Audit guidance: classify this candidate as keep, drop_false_positive, merge_duplicate, partial_keep, or uncertain_needs_visual_review. Also check whether the page contains missing items of this type that were not extracted.")
    parts.append(f"Heuristic triage: {json.dumps(heur, ensure_ascii=False)}")
    return '\n\n'.join(parts).strip()


def _page_false_negative_task(doc: Dict[str, Any], page: Dict[str, Any], candidates: List[Dict[str, Any]]) -> Dict[str, Any]:
    page_text = pdf_to_llm_clean_text(page.get('text', ''))[:MAX_AUDIT_PAGE_TEXT_CHARS]
    candidate_summary = [
        {
            'candidate_id': c['candidate_id'],
            'type': c['candidate_type'],
            'text': c.get('text', '')[:500],
            'heuristic': c.get('audit', {}).get('heuristic', {}).get('heuristic_suggestion'),
        }
        for c in candidates
    ]
    lines = [
        {'line_id': ln.get('id'), 'type': ln.get('type'), 'text': ln.get('text'), 'bbox': ln.get('bbox')}
        for ln in (page.get('lines') or [])[:250]
    ]
    return {
        'task_id': f"{doc['document_id']}::p{page['page_number']:04d}::false_negative_review",
        'document_id': doc.get('document_id'),
        'file_name': doc.get('file_name'),
        'page': page.get('page_number'),
        'task_type': 'page_false_positive_false_negative_review',
        'instructions': [
            'Use only the provided extracted page text, line list, candidates, and visual asset paths.',
            'Mark extracted candidates that are false positives.',
            'List false negatives: equations, tables, figures, captions, or important text that appear in the page evidence but are missing from the candidate list.',
            'Do not invent content that is not supported by this page evidence.',
        ],
        'candidate_summary': candidate_summary,
        'page_text': page_text,
        'page_lines': lines,
        'page_quality': page.get('quality', {}),
        'expected_output_schema': {
            'candidate_decisions': [_review_schema('equation')['decision_json_schema']],
            'false_negatives': [
                {
                    'missing_type': 'equation | table | figure | caption | text | other',
                    'evidence_text': 'exact extracted words/lines supporting the missing item',
                    'suggested_content': 'corrected text, markdown table, latex, or description',
                    'source_line_ids': ['line ids when available'],
                    'reason': 'why this is missing',
                    'confidence': '0 to 1',
                }
            ],
        },
    }


def build_llm_learning_pack_from_document(
    doc: Dict[str, Any],
    extraction_root: Path,
    paper_out_dir: Path,
    *,
    include_audit_tasks_in_rag: bool = INCLUDE_EXTRACTION_AUDIT_TASKS_IN_RAG,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Convert pdf_to_llm_open_source document.json into HybridRAG units and audit files."""
    pdf_id = doc.get('document_id') or stable_id(doc.get('source_path'), doc.get('sha256'))
    file_name = doc.get('file_name')
    source_path = doc.get('source_path')

    units: List[Dict[str, Any]] = []
    candidates: List[Dict[str, Any]] = []
    review_tasks: List[Dict[str, Any]] = []

    units.append({
        'unit_id': f"docmeta_{pdf_id}",
        'pdf_id': pdf_id,
        'file_name': file_name,
        'source_path': source_path,
        'page': None,
        'element_type': 'document_metadata',
        'text': json.dumps({
            'meta': doc.get('meta', {}),
            'sha256': doc.get('sha256'),
            'extraction_policy': doc.get('extraction_policy'),
            'llm_learning_pack_note': 'Structured candidates are audit-ready. Candidate elements are not final facts until reviewed.',
        }, ensure_ascii=False),
        'metadata': {'document_id': pdf_id, 'sha256': doc.get('sha256')},
    })

    for page in doc.get('pages', []):
        page_num = page.get('page_number')
        page_text = pdf_to_llm_clean_text(page.get('text', ''))
        if page_text:
            units.append({
                'unit_id': f"text_{stable_id(pdf_id, page_num, page_text[:200])}",
                'pdf_id': pdf_id,
                'file_name': file_name,
                'source_path': source_path,
                'page': page_num,
                'element_type': 'page_text',
                'text': page_text,
                'metadata': {
                    'document_id': pdf_id,
                    'quality': page.get('quality', {}),
                    'ocr': {
                        'needed': (page.get('ocr') or {}).get('needed'),
                        'applied': (page.get('ocr') or {}).get('applied'),
                        'reason': (page.get('ocr') or {}).get('reason'),
                    },
                    'llm_note': 'Native/OCR page text. Use candidate records for structured elements and audit status.',
                },
            })

        for line in page.get('lines', []) or []:
            if line.get('type') in {'heading', 'caption'} and pdf_to_llm_clean_text(line.get('text', '')):
                element_type = 'heading' if line.get('type') == 'heading' else (
                    'figure_caption' if re.match(r"^\s*fig", line.get('text', ''), re.I) else 'table_caption'
                )
                units.append({
                    'unit_id': f"{element_type}_{stable_id(pdf_id, page_num, line.get('id'), line.get('text'))}",
                    'pdf_id': pdf_id,
                    'file_name': file_name,
                    'source_path': source_path,
                    'page': page_num,
                    'element_type': element_type,
                    'text': line.get('text', ''),
                    'metadata': {'bbox': line.get('bbox'), 'line_id': line.get('id'), 'audit_status': 'direct_line_extraction'},
                })

        page_candidates: List[Dict[str, Any]] = []

        for table in page.get('tables', []) or []:
            cap_text = ((table.get('caption') or {}).get('text') or '').strip()
            payload = '\n\n'.join(x for x in [cap_text, table.get('markdown', '')] if x).strip()
            cand = _make_candidate(doc=doc, page=page, kind='table', record=table, text=payload, extraction_root=extraction_root)
            candidates.append(cand); page_candidates.append(cand)
            units.append({
                'unit_id': f"table_{stable_id(pdf_id, page_num, table.get('id'))}",
                'pdf_id': pdf_id,
                'file_name': file_name,
                'source_path': source_path,
                'page': page_num,
                'element_type': 'table',
                'text': _candidate_unit_text(cand),
                'metadata': {
                    'candidate_id': cand['candidate_id'],
                    'is_extraction_candidate': True,
                    'candidate_type': 'table',
                    'audit': cand['audit'],
                    'bbox': table.get('bbox'),
                    'row_count': table.get('row_count'),
                    'col_count': table.get('col_count'),
                    'files': table.get('files', {}),
                    'matrix': table.get('matrix'),
                },
            })

        for eq in page.get('equations', []) or []:
            eq_text = eq.get('latex') or eq.get('text') or ''
            cand = _make_candidate(doc=doc, page=page, kind='equation', record=eq, text=eq_text, extraction_root=extraction_root)
            candidates.append(cand); page_candidates.append(cand)
            units.append({
                'unit_id': f"equation_{stable_id(pdf_id, page_num, eq.get('id'))}",
                'pdf_id': pdf_id,
                'file_name': file_name,
                'source_path': source_path,
                'page': page_num,
                'element_type': 'equation_candidate',
                'text': _candidate_unit_text(cand),
                'metadata': {
                    'candidate_id': cand['candidate_id'],
                    'is_extraction_candidate': True,
                    'candidate_type': 'equation',
                    'audit': cand['audit'],
                    'bbox': eq.get('bbox'),
                    'crop_file': eq.get('crop_file'),
                    'latex': eq.get('latex'),
                },
            })

        for fig in page.get('figures', []) or []:
            cap_text = ((fig.get('caption') or {}).get('text') or '').strip()
            payload = '\n\n'.join(x for x in [cap_text, fig.get('note', '')] if x).strip() or 'Figure/graphic candidate with crop asset.'
            cand = _make_candidate(doc=doc, page=page, kind='figure', record=fig, text=payload, extraction_root=extraction_root)
            candidates.append(cand); page_candidates.append(cand)
            units.append({
                'unit_id': f"figure_{stable_id(pdf_id, page_num, fig.get('id'))}",
                'pdf_id': pdf_id,
                'file_name': file_name,
                'source_path': source_path,
                'page': page_num,
                'element_type': 'figure_crop',
                'text': _candidate_unit_text(cand),
                'metadata': {
                    'candidate_id': cand['candidate_id'],
                    'is_extraction_candidate': True,
                    'candidate_type': 'figure',
                    'audit': cand['audit'],
                    'bbox': fig.get('bbox'),
                    'crop_file': fig.get('crop_file'),
                },
            })

        for img in page.get('embedded_images', []) or []:
            if not img.get('file'):
                continue
            text = 'Embedded image extracted from the PDF. Review visually before treating as a scientific figure.'
            units.append({
                'unit_id': f"embedded_{stable_id(pdf_id, page_num, img.get('id'), img.get('file'))}",
                'pdf_id': pdf_id,
                'file_name': file_name,
                'source_path': source_path,
                'page': page_num,
                'element_type': 'figure_image',
                'text': text,
                'metadata': {'file': img.get('file'), 'bytes': img.get('bytes'), 'name': img.get('name'), 'is_extraction_candidate': True},
            })

        for link in page.get('links', []) or []:
            uri = pdf_to_llm_clean_text(link.get('uri', ''))
            if uri:
                units.append({
                    'unit_id': f"link_{stable_id(pdf_id, page_num, link.get('id'), uri)}",
                    'pdf_id': pdf_id,
                    'file_name': file_name,
                    'source_path': source_path,
                    'page': page_num,
                    'element_type': 'link',
                    'text': uri,
                    'metadata': {'bbox': link.get('bbox')},
                })

        task = _page_false_negative_task(doc, page, page_candidates)
        review_tasks.append(task)
        if include_audit_tasks_in_rag:
            units.append({
                'unit_id': task['task_id'],
                'pdf_id': pdf_id,
                'file_name': file_name,
                'source_path': source_path,
                'page': page_num,
                'element_type': 'extraction_audit_page',
                'text': json.dumps(task, ensure_ascii=False),
                'metadata': {'not_a_paper_fact': True, 'task_type': task['task_type']},
            })

    pack = {
        'schema_version': '2.0.0-llm-learning-audit-ready',
        'document_id': pdf_id,
        'file_name': file_name,
        'source_path': source_path,
        'sha256': doc.get('sha256'),
        'extraction_policy': doc.get('extraction_policy', {}),
        'important_usage_rule': 'Candidate equations/tables/figures are unverified. A later LLM or human should use candidate_inventory and review_tasks to remove false positives and identify false negatives before treating structured elements as final facts.',
        'source_document_json': str((extraction_root / 'manifest.json').parent) if extraction_root else None,
        'candidate_inventory': candidates,
        'false_positive_false_negative_review_tasks': review_tasks,
        'hybridrag_unit_count': len(units),
        'candidate_counts': {
            'equation': sum(1 for c in candidates if c['candidate_type'] == 'equation'),
            'table': sum(1 for c in candidates if c['candidate_type'] == 'table'),
            'figure': sum(1 for c in candidates if c['candidate_type'] == 'figure'),
        },
    }
    return units, pack


def _write_learning_pack_files(paper_out_dir: Path, pack: Dict[str, Any], units: List[Dict[str, Any]]) -> Dict[str, str]:
    learning_dir = paper_out_dir / 'llm_learning_pack'
    learning_dir.mkdir(parents=True, exist_ok=True)
    pack_path = learning_dir / 'llm_learning_document.json'
    candidates_path = learning_dir / 'candidate_inventory.jsonl'
    review_tasks_path = learning_dir / 'extraction_review_tasks.jsonl'
    units_path = learning_dir / 'hybridrag_units_audit_ready.jsonl'
    md_path = learning_dir / 'README_llm_learning_pack.md'

    pdf_to_llm_write_json(pack_path, pack)
    _jsonl_write(candidates_path, pack.get('candidate_inventory', []))
    _jsonl_write(review_tasks_path, pack.get('false_positive_false_negative_review_tasks', []))
    _jsonl_write(units_path, units)

    md = f"""# LLM learning pack for {pack.get('file_name')}

This folder contains extraction output designed for later LLM reading and audit.

## Important rule

Candidate equations, tables, figures, and embedded images are **not automatically final facts**. Use `candidate_inventory.jsonl` and `extraction_review_tasks.jsonl` to classify false positives and search for false negatives.

## Files

- `llm_learning_document.json`: complete audit-ready package.
- `candidate_inventory.jsonl`: one candidate per row, with page, bbox, crop path, nearby context, and heuristic triage.
- `extraction_review_tasks.jsonl`: one page-level review task per page for false positive and false negative detection.
- `hybridrag_units_audit_ready.jsonl`: the units sent into HybridRAG.

## Candidate counts

```json
{json.dumps(pack.get('candidate_counts', {}), ensure_ascii=False, indent=2)}
```
"""
    pdf_to_llm_write_text(md_path, md)
    return {
        'learning_pack_dir': str(learning_dir),
        'learning_pack_json': str(pack_path),
        'candidate_inventory_jsonl': str(candidates_path),
        'review_tasks_jsonl': str(review_tasks_path),
        'audit_ready_units_jsonl': str(units_path),
        'learning_pack_readme': str(md_path),
    }


def audit_learning_pack_with_openai(
    learning_pack_json: Path,
    model: str = EXTRACTION_AUDIT_MODEL,
    max_pages: Optional[int] = None,
) -> Dict[str, Any]:
    """Optional OpenAI-backed extraction audit. Uses your OpenAI API key."""
    learning_pack_json = Path(learning_pack_json)
    pack = json.loads(learning_pack_json.read_text(encoding='utf-8'))
    tasks = pack.get('false_positive_false_negative_review_tasks', [])
    if max_pages is not None:
        tasks = tasks[:max_pages]

    out_dir = learning_pack_json.parent
    decisions_path = out_dir / 'openai_extraction_audit.jsonl'
    summary_path = out_dir / 'openai_extraction_audit_summary.json'

    if not openai_available():
        summary = {'ok': False, 'error': 'OPENAI_API_KEY is not set', 'model': model}
        pdf_to_llm_write_json(summary_path, summary)
        return summary

    system = (
        'You are a strict PDF extraction auditor. Use only the supplied extracted page evidence. '
        'Your job is to mark false positives and find false negatives. Return valid JSON only.'
    )
    rows: List[Dict[str, Any]] = []
    for task in tqdm(tasks, desc='OpenAI extraction audit', leave=False):
        prompt = f"""
Audit this one PDF page extraction.

Rules:
- Use only the page evidence below.
- A candidate can be a false positive. Example: an equation candidate may be only punctuation, special characters, a unit range, a citation, or prose rather than an equation.
- Also find false negatives: equations, tables, figures, captions, or important text visible in the extracted lines/page text but missing from candidate_summary.
- Do not invent content.
- Return JSON with keys: page, candidate_decisions, false_negatives, notes.

PAGE TASK JSON:
{json.dumps(task, ensure_ascii=False)}
""".strip()
        try:
            raw = openai_generate(prompt=prompt, model=model, system=system, format_json=True, temperature=0.0, timeout=300)
            parsed = json.loads(raw)
        except Exception as exc:
            parsed = {'page': task.get('page'), 'error': repr(exc), 'raw_response': locals().get('raw', '')}
        rows.append(parsed)

    _jsonl_write(decisions_path, rows)
    summary = {
        'ok': True,
        'model': model,
        'learning_pack_json': str(learning_pack_json),
        'decisions_jsonl': str(decisions_path),
        'pages_audited': len(rows),
        'decision_file_note': 'Use these decisions to drop false positives or add false negatives before final downstream use.',
    }
    pdf_to_llm_write_json(summary_path, summary)
    return summary


def _filter_units_with_audit(units: List[Dict[str, Any]], audit_decisions_jsonl: Optional[Path]) -> List[Dict[str, Any]]:
    """Optional: remove candidates marked drop_false_positive by a local audit."""
    if not audit_decisions_jsonl or not Path(audit_decisions_jsonl).exists() or not DROP_LLM_AUDIT_FALSE_POSITIVES_FROM_RAG:
        return units
    drop_ids = set()
    with Path(audit_decisions_jsonl).open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            for decision in row.get('candidate_decisions', []) or []:
                if decision.get('decision') == 'drop_false_positive' and decision.get('candidate_id'):
                    drop_ids.add(decision['candidate_id'])
    if not drop_ids:
        return units
    return [u for u in units if (u.get('metadata', {}) or {}).get('candidate_id') not in drop_ids]


def extract_pdf_units(pdf_path: Path, paper_out_dir: Path) -> List[Dict[str, Any]]:
    """HybridRAG extraction entrypoint backed by pdf_to_llm_open_source.py."""
    if not USE_PDF_TO_LLM_OPEN_SOURCE:
        raise RuntimeError('USE_PDF_TO_LLM_OPEN_SOURCE is False, but the override extractor is active.')

    pdf_path = Path(pdf_path)
    paper_out_dir = Path(paper_out_dir)
    extraction_root = paper_out_dir / PDF_TO_LLM_EXTRACTION_SUBDIR
    extraction_root.mkdir(parents=True, exist_ok=True)

    cfg = PDFToLLMConfig(
        input_dir=PDF_DIR,
        output_dir=extraction_root,
        recursive=True,
        dpi=PDF_RENDER_DPI,
        save_page_images=SAVE_PAGE_RENDERS,
        save_crops=SAVE_EXTRACTION_CROPS,
        crop_padding_points=FIGURE_CROP_PADDING_POINTS,
        enable_ocr=OCR_ENABLED,
        ocr_language=OCR_LANGUAGE,
        min_native_words_for_no_ocr=MIN_NATIVE_WORDS_FOR_NO_OCR,
        scanned_image_coverage_threshold=SCANNED_IMAGE_COVERAGE_THRESHOLD,
        enable_embedded_image_export=EXTRACT_EMBEDDED_IMAGES,
        min_figure_area_fraction=MIN_FIGURE_AREA_FRACTION,
        merge_graphics_margin_points=MERGE_GRAPHICS_MARGIN_POINTS,
        equation_crop_padding_points=EQUATION_CROP_PADDING_POINTS,
        chunk_chars=CHUNK_SIZE_CHARS,
        chunk_overlap_chars=CHUNK_OVERLAP_CHARS,
    )

    existing_document_json = _find_existing_pdf_to_llm_document(extraction_root, pdf_path)
    if existing_document_json is not None:
        document_json = existing_document_json
        result = {
            'document_id': json.loads(document_json.read_text(encoding='utf-8')).get('document_id'),
            'file_name': pdf_path.name,
            'source_path': str(pdf_path),
            'output_folder': str(document_json.parent),
            'document_json': str(document_json),
            'document_markdown': str(document_json.parent / 'document_llm.md'),
            'chunks_jsonl': str(document_json.parent / 'chunks.jsonl'),
            'reused_existing_extraction': True,
        }
    else:
        result = pdf_to_llm_process_pdf(pdf_path, cfg)
        document_json = Path(result['document_json'])
    doc = json.loads(document_json.read_text(encoding='utf-8'))

    units, pack = build_llm_learning_pack_from_document(
        doc,
        extraction_root=extraction_root,
        paper_out_dir=paper_out_dir,
        include_audit_tasks_in_rag=INCLUDE_EXTRACTION_AUDIT_TASKS_IN_RAG,
    )
    paths = _write_learning_pack_files(paper_out_dir, pack, units)

    # Add pointers to generated learning/audit files to the document metadata unit.
    for unit in units:
        if unit.get('element_type') == 'document_metadata':
            unit.setdefault('metadata', {})['pdf_to_llm_result'] = result
            unit['metadata']['llm_learning_pack_files'] = paths
            break

    if RUN_OPENAI_EXTRACTION_AUDIT:
        audit_summary = audit_learning_pack_with_openai(Path(paths['learning_pack_json']), model=EXTRACTION_AUDIT_MODEL)
        for unit in units:
            if unit.get('element_type') == 'document_metadata':
                unit.setdefault('metadata', {})['local_llm_audit_summary'] = audit_summary
                break
        decisions = Path(paths['learning_pack_dir']) / 'openai_extraction_audit.jsonl'
        units = _filter_units_with_audit(units, decisions)

    return units


## 5c. Verified equation-to-LaTeX layer

This layer runs after the PDF-to-LLM extraction adapter. It keeps the original audit-ready candidates, then adds a second equation-specific pass that:

- verifies whether each equation candidate is a real mathematical equation or a false positive,
- scans extracted page text for likely missed equations (false negatives),
- converts verified equations to LaTeX,
- writes `llm_learning_pack/equations_latex.jsonl`, `equation_audit_records.jsonl`, and `equations_latex.md`,
- injects verified `equation_latex` units into VectorRAG and GraphRAG so equation questions retrieve clean LaTeX evidence.


In [7]:

# -----------------------------------------------------------------------------
# Verified equation-to-LaTeX enrichment layer
# -----------------------------------------------------------------------------
# This wrapper runs after the PDF-to-LLM extraction adapter and before chunking.
# It stores verified equations as LaTeX and makes those LaTeX records first-class
# RAG units. It also scans page text to reduce false negatives.

from pathlib import Path
import json
import os
import re
from typing import Any, Dict, List, Optional, Tuple

# Defaults are set here as a safety net. They can also be set in the configuration cell.
ENABLE_EQUATION_LATEX_LAYER = globals().get("ENABLE_EQUATION_LATEX_LAYER", True)
RUN_LLM_EQUATION_LATEX_AUDIT = globals().get("RUN_LLM_EQUATION_LATEX_AUDIT", True)
EQUATION_LATEX_MODEL = globals().get("EQUATION_LATEX_MODEL", globals().get("EXTRACTION_AUDIT_MODEL", globals().get("GENERATION_MODEL", "")))
EQUATION_LATEX_USE_CROPS = globals().get("EQUATION_LATEX_USE_CROPS", True)
EQUATION_LATEX_MAX_CANDIDATES_PER_PDF = int(globals().get("EQUATION_LATEX_MAX_CANDIDATES_PER_PDF", 250))
DROP_EQUATION_FALSE_POSITIVES_FROM_RAG = globals().get("DROP_EQUATION_FALSE_POSITIVES_FROM_RAG", True)
ADD_EQUATION_FALSE_NEGATIVES_TO_RAG = globals().get("ADD_EQUATION_FALSE_NEGATIVES_TO_RAG", True)
EQUATION_LATEX_CONFIDENCE_KEEP_THRESHOLD = float(globals().get("EQUATION_LATEX_CONFIDENCE_KEEP_THRESHOLD", 0.55))
EQUATION_LATEX_STRONG_FP_SKIP_LLM = globals().get("EQUATION_LATEX_STRONG_FP_SKIP_LLM", True)


_LATEX_UNICODE_MAP = {
    "≤": r"\le", "≥": r"\ge", "≠": r"\ne", "≈": r"\approx", "±": r"\pm", "∞": r"\infty",
    "∈": r"\in", "∉": r"\notin", "∑": r"\sum", "∏": r"\prod", "∫": r"\int",
    "√": r"\sqrt{}", "∂": r"\partial", "∇": r"\nabla", "·": r"\cdot", "×": r"\times",
    "−": "-", "–": "-", "—": "-", "→": r"\to", "↦": r"\mapsto",
    "α": r"\alpha", "β": r"\beta", "γ": r"\gamma", "δ": r"\delta", "θ": r"\theta",
    "λ": r"\lambda", "μ": r"\mu", "µ": r"\mu", "σ": r"\sigma", "π": r"\pi",
    "Ω": r"\Omega", "Ψ": r"\Psi", "Θ": r"\Theta",
}


def _eq_norm_text(text: str) -> str:
    text = clean_text(str(text or ""))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _eq_dedupe_key(text: str) -> str:
    text = _eq_norm_text(text).lower()
    text = re.sub(r"[^a-z0-9=<>+\-*/^_().,\\]+", "", text)
    return text[:240]


def _special_character_only(text: str) -> bool:
    t = _eq_norm_text(text)
    if not t:
        return True
    alnum = sum(ch.isalnum() for ch in t)
    math_ops = len(re.findall(r"[=<>+\-*/^_]", t))
    return alnum == 0 or (alnum <= 1 and math_ops == 0)


def _equation_false_positive_reasons(text: str) -> List[str]:
    t = _eq_norm_text(text)
    low = t.lower()
    reasons: List[str] = []
    if not t:
        reasons.append("empty_text")
    if len(t) < 3:
        reasons.append("too_short")
    if len(t) > 650:
        reasons.append("too_long_for_standalone_equation")
    if _special_character_only(t):
        reasons.append("only_punctuation_or_special_characters")
    if re.search(r"https?://|doi\.org|@|www\.", low):
        reasons.append("url_doi_or_email")
    if re.match(r"^\s*(figure|fig\.|table|tab\.|reference|references|appendix|abstract|keywords)\b", low):
        reasons.append("caption_or_section_heading")
    if re.match(r"^\s*\[[0-9,\s-]+\]\s*$", low) or re.match(r"^\s*\([0-9,\s-]+\)\s*$", low):
        reasons.append("citation_or_reference_number")
    # Common table/header/unit fragments: numbers and units but no relation/operator.
    has_relation = bool(re.search(r"(=|≤|≥|<|>|≈|\\le|\\ge|\\approx|arg\s*max|arg\s*min)", t))
    has_operator = bool(re.search(r"[+\-*/^_∑∫√]|\\frac|\\sum|\\int|\\sqrt", t))
    has_letters = bool(re.search(r"[A-Za-zα-ωΑ-Ω]", t))
    if not has_relation and not has_operator and re.search(r"\b(nm|µm|um|mm|cm|m|kg|g|mg|wt%|bar|pa|mpa|kpa|hz|s|min|h|°c|k)\b", low):
        reasons.append("unit_or_range_without_equation_structure")
    if t.count("|") >= 2 and not has_relation:
        reasons.append("table_row_fragment")
    if len(re.findall(r"\b[a-z]{3,}\b", low)) >= 18 and not has_relation:
        reasons.append("prose_sentence_not_equation")
    return reasons


def _looks_like_equation_text(text: str) -> bool:
    t = _eq_norm_text(text)
    if not t:
        return False
    reasons = _equation_false_positive_reasons(t)
    strong_fp = {"empty_text", "too_short", "only_punctuation_or_special_characters", "url_doi_or_email", "caption_or_section_heading", "citation_or_reference_number"}
    if any(r in strong_fp for r in reasons):
        return False
    math_signal = bool(re.search(
        r"(\\frac|\\sum|\\prod|\\int|\\sqrt|\\partial|\\nabla|=|≤|≥|≈|≠|<|>|"
        r"\barg\s*max\b|\barg\s*min\b|\bexp\s*\(|\blog\s*\(|\bsin\s*\(|\bcos\s*\(|"
        r"∑|∏|∫|√|\^|_)",
        t,
        flags=re.I,
    ))
    variable_signal = bool(re.search(r"\b[A-Za-zα-ωΑ-Ω]\s*(?:=|≤|≥|≈|<|>)", t))
    numbered_eq_signal = bool(re.search(r"\(\s*\d+[a-z]?\s*\)\s*$", t)) and math_signal
    return bool(math_signal or variable_signal or numbered_eq_signal)


def _simple_text_to_latex(text: str) -> str:
    """Deterministic fallback conversion. It preserves evidence and normalizes common symbols."""
    t = _eq_norm_text(text)
    # Move trailing equation number outside the expression.
    t = re.sub(r"\s+\((\d+[a-z]?)\)\s*$", r" \tag{\1}", t)
    for src, dst in _LATEX_UNICODE_MAP.items():
        t = t.replace(src, " " + dst + " ")
    t = re.sub(r"\barg\s*max\b", r"\\arg\\max", t, flags=re.I)
    t = re.sub(r"\barg\s*min\b", r"\\arg\\min", t, flags=re.I)
    t = re.sub(r"\bexp\s*\(", r"\\exp(", t)
    t = re.sub(r"\blog\s*\(", r"\\log(", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def _scan_page_text_for_equation_false_negatives(units: List[Dict[str, Any]], existing_keys: set) -> List[Dict[str, Any]]:
    """Find likely missed equation lines from page text units."""
    found: List[Dict[str, Any]] = []
    if not ADD_EQUATION_FALSE_NEGATIVES_TO_RAG:
        return found
    for unit in units:
        if unit.get("element_type") != "page_text":
            continue
        page = unit.get("page")
        file_name = unit.get("file_name")
        text = str(unit.get("text") or "")
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        for idx, line in enumerate(lines):
            line = _eq_norm_text(line)
            if not _looks_like_equation_text(line):
                continue
            # Avoid obvious prose paragraphs that happen to contain a symbol.
            if len(line.split()) > 28 and not re.search(r"\(\s*\d+[a-z]?\s*\)\s*$", line):
                continue
            key = _eq_dedupe_key(line)
            if not key or key in existing_keys:
                continue
            existing_keys.add(key)
            context = []
            for j in range(max(0, idx - 2), min(len(lines), idx + 3)):
                context.append({"relative_line": j - idx, "text": lines[j]})
            found.append({
                "candidate_id": f"false_negative_scan::{stable_id(file_name, page, idx, line)}",
                "document_id": unit.get("pdf_id"),
                "file_name": file_name,
                "page": page,
                "candidate_type": "equation",
                "source_record_id": None,
                "source_method": "page_text_false_negative_scan",
                "text": line,
                "caption": None,
                "bbox": None,
                "asset_path": None,
                "line_context": {"nearest": context},
                "audit": {
                    "status": "unreviewed",
                    "is_candidate_not_final_fact": True,
                    "heuristic": {
                        "heuristic_suggestion": "possible_false_negative_equation_from_page_text",
                        "reasons": _equation_false_positive_reasons(line),
                    },
                },
            })
    return found


def _candidate_image_path(candidate: Dict[str, Any], extraction_root: Path) -> Optional[Path]:
    rel = candidate.get("asset_path")
    if not rel:
        return None
    p = extraction_root / rel
    if p.exists():
        return p
    p2 = Path(rel)
    if p2.exists():
        return p2
    return None


def _equation_llm_available() -> bool:
    if not RUN_LLM_EQUATION_LATEX_AUDIT:
        return False
    if "openai_generate" in globals():
        try:
            return bool(openai_available())
        except Exception:
            return False
    if "ollama_generate" in globals():
        try:
            return bool(ollama_available())
        except Exception:
            return False
    return False


def _llm_validate_equation_to_latex(candidate: Dict[str, Any], extraction_root: Path) -> Dict[str, Any]:
    """Use the configured OpenAI/Ollama model to validate a candidate and emit LaTeX."""
    text = _eq_norm_text(candidate.get("text", ""))
    context = candidate.get("line_context", {}) or {}
    image_path = _candidate_image_path(candidate, extraction_root) if EQUATION_LATEX_USE_CROPS else None

    system = (
        "You are a strict scientific PDF equation extraction auditor. "
        "Decide whether the candidate is a real mathematical equation, not a table header, caption, citation, unit range, "
        "decorative symbol, punctuation, or prose. Convert real equations to clean LaTeX. "
        "Use only the provided evidence. Return valid JSON only."
    )
    prompt = f"""
Candidate evidence JSON:
{json.dumps({
    "candidate_id": candidate.get("candidate_id"),
    "file_name": candidate.get("file_name"),
    "page": candidate.get("page"),
    "candidate_text": text,
    "nearby_context": context,
    "heuristic_reasons": _equation_false_positive_reasons(text),
    "has_visual_crop": bool(image_path),
}, ensure_ascii=False)}

Return JSON with exactly these keys:
{{
  "is_equation": true,
  "latex": "LaTeX expression only, without surrounding $$, or empty string if not an equation",
  "equation_label": "equation number/label if visible, else null",
  "confidence": 0.0,
  "false_positive_reason": "short reason if is_equation is false, else empty string",
  "notes": "brief evidence-based note"
}}

Rules:
- If the candidate is just special characters, punctuation, a citation, a section/table/figure heading, a unit range, or prose, set is_equation=false.
- If the text contains a real equation with a visible number like (3), put the number in equation_label and preserve it as \\tag{{3}} only if natural.
- Do not invent variables or terms not present in the evidence.
- Prefer syntactically valid LaTeX.
""".strip()

    try:
        if "openai_generate" in globals():
            images = [str(image_path)] if image_path and image_path.exists() else None
            raw = openai_generate(
                prompt=prompt,
                model=EQUATION_LATEX_MODEL,
                system=system,
                format_json=True,
                temperature=0.0,
                timeout=300,
                images=images,
            )
        else:
            images = None
            if image_path and image_path.exists() and "image_to_base64" in globals():
                images = [image_to_base64(image_path)]
            raw = ollama_generate(
                prompt=prompt,
                model=EQUATION_LATEX_MODEL,
                system=system,
                format_json=True,
                temperature=0.0,
                timeout=300,
                images=images,
            )
        data = parse_json_loose(raw)
        return {
            "is_equation": bool(data.get("is_equation")),
            "latex": _eq_norm_text(data.get("latex", "")),
            "equation_label": data.get("equation_label"),
            "confidence": float(data.get("confidence", 0.0) or 0.0),
            "false_positive_reason": _eq_norm_text(data.get("false_positive_reason", "")),
            "notes": _eq_norm_text(data.get("notes", "")),
            "verification_method": "llm",
            "raw_response": raw[:2000] if isinstance(raw, str) else str(raw)[:2000],
        }
    except Exception as exc:
        return {
            "is_equation": _looks_like_equation_text(text),
            "latex": _simple_text_to_latex(text) if _looks_like_equation_text(text) else "",
            "equation_label": None,
            "confidence": 0.45 if _looks_like_equation_text(text) else 0.0,
            "false_positive_reason": "" if _looks_like_equation_text(text) else ";".join(_equation_false_positive_reasons(text)),
            "notes": f"LLM equation audit failed, used deterministic fallback: {exc!r}",
            "verification_method": "fallback_after_llm_error",
            "raw_response": "",
        }


def _verify_candidate_equation(candidate: Dict[str, Any], extraction_root: Path) -> Dict[str, Any]:
    text = _eq_norm_text(candidate.get("text", ""))
    reasons = _equation_false_positive_reasons(text)
    strong_fp = any(r in {
        "empty_text", "too_short", "only_punctuation_or_special_characters", "url_doi_or_email",
        "caption_or_section_heading", "citation_or_reference_number",
    } for r in reasons)

    if strong_fp and EQUATION_LATEX_STRONG_FP_SKIP_LLM:
        decision = {
            "is_equation": False,
            "latex": "",
            "equation_label": None,
            "confidence": 0.99,
            "false_positive_reason": ";".join(reasons),
            "notes": "Rejected by deterministic strong false-positive rules before LLM call.",
            "verification_method": "heuristic_strong_false_positive",
            "raw_response": "",
        }
    elif _equation_llm_available():
        decision = _llm_validate_equation_to_latex(candidate, extraction_root)
        if decision.get("is_equation") and not decision.get("latex"):
            decision["latex"] = _simple_text_to_latex(text)
    else:
        is_eq = _looks_like_equation_text(text)
        decision = {
            "is_equation": is_eq,
            "latex": _simple_text_to_latex(text) if is_eq else "",
            "equation_label": None,
            "confidence": 0.62 if is_eq else 0.8,
            "false_positive_reason": "" if is_eq else ";".join(reasons),
            "notes": "Deterministic equation validation/conversion. Enable RUN_LLM_EQUATION_LATEX_AUDIT for stronger review.",
            "verification_method": "heuristic_fallback",
            "raw_response": "",
        }

    confidence = float(decision.get("confidence", 0.0) or 0.0)
    is_equation = bool(decision.get("is_equation")) and confidence >= EQUATION_LATEX_CONFIDENCE_KEEP_THRESHOLD
    latex = _eq_norm_text(decision.get("latex", ""))
    return {
        "equation_id": f"eq_latex_{stable_id(candidate.get('candidate_id'), latex or text)}",
        "candidate_id": candidate.get("candidate_id"),
        "document_id": candidate.get("document_id"),
        "file_name": candidate.get("file_name"),
        "page": candidate.get("page"),
        "source_method": candidate.get("source_method"),
        "source_record_id": candidate.get("source_record_id"),
        "original_text": text,
        "latex": latex if is_equation else "",
        "equation_label": decision.get("equation_label"),
        "is_equation": is_equation,
        "confidence": confidence,
        "false_positive_reason": "" if is_equation else (decision.get("false_positive_reason") or ";".join(reasons)),
        "verification_method": decision.get("verification_method"),
        "notes": decision.get("notes", ""),
        "bbox": candidate.get("bbox"),
        "asset_path": candidate.get("asset_path"),
        "line_context": candidate.get("line_context"),
        "heuristic_false_positive_reasons": reasons,
    }


def _equation_latex_unit(record: Dict[str, Any], source_path: Optional[str], pdf_id: Optional[str]) -> Dict[str, Any]:
    label = record.get("equation_label")
    label_text = f" ({label})" if label else ""
    text = (
        f"VERIFIED EQUATION LATEX{label_text}\n"
        f"PDF: {record.get('file_name')} | page: {record.get('page')}\n"
        f"LaTeX:\n$${record.get('latex')}$$\n\n"
        f"Original extracted text:\n{record.get('original_text')}\n\n"
        f"Audit: is_equation={record.get('is_equation')}; confidence={record.get('confidence')}; "
        f"method={record.get('verification_method')}; source={record.get('source_method')}"
    )
    return {
        "unit_id": record["equation_id"],
        "pdf_id": pdf_id or record.get("document_id"),
        "file_name": record.get("file_name"),
        "source_path": source_path or "",
        "page": record.get("page"),
        "element_type": "equation_latex",
        "text": text,
        "metadata": {
            "equation_id": record.get("equation_id"),
            "candidate_id": record.get("candidate_id"),
            "latex": record.get("latex"),
            "equation_label": record.get("equation_label"),
            "confidence": record.get("confidence"),
            "verification_method": record.get("verification_method"),
            "source_method": record.get("source_method"),
            "bbox": record.get("bbox"),
            "asset_path": record.get("asset_path"),
            "is_verified_equation_latex": True,
        },
    }


def _write_equation_latex_artifacts(
    paper_out_dir: Path,
    records: List[Dict[str, Any]],
    audit_records: List[Dict[str, Any]],
    pack_json_path: Optional[Path] = None,
) -> Dict[str, str]:
    learning_dir = Path(paper_out_dir) / "llm_learning_pack"
    learning_dir.mkdir(parents=True, exist_ok=True)
    latex_jsonl = learning_dir / "equations_latex.jsonl"
    audit_jsonl = learning_dir / "equation_audit_records.jsonl"
    latex_md = learning_dir / "equations_latex.md"

    write_jsonl(latex_jsonl, records)
    write_jsonl(audit_jsonl, audit_records)

    md_lines = ["# Verified equations as LaTeX", ""]
    if not records:
        md_lines.append("No verified equations were found by the equation-to-LaTeX layer.")
    for rec in records:
        md_lines.extend([
            f"## Page {rec.get('page')} - {rec.get('equation_id')}",
            "",
            f"Confidence: {rec.get('confidence')} | method: {rec.get('verification_method')}",
            "",
            f"Original: `{rec.get('original_text')}`",
            "",
            "$$",
            rec.get("latex", ""),
            "$$",
            "",
        ])
    latex_md.write_text("\n".join(md_lines), encoding="utf-8")

    if pack_json_path and Path(pack_json_path).exists():
        try:
            pack = json.loads(Path(pack_json_path).read_text(encoding="utf-8"))
            pack["latex_equation_extraction"] = {
                "enabled": bool(ENABLE_EQUATION_LATEX_LAYER),
                "llm_audit_enabled": bool(RUN_LLM_EQUATION_LATEX_AUDIT),
                "model": EQUATION_LATEX_MODEL,
                "verified_equation_count": len(records),
                "audit_record_count": len(audit_records),
                "false_positive_count": sum(1 for r in audit_records if not r.get("is_equation")),
                "files": {
                    "equations_latex_jsonl": str(latex_jsonl),
                    "equation_audit_records_jsonl": str(audit_jsonl),
                    "equations_latex_markdown": str(latex_md),
                },
            }
            pack["latex_equations"] = records
            Path(pack_json_path).write_text(json.dumps(pack, indent=2, ensure_ascii=False), encoding="utf-8")
        except Exception as exc:
            print(f"Warning: could not update learning pack JSON with LaTeX equations: {exc}")

    return {
        "equations_latex_jsonl": str(latex_jsonl),
        "equation_audit_records_jsonl": str(audit_jsonl),
        "equations_latex_markdown": str(latex_md),
    }


def _cached_equation_latex_artifacts_are_current(job: Dict[str, Any]) -> bool:
    """Used by the build cell so old caches are rebuilt once after this feature is added."""
    if not ENABLE_EQUATION_LATEX_LAYER:
        return True
    try:
        learning_dir = Path(job["vector_dir"]) / "llm_learning_pack"
        return (learning_dir / "equations_latex.jsonl").exists() and (learning_dir / "equation_audit_records.jsonl").exists()
    except Exception:
        return False


def enrich_units_with_equation_latex(units: List[Dict[str, Any]], paper_out_dir: Path, pdf_path: Path) -> List[Dict[str, Any]]:
    if not ENABLE_EQUATION_LATEX_LAYER:
        return units

    paper_out_dir = Path(paper_out_dir)
    extraction_root = paper_out_dir / PDF_TO_LLM_EXTRACTION_SUBDIR
    learning_dir = paper_out_dir / "llm_learning_pack"
    candidates_path = learning_dir / "candidate_inventory.jsonl"
    pack_json_path = learning_dir / "llm_learning_document.json"

    candidates: List[Dict[str, Any]] = []
    if candidates_path.exists():
        try:
            candidates = [row for row in read_jsonl(candidates_path) if row.get("candidate_type") == "equation"]
        except Exception:
            candidates = []

    # Fallback if the learning-pack candidate file is missing.
    if not candidates:
        for unit in units:
            if unit.get("element_type") == "equation_candidate":
                md = unit.get("metadata", {}) or {}
                text = str(unit.get("text", ""))
                # Try to recover the extractor text from the candidate unit text.
                m = re.search(r"Extractor text:\n(.*?)(?:\n\nNearest caption:|\n\nVisual crop/asset path:|\n\nNearby page context:|\n\nAudit guidance:|$)", text, flags=re.DOTALL)
                cand_text = _eq_norm_text(m.group(1)) if m else _eq_norm_text(text)
                candidates.append({
                    "candidate_id": md.get("candidate_id") or unit.get("unit_id"),
                    "document_id": unit.get("pdf_id"),
                    "file_name": unit.get("file_name"),
                    "page": unit.get("page"),
                    "candidate_type": "equation",
                    "source_record_id": md.get("source_record_id"),
                    "source_method": "equation_candidate_unit_fallback",
                    "text": cand_text,
                    "caption": None,
                    "bbox": md.get("bbox"),
                    "asset_path": md.get("crop_file"),
                    "line_context": {},
                    "audit": md.get("audit", {}),
                })

    existing_keys = {_eq_dedupe_key(c.get("text", "")) for c in candidates if _eq_dedupe_key(c.get("text", ""))}
    candidates.extend(_scan_page_text_for_equation_false_negatives(units, existing_keys))

    if EQUATION_LATEX_MAX_CANDIDATES_PER_PDF and len(candidates) > EQUATION_LATEX_MAX_CANDIDATES_PER_PDF:
        print(
            f"Equation LaTeX layer: limiting {len(candidates)} candidates to "
            f"{EQUATION_LATEX_MAX_CANDIDATES_PER_PDF}. Increase EQUATION_LATEX_MAX_CANDIDATES_PER_PDF if needed."
        )
        candidates = candidates[:EQUATION_LATEX_MAX_CANDIDATES_PER_PDF]

    audit_records: List[Dict[str, Any]] = []
    verified_records: List[Dict[str, Any]] = []
    for cand in tqdm(candidates, desc=f"Equation LaTeX {Path(pdf_path).name}", leave=False):
        record = _verify_candidate_equation(cand, extraction_root=extraction_root)
        audit_records.append(record)
        if record.get("is_equation") and record.get("latex"):
            verified_records.append(record)

    # Remove previous equation_latex units if the wrapper is rerun in the same kernel.
    base_units = [u for u in units if u.get("element_type") != "equation_latex"]

    false_positive_candidate_ids = {
        r.get("candidate_id") for r in audit_records
        if not r.get("is_equation") and r.get("candidate_id")
    }
    if DROP_EQUATION_FALSE_POSITIVES_FROM_RAG:
        filtered_units = []
        for unit in base_units:
            md = unit.get("metadata", {}) or {}
            if unit.get("element_type") == "equation_candidate" and md.get("candidate_id") in false_positive_candidate_ids:
                continue
            filtered_units.append(unit)
        base_units = filtered_units

    latex_by_candidate = {r.get("candidate_id"): r for r in verified_records if r.get("candidate_id")}
    for unit in base_units:
        if unit.get("element_type") == "equation_candidate":
            md = unit.setdefault("metadata", {})
            rec = latex_by_candidate.get(md.get("candidate_id"))
            if rec:
                md["verified_latex"] = rec.get("latex")
                md["equation_latex_id"] = rec.get("equation_id")
                md["equation_verification_confidence"] = rec.get("confidence")
                md["audit_status"] = "verified_equation_latex"

    source_path = str(pdf_path)
    pdf_id = None
    for u in base_units:
        if u.get("pdf_id"):
            pdf_id = u.get("pdf_id")
            break

    latex_units = [_equation_latex_unit(r, source_path=source_path, pdf_id=pdf_id) for r in verified_records]
    artifact_paths = _write_equation_latex_artifacts(
        paper_out_dir=paper_out_dir,
        records=verified_records,
        audit_records=audit_records,
        pack_json_path=pack_json_path,
    )

    for unit in base_units:
        if unit.get("element_type") == "document_metadata":
            md = unit.setdefault("metadata", {})
            md["equation_latex_layer"] = {
                "enabled": True,
                "llm_audit_enabled": bool(RUN_LLM_EQUATION_LATEX_AUDIT),
                "model": EQUATION_LATEX_MODEL,
                "verified_equation_count": len(verified_records),
                "false_positive_count": sum(1 for r in audit_records if not r.get("is_equation")),
                "audit_record_count": len(audit_records),
                "artifact_paths": artifact_paths,
            }
            md.setdefault("llm_learning_pack_files", {}).update(artifact_paths)
            break

    print(
        f"Equation LaTeX layer for {Path(pdf_path).name}: "
        f"{len(verified_records)} verified equation(s), "
        f"{sum(1 for r in audit_records if not r.get('is_equation'))} false positive(s), "
        f"{sum(1 for r in audit_records if r.get('source_method') == 'page_text_false_negative_scan')} scanned false-negative candidate(s)."
    )
    return base_units + latex_units


# Wrap the active extractor exactly once.
if "_BASE_EXTRACT_PDF_UNITS_BEFORE_EQUATION_LATEX" not in globals():
    _BASE_EXTRACT_PDF_UNITS_BEFORE_EQUATION_LATEX = extract_pdf_units

    def extract_pdf_units(pdf_path: Path, paper_out_dir: Path) -> List[Dict[str, Any]]:
        units = _BASE_EXTRACT_PDF_UNITS_BEFORE_EQUATION_LATEX(pdf_path, paper_out_dir)
        return enrich_units_with_equation_latex(units, paper_out_dir=paper_out_dir, pdf_path=pdf_path)

print(
    "Equation LaTeX layer enabled:",
    ENABLE_EQUATION_LATEX_LAYER,
    "| LLM audit:",
    RUN_LLM_EQUATION_LATEX_AUDIT,
    "| model:",
    EQUATION_LATEX_MODEL,
)


Equation LaTeX layer enabled: True | LLM audit: True | model: gpt-5.2


## 6. Chunking and paper-profile extraction

In [8]:

def split_text_by_chars(text: str, max_chars: int = CHUNK_SIZE_CHARS, overlap: int = CHUNK_OVERLAP_CHARS) -> List[str]:
    text = clean_text(text)
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        if end < n:
            cut_candidates = [
                text.rfind("\n\n", start, end),
                text.rfind("\n", start, end),
                text.rfind(". ", start, end),
                text.rfind(" ", start, end),
            ]
            cut = max(cut_candidates)
            if cut > start + max_chars * 0.55:
                end = cut + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        start = max(0, end - overlap)
    return chunks


def infer_paper_stem_from_record(record: Dict[str, Any], fallback: str = "") -> str:
    """Return a stable paper identifier even for older cached chunks.

    Older notebook runs did not store paper_stem on every chunk. Retrieval and
    GraphRAG routing now use this function so old artifacts continue to work
    without forcing a full rebuild.
    """
    meta = record.get("metadata") if isinstance(record.get("metadata"), dict) else {}
    return (
        str(record.get("paper_stem") or "").strip()
        or str(meta.get("paper_stem") or "").strip()
        or (safe_filename(Path(str(record.get("file_name") or "")).stem) if record.get("file_name") else "")
        or fallback
    )


def normalize_chunk_for_paper(chunk: Dict[str, Any], paper_stem: str = "") -> Dict[str, Any]:
    """Add missing retrieval metadata to a chunk in-place and return it."""
    stem = infer_paper_stem_from_record(chunk, fallback=paper_stem)
    if stem:
        chunk["paper_stem"] = stem
        meta = chunk.get("metadata")
        if not isinstance(meta, dict):
            meta = {}
        meta.setdefault("paper_stem", stem)
        chunk["metadata"] = meta
    return chunk


def normalize_chunks_for_paper(chunks: List[Dict[str, Any]], paper_stem: str = "") -> List[Dict[str, Any]]:
    return [normalize_chunk_for_paper(c, paper_stem=paper_stem) for c in chunks]


def build_chunks(units: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []
    for unit in units:
        text = unit.get("text", "") or ""
        if not text.strip() or unit.get("element_type") == "document_metadata":
            continue
        unit_file_name = unit.get("file_name")
        unit_paper_stem = infer_paper_stem_from_record(unit)
        for idx, part in enumerate(split_text_by_chars(text)):
            chunk_id = f"chunk_{stable_id(unit.get('unit_id'), idx)}"
            page = unit.get("page")
            element_type = unit.get("element_type")
            citation = f"{unit_file_name} p.{page} {element_type} {chunk_id}"
            metadata = unit.get("metadata", {}) if isinstance(unit.get("metadata", {}), dict) else {}
            if unit_paper_stem:
                metadata = dict(metadata)
                metadata.setdefault("paper_stem", unit_paper_stem)
            chunks.append({
                "chunk_id": chunk_id,
                "unit_id": unit.get("unit_id"),
                "pdf_id": unit.get("pdf_id"),
                "paper_stem": unit_paper_stem,
                "file_name": unit_file_name,
                "source_path": unit.get("source_path"),
                "page": page,
                "element_type": element_type,
                "text": part,
                "citation": citation,
                "metadata": metadata,
            })
    return chunks


def gather_profile_context(chunks: List[Dict[str, Any]], max_chars: int = 14000) -> str:
    preferred = []
    keyword_boost = ("abstract", "introduction", "method", "materials", "dataset", "experiment", "results", "discussion", "conclusion", "limitation")
    for chunk in chunks:
        score = 0
        if chunk.get("page") in (1, 2, 3):
            score += 4
        if chunk.get("element_type") in ("table", "equation_candidate", "figure_caption"):
            score += 2
        lower = chunk.get("text", "").lower()
        score += sum(1 for token in keyword_boost if token in lower)
        preferred.append((score, chunk))
    preferred.sort(key=lambda x: (-x[0], x[1].get("page") or 999999))
    selected, total = [], 0
    seen = set()
    for score, chunk in preferred:
        cid = chunk["chunk_id"]
        if cid in seen:
            continue
        seen.add(cid)
        piece = f"[{chunk['citation']}]\n{chunk['text']}"
        if total + len(piece) > max_chars and selected:
            break
        selected.append(piece)
        total += len(piece)
    return "\n\n".join(selected)


PROFILE_SYSTEM = "Use only the supplied PDF text and citations. Return strict JSON only."


def extract_paper_profile(pdf_path: Path, chunks: List[Dict[str, Any]], max_chars: int = 14000) -> Dict[str, Any]:
    context = gather_profile_context(chunks, max_chars=max_chars)
    prompt = f"""
Extract a structured profile for this scientific paper.
Return strict JSON with exactly these keys:
{{
  "title": "",
  "document_type": "",
  "authors": [],
  "affiliations": [],
  "journal_or_venue": "",
  "publication_year": null,
  "doi": "",
  "abstract": "",
  "keywords": [],
  "main_topics": [],
  "methods": [],
  "materials_or_datasets": [],
  "results": [],
  "conclusions": [],
  "limitations": [],
  "applications": [],
  "summary": ""
}}

Rules:
- Use only the supplied text.
- Keep list fields concise but informative.
- Prefer grounded scientific phrasing.
- If something is not present, return "" for strings, [] for lists, and null for year.

PDF file: {pdf_path.name}

TEXT:
{context}
""".strip()

    profile: Dict[str, Any]
    try:
        raw = openai_generate(
            prompt=prompt,
            model=GENERATION_MODEL,
            system=PROFILE_SYSTEM,
            format_json=True,
            temperature=0.0,
            timeout=300,
        )
        profile = parse_json_loose(raw)
        if not isinstance(profile, dict):
            profile = {}
    except Exception as exc:
        print(f"Profile extraction failed for {pdf_path.name}: {exc}")
        profile = {}

    profile = dict(profile)
    profile["file_name"] = pdf_path.name
    profile["paper_stem"] = safe_filename(pdf_path.stem)
    profile["source_path"] = str(pdf_path)
    profile["authors"] = ensure_list_of_strings(profile.get("authors"))
    profile["affiliations"] = ensure_list_of_strings(profile.get("affiliations"))
    profile["keywords"] = ensure_list_of_strings(profile.get("keywords"))
    profile["main_topics"] = ensure_list_of_strings(profile.get("main_topics"))
    profile["methods"] = ensure_list_of_strings(profile.get("methods"))
    profile["materials_or_datasets"] = ensure_list_of_strings(profile.get("materials_or_datasets"))
    profile["results"] = ensure_list_of_strings(profile.get("results"))
    profile["conclusions"] = ensure_list_of_strings(profile.get("conclusions"))
    profile["limitations"] = ensure_list_of_strings(profile.get("limitations"))
    profile["applications"] = ensure_list_of_strings(profile.get("applications"))
    profile["title"] = ensure_text(profile.get("title")) or pdf_path.stem
    profile["document_type"] = ensure_text(profile.get("document_type"))
    profile["journal_or_venue"] = ensure_text(profile.get("journal_or_venue"))
    profile["doi"] = ensure_text(profile.get("doi"))
    profile["abstract"] = ensure_text(profile.get("abstract"))
    profile["summary"] = ensure_text(profile.get("summary"))
    try:
        year = profile.get("publication_year")
        profile["publication_year"] = int(year) if year not in (None, "", "null") else None
    except Exception:
        profile["publication_year"] = None
    return profile


def build_paper_section_chunks(profile: Dict[str, Any]) -> List[Dict[str, Any]]:
    sections = {
        "summary": profile.get("summary") or profile.get("abstract"),
        "methods": ensure_text(profile.get("methods")),
        "materials_or_datasets": ensure_text(profile.get("materials_or_datasets")),
        "results": ensure_text(profile.get("results")),
        "conclusions": ensure_text(profile.get("conclusions")),
        "limitations": ensure_text(profile.get("limitations")),
        "applications": ensure_text(profile.get("applications")),
    }
    chunks = []
    file_name = profile["file_name"]
    paper_stem = profile["paper_stem"]
    title = profile.get("title", paper_stem)
    keywords = ", ".join(profile.get("keywords", []))
    topics = ", ".join(profile.get("main_topics", []))
    for section_name, content in sections.items():
        content = ensure_text(content)
        if not content:
            continue
        text = clean_text(
            f"Paper: {title}\n"
            f"File: {file_name}\n"
            f"Section: {section_name}\n"
            f"Keywords: {keywords}\n"
            f"Topics: {topics}\n"
            f"Content: {content}"
        )
        chunk_id = f"profile_{section_name}_{stable_id(file_name, section_name)}"
        citation = f"{file_name} profile {section_name}"
        chunks.append({
            "chunk_id": chunk_id,
            "paper_stem": paper_stem,
            "file_name": file_name,
            "section": section_name,
            "text": text,
            "citation": citation,
            "title": title,
        })
    return chunks


## 7. Per-paper VectorRAG and GraphRAG builders

In [9]:


# -------------------------------------------------------------------------
# Graph JSON loader is used by the main build cell, so it must be defined
# before the pipeline is run top-to-bottom.
# -------------------------------------------------------------------------
def load_graph_from_json(path: Path) -> nx.MultiDiGraph:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    G = nx.MultiDiGraph()
    for n in data.get("nodes", []):
        attrs = dict(n)
        node_id = attrs.pop("id")
        G.add_node(node_id, **attrs)
    for e in data.get("edges", []):
        attrs = dict(e)
        u = attrs.pop("source")
        v = attrs.pop("target")
        key = attrs.pop("key", None)
        G.add_edge(u, v, key=key, **attrs)
    return G


# -------------------------------------------------------------------------
# Robust Chroma helpers.
#
# Chroma's PersistentClient can reuse an in-process system cache for a path.
# If a local SQLite/Chroma folder was deleted, interrupted, or created by an
# incompatible version, initialization can fail with errors such as:
#   "no such table: tenants"
# These helpers clear Chroma's process cache and, if needed, rebuild into a
# fresh physical directory instead of crashing the notebook.
# -------------------------------------------------------------------------
def clear_chroma_system_cache() -> None:
    """Best-effort cleanup of Chroma's in-process shared client/system cache."""
    try:
        gc.collect()
    except Exception:
        pass

    module_names = [
        "chromadb.api.client",
        "chromadb.api.shared_system_client",
    ]
    for module_name in module_names:
        try:
            module = __import__(module_name, fromlist=["SharedSystemClient"])
            cls = getattr(module, "SharedSystemClient", None)
            if cls is None:
                continue

            clear_fn = getattr(cls, "clear_system_cache", None)
            if callable(clear_fn):
                clear_fn()
                continue

            # Fallback for Chroma versions that expose only private caches.
            for attr in ("_identifier_to_system", "_system_cache"):
                cache = getattr(cls, attr, None)
                if isinstance(cache, dict):
                    cache.clear()
        except Exception:
            continue


def chroma_error_is_recoverable(exc: Exception) -> bool:
    msg = str(exc).lower()
    recoverable_markers = [
        "no such table: tenants",
        "no such table: databases",
        "database disk image is malformed",
        "database is locked",
        "attempt to write a readonly database",
        "could not connect to tenant",
        "database error",
        "sqlite",
    ]
    return any(marker in msg for marker in recoverable_markers)


def unique_chroma_dir(base_path: Path, reason: str = "rebuilt") -> Path:
    stamp = int(time.time())
    token = stable_id(str(base_path), stamp, time.time(), length=8)
    candidate = base_path.parent / f"{base_path.name}_{reason}_{stamp}_{token}"
    if candidate.exists():
        shutil.rmtree(candidate, ignore_errors=True)
    candidate.mkdir(parents=True, exist_ok=True)
    return candidate


def prepare_chroma_dir(path: Path, force_rebuild: bool = True) -> Path:
    """Prepare a writable Chroma directory and clear stale in-process Chroma state."""
    path = Path(path)
    clear_chroma_system_cache()

    candidate = path
    if force_rebuild and candidate.exists():
        try:
            shutil.rmtree(candidate)
            gc.collect()
        except Exception as exc:
            print(f"Warning: could not fully remove {candidate}: {exc}")
            candidate = unique_chroma_dir(path, reason="fallback")

    candidate.mkdir(parents=True, exist_ok=True)
    try:
        probe = candidate / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink(missing_ok=True)
    except Exception as exc:
        fallback = unique_chroma_dir(path, reason="writable")
        print(f"Warning: {candidate} is not writable ({exc}). Falling back to {fallback}.")
        candidate = fallback

    return candidate


def create_chroma_persistent_client(chroma_dir: Path, *, repair_if_needed: bool = True):
    """Create a Chroma PersistentClient, repairing stale/corrupt local stores when safe."""
    chroma_dir = Path(chroma_dir)
    clear_chroma_system_cache()
    try:
        return chromadb.PersistentClient(path=str(chroma_dir)), chroma_dir
    except Exception as exc:
        if not (repair_if_needed and chroma_error_is_recoverable(exc)):
            raise

        print(
            "Warning: Chroma could not open the local store at "
            f"{chroma_dir} ({exc}). Rebuilding a fresh Chroma directory."
        )
        clear_chroma_system_cache()
        try:
            if chroma_dir.exists():
                shutil.rmtree(chroma_dir, ignore_errors=True)
        except Exception:
            pass

        fallback = unique_chroma_dir(chroma_dir, reason="repair")
        clear_chroma_system_cache()
        try:
            return chromadb.PersistentClient(path=str(fallback)), fallback
        except Exception as second_exc:
            raise RuntimeError(
                f"Chroma failed for both {chroma_dir} and fallback {fallback}. "
                f"Original error: {exc}. Fallback error: {second_exc}"
            ) from second_exc



def chroma_collection_is_readable(chroma_dir: Path, collection_name: str) -> bool:
    """Return True only if an existing Chroma collection can be opened cleanly."""
    try:
        client, _actual = create_chroma_persistent_client(Path(chroma_dir), repair_if_needed=False)
        client.get_collection(collection_name)
        return True
    except Exception as exc:
        print(f"Warning: existing Chroma store is not reusable: {chroma_dir} / {collection_name}: {exc}")
        clear_chroma_system_cache()
        return False


def sanitize_chroma_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    clean = {}
    for k, v in meta.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v if v is not None else ""
        else:
            clean[k] = json.dumps(v, ensure_ascii=False)
    return clean



def dedupe_records_by_id(rows: List[Dict[str, Any]], key: str = "chunk_id") -> List[Dict[str, Any]]:
    out, seen = [], set()
    for row in rows:
        row_id = str(row.get(key) or "")
        if not row_id or row_id in seen:
            continue
        seen.add(row_id)
        out.append(row)
    return out


def build_per_pdf_vector_store(paper_stem: str, vector_dir: Path, chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    chroma_dir = prepare_chroma_dir(vector_dir / "chroma_db", force_rebuild=True)

    collection_name = safe_filename(f"{paper_stem}{PER_PDF_VECTOR_SUFFIX}", 60)
    client, chroma_dir = create_chroma_persistent_client(chroma_dir, repair_if_needed=True)
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine", "embedding_model": EMBEDDING_MODEL, "provider": "openai"},
    )

    index_rows = dedupe_records_by_id(chunks, key="chunk_id")

    batch_size = CHROMA_UPSERT_BATCH_SIZE
    for start in tqdm(range(0, len(index_rows), batch_size), desc=f"Vector store {paper_stem}", leave=False):
        batch = index_rows[start:start + batch_size]
        texts = [c["text"] for c in batch]
        ids = [c["chunk_id"] for c in batch]
        metadatas = [sanitize_chroma_metadata({
            "paper_stem": c.get("paper_stem") or paper_stem,
            "file_name": c.get("file_name", ""),
            "page": int(c["page"]) if c.get("page") is not None else -1,
            "element_type": c.get("element_type", ""),
            "citation": c.get("citation", ""),
            "unit_id": c.get("unit_id", ""),
            "pdf_id": c.get("pdf_id", ""),
            "source_path": c.get("source_path", ""),
        }) for c in batch]
        embeddings = openai_embed(texts, model=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH_SIZE)
        collection.upsert(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings)

    return {
        "collection_name": collection_name,
        "chroma_dir": str(chroma_dir),
        "count": collection.count(),
    }


ALLOWED_NODE_TYPES = [
    "Document", "Concept", "Method", "Material", "Dataset", "Instrument", "Variable",
    "Descriptor", "Equation", "Finding", "Application", "Limitation", "Organization", "Person"
]
ALLOWED_RELATION_TYPES = [
    "PART_OF", "USES", "MEASURES", "HAS_DESCRIPTOR", "HAS_PROPERTY", "AFFECTS",
    "INFLUENCES", "CAUSES", "COMPARES_WITH", "VALIDATES", "LIMITED_BY", "REPORTED_IN",
    "DERIVED_FROM", "APPLIES_TO", "SUPPORTS", "CONTRADICTS", "MENTIONS", "EVIDENCED_BY"
]
KG_SYSTEM = "Use only the supplied chunk. Extract grounded scientific knowledge graphs. Return strict JSON only."

LOW_VALUE_SECTION_MARKERS = [
    "references", "bibliography", "acknowledgments", "acknowledgements", "conflict of interest",
    "competing interests", "author contributions", "financial support", "funding", "copyright",
    "creative commons", "downloaded from", "terms of use", "supplementary material"
]

KG_VALUE_MARKERS = {
    "method": 1.3,
    "model": 1.2,
    "dataset": 1.2,
    "data": 0.8,
    "instrument": 1.2,
    "measurement": 1.3,
    "equation": 1.8,
    "variable": 1.5,
    "descriptor": 1.8,
    "property": 1.0,
    "parameter": 1.0,
    "result": 1.2,
    "finding": 1.4,
    "correlation": 1.6,
    "relationship": 1.4,
    "distribution": 1.4,
    "analysis": 1.0,
    "experiment": 1.2,
    "material": 1.0,
    "sample": 0.8,
    "composition": 1.2,
    "limitation": 1.4,
    "application": 1.0,
    "affect": 1.2,
    "influence": 1.2,
    "validate": 1.2,
    "compare": 1.0,
}

ELEMENT_TYPE_BONUS = {
    "equation_latex": 3.5,
    "equation_candidate": 2.0,
    "table": 2.5,
    "figure_caption": 1.8,
    "figure_image": 1.5,
    "page_text": 0.8,
}


def normalize_float(value: Any, default: float = 0.5) -> float:
    try:
        x = float(value)
    except Exception:
        x = default
    return max(0.0, min(1.0, x))


def canonical_node_key(label: str, node_type: str) -> str:
    label_norm = re.sub(r"\s+", " ", str(label or "").strip()).lower()
    label_norm = re.sub(r"[^a-z0-9 μµ._:/+-]+", "_", label_norm).strip("_")
    type_norm = re.sub(r"\W+", "_", str(node_type or "Concept").strip()) or "Concept"
    return f"{type_norm}:{label_norm[:90]}"


def graph_prompt_for_chunk(chunk: Dict[str, Any]) -> str:
    chunk_text = str(chunk.get("text", ""))[:KG_MAX_CHARS_PER_CHUNK]
    return f"""
Extract a knowledge graph from this PDF chunk.
Allowed node types: {ALLOWED_NODE_TYPES}
Allowed relationship types: {ALLOWED_RELATION_TYPES}
Return strict JSON:
{{
  "nodes": [{{"id": "short label", "type": "Concept|Method|...", "description": "one sentence", "relevance": 0.0}}],
  "relationships": [{{"source": "node id", "target": "node id", "type": "RELATION_TYPE", "description": "grounded relationship", "evidence": "short exact supporting phrase", "confidence": 0.0}}]
}}
Rules:
- Use only facts visible in the chunk.
- Prefer high-value scientific relationships.
- Include Equation nodes when present.
- Include variables/descriptors and what they measure.
- If no graph facts exist, return {{"nodes": [], "relationships": []}}.

Citation for this chunk: {chunk['citation']}

CHUNK TEXT:
{chunk_text}
""".strip()


def high_value_chunk_score(chunk: Dict[str, Any]) -> float:
    text = (chunk.get("text") or "").strip()
    if len(text) < KG_MIN_CHARS:
        return -1.0

    head = text[:400].lower()
    if any(marker in head for marker in LOW_VALUE_SECTION_MARKERS):
        return -1.0

    score = ELEMENT_TYPE_BONUS.get(chunk.get("element_type"), 0.0)
    lower = text.lower()
    for term, weight in KG_VALUE_MARKERS.items():
        if term in lower:
            score += weight
    if re.search(r"[=∑∫π√≤≥±≈≠×÷∞μσθδαβγλΩΔ]", text):
        score += 1.2
    if re.search(r"\b(fig\.?|figure|table)\s*\d+\b", lower):
        score += 0.8
    if re.search(r"\b(method|materials?|dataset|results?|conclusion|discussion|limitation)s?\b", lower):
        score += 1.2
    return float(score)


def select_chunks_for_kg(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    selected = []
    for chunk in chunks:
        score = high_value_chunk_score(chunk) if KG_ONLY_HIGH_VALUE_CHUNKS else 0.0
        chunk = dict(chunk)
        chunk["kg_score"] = score
        if not KG_ONLY_HIGH_VALUE_CHUNKS or score >= KG_MIN_SCORE:
            selected.append(chunk)
    if KG_SORT_BY_SCORE:
        selected.sort(key=lambda x: (x.get("kg_score", 0.0), -(x.get("page") or 0)), reverse=True)
    if MAX_CHUNKS_PER_PDF_FOR_KG is not None:
        selected = selected[:MAX_CHUNKS_PER_PDF_FOR_KG]
    return selected


def load_kg_cache(cache_path: Path) -> Dict[str, Dict[str, Any]]:
    cache = {}
    if not KG_RESUME_FROM_CACHE or not cache_path.exists():
        return cache
    for row in read_jsonl(cache_path):
        cid = row.get("chunk_id")
        if cid:
            cache[cid] = row.get("data", {"nodes": [], "relationships": []})
    return cache


def extract_graph_from_chunk(chunk: Dict[str, Any]) -> Dict[str, Any]:
    try:
        raw = openai_generate(
            graph_prompt_for_chunk(chunk),
            model=GENERATION_MODEL,
            system=KG_SYSTEM,
            format_json=True,
            temperature=0.0,
            timeout=300,
        )
        data = parse_json_loose(raw)
        if not isinstance(data, dict):
            return {"nodes": [], "relationships": []}
        data.setdefault("nodes", [])
        data.setdefault("relationships", [])
        return data
    except Exception as exc:
        print(f"KG extraction failed for {chunk['chunk_id']}: {exc}")
        return {"nodes": [], "relationships": []}


def build_knowledge_graph(chunks: List[Dict[str, Any]], profile: Dict[str, Any], graph_dir: Path) -> Tuple[nx.MultiDiGraph, List[Dict[str, Any]]]:
    cache_path = graph_dir / "kg_cache.jsonl"
    cache = load_kg_cache(cache_path)

    G = nx.MultiDiGraph()
    paper_node = f"Paper:{profile['paper_stem']}"
    paper_label = profile.get("title") or profile["file_name"]
    paper_desc = profile.get("summary") or profile.get("abstract") or profile["file_name"]
    G.add_node(
        paper_node,
        label=paper_label,
        type="Document",
        description=paper_desc,
        citations=profile["file_name"],
        pages="",
        file_name=profile["file_name"],
        relevance="1.0",
        paper_stem=profile["paper_stem"],
    )

    selected_chunks = select_chunks_for_kg(chunks)

    for chunk in tqdm(selected_chunks, desc=f"GraphRAG {profile['paper_stem']}", leave=False):
        data = cache.get(chunk["chunk_id"])
        if data is None:
            data = extract_graph_from_chunk(chunk)
            if KG_RESUME_FROM_CACHE:
                append_jsonl(cache_path, {"chunk_id": chunk["chunk_id"], "data": data})
        chunk_node = f"Chunk:{chunk['chunk_id']}"
        G.add_node(
            chunk_node,
            label=chunk["chunk_id"],
            type="Chunk",
            description=chunk["citation"],
            citations=chunk["citation"],
            pages=str(chunk.get("page")),
            file_name=chunk["file_name"],
            relevance="1.0",
            paper_stem=profile["paper_stem"],
        )
        G.add_edge(chunk_node, paper_node, key=f"PART_OF:{chunk['chunk_id']}", type="PART_OF", description="chunk belongs to paper", evidence=chunk["citation"], confidence="1.0", citations=chunk["citation"])

        node_map = {}
        for n in data.get("nodes", []) or []:
            if not isinstance(n, dict):
                continue
            label = str(n.get("id") or n.get("label") or "").strip()
            if not label:
                continue
            node_type = str(n.get("type") or "Concept").strip()
            if node_type not in ALLOWED_NODE_TYPES:
                node_type = "Concept"
            key = canonical_node_key(label, node_type)
            node_map[label] = key
            description = str(n.get("description") or "").strip()
            relevance = normalize_float(n.get("relevance", 0.5))

            if G.has_node(key):
                cites = set(str(G.nodes[key].get("citations", "")).split(" || "))
                cites.add(chunk["citation"])
                G.nodes[key]["citations"] = " || ".join(sorted(x for x in cites if x))
                pages = set(str(G.nodes[key].get("pages", "")).split(","))
                pages.add(str(chunk.get("page")))
                G.nodes[key]["pages"] = ",".join(sorted(x for x in pages if x))
                existing_desc = str(G.nodes[key].get("description", ""))
                if description and description not in existing_desc:
                    G.nodes[key]["description"] = (existing_desc + " | " + description).strip(" |")[:1200]
            else:
                G.add_node(
                    key,
                    label=label,
                    type=node_type,
                    description=description,
                    citations=chunk["citation"],
                    pages=str(chunk.get("page")),
                    file_name=chunk["file_name"],
                    relevance=str(relevance),
                    paper_stem=profile["paper_stem"],
                )

            G.add_edge(
                key,
                chunk_node,
                key=f"EVIDENCED_BY:{stable_id(key, chunk_node)}",
                type="EVIDENCED_BY",
                description="supported by chunk",
                evidence=chunk["citation"],
                confidence="1.0",
                citations=chunk["citation"],
            )
            G.add_edge(
                key,
                paper_node,
                key=f"REPORTED_IN:{stable_id(key, paper_node)}",
                type="REPORTED_IN",
                description="reported in this paper",
                evidence=chunk["citation"],
                confidence="1.0",
                citations=chunk["citation"],
            )

        for rel in data.get("relationships", []) or []:
            if not isinstance(rel, dict):
                continue
            src_label = str(rel.get("source") or "").strip()
            tgt_label = str(rel.get("target") or "").strip()
            if not src_label or not tgt_label:
                continue
            src = node_map.get(src_label) or canonical_node_key(src_label, "Concept")
            tgt = node_map.get(tgt_label) or canonical_node_key(tgt_label, "Concept")
            if not G.has_node(src):
                G.add_node(src, label=src_label, type="Concept", description="", citations=chunk["citation"], pages=str(chunk.get("page")), file_name=chunk["file_name"], relevance="0.5", paper_stem=profile["paper_stem"])
            if not G.has_node(tgt):
                G.add_node(tgt, label=tgt_label, type="Concept", description="", citations=chunk["citation"], pages=str(chunk.get("page")), file_name=chunk["file_name"], relevance="0.5", paper_stem=profile["paper_stem"])

            rtype = str(rel.get("type") or "MENTIONS").strip().upper().replace(" ", "_")
            if rtype not in ALLOWED_RELATION_TYPES:
                rtype = "MENTIONS"

            edge_key = f"{rtype}:{stable_id(src, tgt, rtype, chunk['chunk_id'])}"
            G.add_edge(
                src,
                tgt,
                key=edge_key,
                type=rtype,
                description=str(rel.get("description") or ""),
                evidence=str(rel.get("evidence") or ""),
                confidence=str(normalize_float(rel.get("confidence", 0.5))),
                citations=chunk["citation"],
                chunk_id=chunk["chunk_id"],
                page=str(chunk.get("page")),
                file_name=chunk["file_name"],
            )

        if KG_BATCH_SLEEP_SECONDS:
            time.sleep(KG_BATCH_SLEEP_SECONDS)

    return G, selected_chunks


def graph_to_json(G: nx.MultiDiGraph) -> Dict[str, Any]:
    nodes = []
    for node_id, attrs in G.nodes(data=True):
        row = {"id": node_id}
        row.update({k: str(v) for k, v in attrs.items()})
        nodes.append(row)
    edges = []
    for u, v, key, attrs in G.edges(keys=True, data=True):
        row = {"source": u, "target": v, "key": key}
        row.update({k: str(vv) for k, vv in attrs.items()})
        edges.append(row)
    return {"nodes": nodes, "edges": edges}


def save_graph_bundle(G: nx.MultiDiGraph, graph_dir: Path) -> Dict[str, str]:
    graph_dir.mkdir(parents=True, exist_ok=True)
    kg_data = graph_to_json(G)

    kg_json_path = graph_dir / "knowledge_graph.json"
    kg_json_path.write_text(json.dumps(kg_data, indent=2, ensure_ascii=False), encoding="utf-8")

    nodes_csv = graph_dir / "kg_nodes.csv"
    edges_csv = graph_dir / "kg_edges.csv"
    pd.DataFrame(kg_data["nodes"]).to_csv(nodes_csv, index=False)
    pd.DataFrame(kg_data["edges"]).to_csv(edges_csv, index=False)

    graphml_path = graph_dir / "knowledge_graph.graphml"
    try:
        nx.write_graphml(G, graphml_path)
    except Exception as exc:
        print(f"GraphML export failed for {graph_dir.name}: {exc}")

    html_path = graph_dir / "graph_visualization.html"
    try:
        from pyvis.network import Network
        net = Network(height="850px", width="100%", directed=True, notebook=True, bgcolor="#ffffff")
        drawn_nodes = list(G.nodes())[:350]
        drawn = set(drawn_nodes)
        for node_id in drawn_nodes:
            attrs = G.nodes[node_id]
            label = attrs.get("label", node_id)
            group = attrs.get("type", "Unknown")
            title = f"{group}: {label}<br>{attrs.get('description', '')}<br>{attrs.get('citations', '')}"
            net.add_node(node_id, label=str(label)[:60], title=title, group=group)
        for u, v, key, attrs in G.edges(keys=True, data=True):
            if u in drawn and v in drawn:
                net.add_edge(u, v, label=attrs.get("type", ""), title=attrs.get("evidence", ""))
        net.write_html(str(html_path), notebook=False)
    except Exception as exc:
        print(f"Graph visualization skipped for {graph_dir.name}: {exc}")

    return {
        "knowledge_graph_json": str(kg_json_path),
        "knowledge_graph_graphml": str(graphml_path),
        "kg_nodes_csv": str(nodes_csv),
        "kg_edges_csv": str(edges_csv),
        "graph_visualization": str(html_path),
    }


## 8. Run the per-paper pipeline

In [10]:
from concurrent.futures import ThreadPoolExecutor, as_completed

pdf_files = list_pdf_files(PDF_DIR)
print(f"Found {len(pdf_files)} PDF(s).")

paper_registry: List[Dict[str, Any]] = []
all_profiles: List[Dict[str, Any]] = []
all_corpus_chunks: List[Dict[str, Any]] = []


def _paper_paths(pdf_path: Path) -> Dict[str, Any]:
    paper_stem = safe_filename(pdf_path.stem)
    vector_dir = OUT_DIR / f"{paper_stem}{PER_PDF_VECTOR_SUFFIX}"
    graph_dir = OUT_DIR / f"{paper_stem}{PER_PDF_GRAPH_SUFFIX}"
    vector_dir.mkdir(parents=True, exist_ok=True)
    graph_dir.mkdir(parents=True, exist_ok=True)
    return {
        "pdf_path": pdf_path,
        "paper_stem": paper_stem,
        "vector_dir": vector_dir,
        "graph_dir": graph_dir,
        "units_path": vector_dir / f"{paper_stem}_units.jsonl",
        "chunks_path": vector_dir / f"{paper_stem}_chunks.jsonl",
        "profile_path": vector_dir / "paper_profile.json",
        "graph_profile_path": graph_dir / "paper_profile.json",
        "selected_kg_chunks_path": graph_dir / "selected_kg_chunks.jsonl",
    }


def _artifacts_are_current(output_path: Path, input_paths: List[Path]) -> bool:
    if FORCE_REBUILD_ALL or not output_path.exists():
        return False
    try:
        out_mtime = output_path.stat().st_mtime
        return all((not p.exists()) or out_mtime >= p.stat().st_mtime for p in input_paths)
    except Exception:
        return False


def _vector_manifest_matches_current_embedding(manifest_path: Path) -> bool:
    """Avoid reusing a vector store built with a different embedding model/dimension."""
    try:
        manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
        if manifest.get("embedding_model") != EMBEDDING_MODEL:
            return False
        existing_dims = manifest.get("embedding_dimensions")
        current_dims = EMBEDDING_DIMENSIONS if EMBEDDING_DIMENSIONS is not None else "model_default"
        return existing_dims == current_dims
    except Exception:
        return False




def _vector_store_cache_is_usable(manifest_path: Path, default_chroma_dir: Path) -> bool:
    """Validate that an existing per-paper Chroma store can actually be opened."""
    try:
        manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
        collection_name = manifest.get("vector_collection_name") or manifest.get("collection_name")
        chroma_path = Path(manifest.get("chroma_dir") or manifest.get("vector_chroma_dir") or default_chroma_dir)
        if not collection_name or not chroma_path.exists():
            return False
        return chroma_collection_is_readable(chroma_path, collection_name)
    except Exception as exc:
        print(f"Warning: could not validate vector cache {manifest_path}: {exc}")
        return False

def _extract_and_chunk(job: Dict[str, Any]) -> Dict[str, Any]:
    pdf_path = job["pdf_path"]
    units_path = job["units_path"]
    chunks_path = job["chunks_path"]
    paper_stem = job["paper_stem"]

    if (
        REUSE_EXISTING_UNITS_AND_CHUNKS
        and not FORCE_REBUILD_ALL
        and units_path.exists()
        and chunks_path.exists()
        and _artifacts_are_current(chunks_path, [pdf_path])
        and _cached_equation_latex_artifacts_are_current(job)
    ):
        units = read_jsonl(units_path)
        chunks = normalize_chunks_for_paper(read_jsonl(chunks_path), paper_stem=paper_stem)
        # Upgrade older cache files once so future runs and Open WebUI services see paper_stem.
        write_jsonl(chunks_path, chunks)
        return {**job, "units": units, "chunks": chunks, "reused_units_chunks": True}

    units = extract_pdf_units(pdf_path, job["vector_dir"])
    chunks = normalize_chunks_for_paper(build_chunks(units), paper_stem=paper_stem)
    write_jsonl(units_path, units)
    write_jsonl(chunks_path, chunks)
    return {**job, "units": units, "chunks": chunks, "reused_units_chunks": False}


jobs = [_paper_paths(p) for p in pdf_files]
extracted_jobs: List[Dict[str, Any]] = []

print(f"PDF extraction workers: {PDF_EXTRACTION_WORKERS}")
if PDF_EXTRACTION_WORKERS > 1 and len(jobs) > 1:
    with ThreadPoolExecutor(max_workers=PDF_EXTRACTION_WORKERS) as pool:
        future_map = {pool.submit(_extract_and_chunk, job): job for job in jobs}
        for fut in tqdm(as_completed(future_map), total=len(future_map), desc="PDF extraction/chunking"):
            job = future_map[fut]
            try:
                extracted_jobs.append(fut.result())
            except Exception as exc:
                print(f"Extraction failed for {job['pdf_path'].name}: {exc}")
                raise
    # Restore original PDF ordering for deterministic manifests.
    order = {str(p): i for i, p in enumerate(pdf_files)}
    extracted_jobs.sort(key=lambda j: order[str(j["pdf_path"])])
else:
    for job in tqdm(jobs, desc="PDF extraction/chunking"):
        extracted_jobs.append(_extract_and_chunk(job))

for job in extracted_jobs:
    pdf_path = job["pdf_path"]
    paper_stem = job["paper_stem"]
    vector_dir = job["vector_dir"]
    graph_dir = job["graph_dir"]
    chunks = job["chunks"]

    print(f"\n=== Building RAG for {pdf_path.name} ===")

    # Profile extraction is LLM-backed, so cache/reuse it whenever possible.
    if (
        REUSE_EXISTING_PROFILES
        and not FORCE_REBUILD_ALL
        and job["profile_path"].exists()
        and _artifacts_are_current(job["profile_path"], [job["chunks_path"]])
    ):
        profile = json.loads(job["profile_path"].read_text(encoding="utf-8"))
    else:
        profile = extract_paper_profile(pdf_path, chunks)
        job["profile_path"].write_text(json.dumps(profile, indent=2, ensure_ascii=False), encoding="utf-8")
    job["graph_profile_path"].write_text(json.dumps(profile, indent=2, ensure_ascii=False), encoding="utf-8")

    section_chunks = build_paper_section_chunks(profile)

    # VectorRAG: reuse existing vector store if it is newer than chunks and the user did not request rebuild.
    vector_manifest_path = vector_dir / "manifest.json"
    if (
        REUSE_EXISTING_VECTOR_STORES
        and not FORCE_REBUILD_CHROMA
        and vector_manifest_path.exists()
        and _vector_manifest_matches_current_embedding(vector_manifest_path)
        and _artifacts_are_current(vector_manifest_path, [job["chunks_path"]])
        and _vector_store_cache_is_usable(vector_manifest_path, vector_dir / "chroma_db")
    ):
        old_manifest = json.loads(vector_manifest_path.read_text(encoding="utf-8"))
        vector_info = {
            "collection_name": old_manifest.get("vector_collection_name") or old_manifest.get("collection_name"),
            "chroma_dir": old_manifest.get("chroma_dir") or old_manifest.get("vector_chroma_dir") or str(vector_dir / "chroma_db"),
            "count": int(old_manifest.get("vector_count", 0) or old_manifest.get("count", 0) or 0),
            "reused_existing_vector_store": True,
        }
    else:
        vector_info = build_per_pdf_vector_store(paper_stem, vector_dir, chunks)

    # GraphRAG: either reuse the graph bundle or build/update from selected high-value chunks.
    graph_manifest_path = graph_dir / "manifest.json"
    graph_json_path = graph_dir / "knowledge_graph.json"
    if (
        RUN_KG_EXTRACTION
        and REUSE_EXISTING_GRAPHS
        and not FORCE_REBUILD_GRAPHS
        and graph_manifest_path.exists()
        and graph_json_path.exists()
        and _artifacts_are_current(graph_manifest_path, [job["chunks_path"], job["profile_path"]])
    ):
        G = load_graph_from_json(graph_json_path)
        selected_kg_chunks = read_jsonl(job["selected_kg_chunks_path"])
        graph_files = {
            "knowledge_graph_json": str(graph_json_path),
            "knowledge_graph_graphml": str(graph_dir / "knowledge_graph.graphml"),
            "knowledge_graph_html": str(graph_dir / "knowledge_graph.html"),
            "reused_existing_graph": True,
        }
    elif RUN_KG_EXTRACTION:
        G, selected_kg_chunks = build_knowledge_graph(chunks, profile, graph_dir)
        write_jsonl(job["selected_kg_chunks_path"], selected_kg_chunks)
        graph_files = save_graph_bundle(G, graph_dir)
    else:
        G = nx.MultiDiGraph()
        selected_kg_chunks = []
        write_jsonl(job["selected_kg_chunks_path"], selected_kg_chunks)
        graph_files = {}

    vector_manifest = {
        "paper_stem": paper_stem,
        "file_name": pdf_path.name,
        "paper_profile": str(job["profile_path"]),
        "units": str(job["units_path"]),
        "chunks": str(job["chunks_path"]),
        "vector_dir": str(vector_dir),
        "vector_collection_name": vector_info["collection_name"],
        "chroma_dir": vector_info["chroma_dir"],
        "vector_chroma_dir": vector_info["chroma_dir"],
        "vector_count": vector_info["count"],
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimensions": EMBEDDING_DIMENSIONS if EMBEDDING_DIMENSIONS is not None else "model_default",
        "provider": "openai",
        "device_mode": DEVICE_MODE,
        "llm_learning_pack_note": "See <paper>_vector/llm_learning_pack/ for audit-ready extraction candidates.",
        "note": "Per-paper VectorRAG bundle. Use retrieval at query time for grounded answers.",
    }
    vector_manifest_path.write_text(json.dumps(vector_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    graph_manifest = {
        "paper_stem": paper_stem,
        "file_name": pdf_path.name,
        "paper_profile": str(job["graph_profile_path"]),
        "selected_kg_chunks": str(job["selected_kg_chunks_path"]),
        "graph_dir": str(graph_dir),
        "graph_node_count": G.number_of_nodes(),
        "graph_edge_count": G.number_of_edges(),
        "kg_chunk_limit": MAX_CHUNKS_PER_PDF_FOR_KG,
        "device_mode": DEVICE_MODE,
        **graph_files,
        "note": "Per-paper GraphRAG bundle. Use together with vector retrieval for best answers.",
    }
    graph_manifest_path.write_text(json.dumps(graph_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    registry_row = {
        "paper_stem": paper_stem,
        "file_name": pdf_path.name,
        "source_path": str(pdf_path),
        "title": profile.get("title", pdf_path.stem),
        "vector_dir": str(vector_dir),
        "graph_dir": str(graph_dir),
        "vector_collection_name": vector_info["collection_name"],
        "chroma_dir": vector_info["chroma_dir"],
        "vector_chroma_dir": vector_info["chroma_dir"],
        "vector_count": vector_info["count"],
        "graph_node_count": G.number_of_nodes(),
        "graph_edge_count": G.number_of_edges(),
        "device_mode": DEVICE_MODE,
    }
    paper_registry.append(registry_row)
    all_profiles.append(profile)
    all_corpus_chunks.extend(section_chunks)

paper_registry_path = MANIFEST_DIR / "paper_registry.json"
paper_registry_path.write_text(json.dumps(paper_registry, indent=2, ensure_ascii=False), encoding="utf-8")
profiles_path = MANIFEST_DIR / "paper_profiles.json"
profiles_path.write_text(json.dumps(all_profiles, indent=2, ensure_ascii=False), encoding="utf-8")
corpus_chunks_path = MANIFEST_DIR / "paper_profile_chunks.jsonl"
write_jsonl(corpus_chunks_path, all_corpus_chunks)

build_summary = {
    "device_mode": DEVICE_MODE,
    "pdf_count": len(pdf_files),
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "answer_model": ANSWER_MODEL,
    "embedding_dimensions": EMBEDDING_DIMENSIONS if EMBEDDING_DIMENSIONS is not None else "model_default",
    "provider": "openai",
    "openai_reasoning_effort": OPENAI_REASONING_EFFORT,
    "openai_text_verbosity": OPENAI_TEXT_VERBOSITY,
    "embedding_cache_enabled": EMBEDDING_CACHE_ENABLED,
    "force_rebuild_all": FORCE_REBUILD_ALL,
    "force_rebuild_chroma": FORCE_REBUILD_CHROMA,
    "force_rebuild_graphs": FORCE_REBUILD_GRAPHS,
    "kg_extraction_enabled": RUN_KG_EXTRACTION,
    "kg_chunk_limit": MAX_CHUNKS_PER_PDF_FOR_KG,
    "pdf_extraction_workers": PDF_EXTRACTION_WORKERS,
}
(MANIFEST_DIR / "build_summary.json").write_text(json.dumps(build_summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nDone.")
print("Registry:", paper_registry_path)
print("Profiles:", profiles_path)
print("Paper profile chunks:", corpus_chunks_path)
print("Build summary:", MANIFEST_DIR / "build_summary.json")
pd.DataFrame(paper_registry)


Found 2 PDF(s).
PDF extraction workers: 4


PDF extraction/chunking:   0%|          | 0/2 [00:00<?, ?it/s]





















































































PDF extraction/chunking:  50%|█████     | 1/2 [06:44<06:44, 404.55s/it]

Equation LaTeX layer for mitra.pdf: 29 verified equation(s), 29 false positive(s), 28 scanned false-negative candidate(s).


PDF extraction/chunking: 100%|██████████| 2/2 [09:08<00:00, 274.21s/it]


Equation LaTeX layer for regolith.pdf: 35 verified equation(s), 30 false positive(s), 59 scanned false-negative candidate(s).

=== Building RAG for mitra.pdf ===


GraphML export failed for mitra_graph: All strings must be XML compatible: Unicode or ASCII, no NULL bytes or control characters

=== Building RAG for regolith.pdf ===



Done.
Registry: rag_output_openai/manifests/paper_registry.json
Profiles: rag_output_openai/manifests/paper_profiles.json
Paper profile chunks: rag_output_openai/manifests/paper_profile_chunks.jsonl
Build summary: rag_output_openai/manifests/build_summary.json


,paper_stem,file_name,source_path,title,vector_dir,graph_dir,vector_collection_name,chroma_dir,vector_chroma_dir,vector_count,graph_node_count,graph_edge_count,device_mode
0,mitra,mitra.pdf,myPDFs/mitra.pdf,The Impact of the Feed Rate and the Binder Con...,rag_output_openai/mitra_vector,rag_output_openai/mitra_graph,mitra_vector,rag_output_openai/mitra_vector/chroma_db,rag_output_openai/mitra_vector/chroma_db,229,164,501,openai_api
1,regolith,regolith.pdf,myPDFs/regolith.pdf,Statistical Analysis and Modeling of the 3D Mo...,rag_output_openai/regolith_vector,rag_output_openai/regolith_graph,regolith_vector,rag_output_openai/regolith_vector/chroma_db,rag_output_openai/regolith_vector/chroma_db,199,216,671,openai_api


## 9. Build corpus-level comparison artifacts

In [11]:

def build_corpus_vector_store(corpus_chunks: List[Dict[str, Any]], out_dir: Path) -> Dict[str, Any]:
    chroma_dir = prepare_chroma_dir(out_dir / "chroma_db", force_rebuild=True)
    collection_name = "corpus_vector"
    client, chroma_dir = create_chroma_persistent_client(chroma_dir, repair_if_needed=True)
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine", "embedding_model": EMBEDDING_MODEL, "provider": "openai"},
    )

    index_rows = dedupe_records_by_id(corpus_chunks, key="chunk_id")

    batch_size = CHROMA_UPSERT_BATCH_SIZE
    for start in tqdm(range(0, len(index_rows), batch_size), desc="Corpus vector store", leave=False):
        batch = index_rows[start:start + batch_size]
        texts = [c["text"] for c in batch]
        ids = [c["chunk_id"] for c in batch]
        metas = [sanitize_chroma_metadata({
            "paper_stem": c["paper_stem"],
            "file_name": c["file_name"],
            "section": c["section"],
            "citation": c["citation"],
            "title": c["title"],
        }) for c in batch]
        embeddings = openai_embed(texts, model=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH_SIZE)
        collection.upsert(ids=ids, documents=texts, metadatas=metas, embeddings=embeddings)

    return {
        "collection_name": collection_name,
        "chroma_dir": str(chroma_dir),
        "count": collection.count(),
    }



def merge_graph_attributes(target: Dict[str, Any], source: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(target)
    for key, value in source.items():
        if key in ("citations", "file_name", "paper_stem", "pages"):
            existing = str(out.get(key, ""))
            incoming = str(value or "")
            pieces = [x for x in (existing.split(" || ") + incoming.split(" || ")) if x]
            out[key] = " || ".join(sorted(set(pieces))) if pieces else ""
        elif key == "description":
            existing = str(out.get("description", ""))
            incoming = str(value or "")
            if incoming and incoming not in existing:
                out[key] = (existing + " | " + incoming).strip(" |")[:1500]
        else:
            out.setdefault(key, value)
    return out


def profile_compare_text(profile: Dict[str, Any]) -> str:
    return clean_text(
        f"Title: {profile.get('title', '')}\n"
        f"Keywords: {', '.join(profile.get('keywords', []))}\n"
        f"Topics: {', '.join(profile.get('main_topics', []))}\n"
        f"Methods: {ensure_text(profile.get('methods'))}\n"
        f"Materials or datasets: {ensure_text(profile.get('materials_or_datasets'))}\n"
        f"Results: {ensure_text(profile.get('results'))}\n"
        f"Conclusions: {ensure_text(profile.get('conclusions'))}\n"
        f"Limitations: {ensure_text(profile.get('limitations'))}"
    )


def build_pairwise_similarity(profiles: List[Dict[str, Any]], top_k: int = PAIRWISE_TOP_K) -> List[Dict[str, Any]]:
    if not profiles:
        return []
    texts = [profile_compare_text(p) for p in profiles]
    embeddings = np.array(openai_embed(texts, model=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH_SIZE), dtype=float)
    sim = cosine_similarity_matrix(embeddings)
    rows = []
    seen_pairs = set()
    for i, p in enumerate(profiles):
        ranking = np.argsort(-sim[i])
        kept = 0
        for j in ranking:
            if i == j:
                continue
            pair_key = tuple(sorted((profiles[i]["paper_stem"], profiles[j]["paper_stem"])))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)

            pi = profiles[i]
            pj = profiles[j]
            methods_i = set(x.lower() for x in ensure_list_of_strings(pi.get("methods")))
            methods_j = set(x.lower() for x in ensure_list_of_strings(pj.get("methods")))
            results_i = set(x.lower() for x in ensure_list_of_strings(pi.get("results")))
            results_j = set(x.lower() for x in ensure_list_of_strings(pj.get("results")))
            concl_i = set(x.lower() for x in ensure_list_of_strings(pi.get("conclusions")))
            concl_j = set(x.lower() for x in ensure_list_of_strings(pj.get("conclusions")))
            topics_i = set(x.lower() for x in ensure_list_of_strings(pi.get("main_topics")))
            topics_j = set(x.lower() for x in ensure_list_of_strings(pj.get("main_topics")))

            rows.append({
                "paper_a": pi["paper_stem"],
                "paper_b": pj["paper_stem"],
                "file_a": pi["file_name"],
                "file_b": pj["file_name"],
                "title_a": pi.get("title", pi["file_name"]),
                "title_b": pj.get("title", pj["file_name"]),
                "similarity": float(sim[i, j]),
                "shared_methods": sorted(methods_i & methods_j),
                "shared_results_terms": sorted(results_i & results_j),
                "shared_conclusion_terms": sorted(concl_i & concl_j),
                "shared_topics": sorted(topics_i & topics_j),
            })
            kept += 1
            if kept >= top_k:
                break
    rows.sort(key=lambda x: x["similarity"], reverse=True)
    return rows


def build_corpus_graph(paper_registry: List[Dict[str, Any]], profiles: List[Dict[str, Any]], similarity_rows: List[Dict[str, Any]], out_dir: Path) -> nx.MultiDiGraph:
    profile_by_stem = {p["paper_stem"]: p for p in profiles}
    G = nx.MultiDiGraph()

    for row in paper_registry:
        paper_stem = row["paper_stem"]
        profile = profile_by_stem.get(paper_stem, {})
        paper_node = f"Paper:{paper_stem}"
        G.add_node(
            paper_node,
            label=profile.get("title", row["file_name"]),
            type="Document",
            description=profile.get("summary") or profile.get("abstract") or row["file_name"],
            citations=row["file_name"],
            file_name=row["file_name"],
            paper_stem=paper_stem,
        )

        graph_json_path = Path(row["graph_dir"]) / "knowledge_graph.json"
        if not graph_json_path.exists():
            continue
        sub = load_graph_from_json(graph_json_path)

        for node_id, attrs in sub.nodes(data=True):
            if str(node_id).startswith("Chunk:"):
                continue
            if G.has_node(node_id):
                G.nodes[node_id].update(merge_graph_attributes(G.nodes[node_id], attrs))
            else:
                G.add_node(node_id, **attrs)
            if not str(node_id).startswith("Paper:"):
                edge_key = f"REPORTED_IN:{stable_id(node_id, paper_node)}"
                if not G.has_edge(node_id, paper_node, key=edge_key):
                    G.add_edge(
                        node_id,
                        paper_node,
                        key=edge_key,
                        type="REPORTED_IN",
                        description="concept reported in this paper",
                        evidence=row["file_name"],
                        confidence="1.0",
                        citations=row["file_name"],
                    )

        for u, v, key, attrs in sub.edges(keys=True, data=True):
            if str(u).startswith("Chunk:") or str(v).startswith("Chunk:"):
                continue
            if str(key).startswith("PART_OF:"):
                continue
            G.add_edge(u, v, key=key, **attrs)

    for sim_row in similarity_rows:
        a = f"Paper:{sim_row['paper_a']}"
        b = f"Paper:{sim_row['paper_b']}"
        G.add_edge(
            a,
            b,
            key=f"COMPARES_WITH:{stable_id(a, b)}",
            type="COMPARES_WITH",
            description=f"Embedding/profile similarity={sim_row['similarity']:.3f}; shared topics: {', '.join(sim_row['shared_topics'][:8])}",
            evidence=f"{sim_row['file_a']} || {sim_row['file_b']}",
            confidence=f"{min(1.0, max(0.0, sim_row['similarity'])):.3f}",
            citations=f"{sim_row['file_a']} || {sim_row['file_b']}",
        )

    graph_files = save_graph_bundle(G, out_dir)
    return G


def export_finetune_seed_files(profiles: List[Dict[str, Any]], similarity_rows: List[Dict[str, Any]], out_dir: Path) -> Dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)

    profile_rows = []
    for p in profiles:
        system = "Answer using only the supplied paper profile. Be concise and factual."
        prompts = [
            (f"Summarize the methodology of {p.get('title', p['file_name'])}.", ensure_text(p.get("methods")) or ensure_text(p.get("summary"))),
            (f"What are the main results of {p.get('title', p['file_name'])}?", ensure_text(p.get("results"))),
            (f"What conclusions does {p.get('title', p['file_name'])} report?", ensure_text(p.get("conclusions"))),
            (f"What limitations are described in {p.get('title', p['file_name'])}?", ensure_text(p.get("limitations")) or "No explicit limitations extracted."),
        ]
        profile_text = profile_compare_text(p)
        for user_prompt, answer_text in prompts:
            if not answer_text:
                continue
            profile_rows.append({
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": f"{user_prompt}\n\nPaper profile:\n{profile_text}"},
                    {"role": "assistant", "content": answer_text},
                ],
                "paper_stem": p["paper_stem"],
                "file_name": p["file_name"],
                "task": "paper_profile_qa",
            })

    compare_rows = []
    for row in similarity_rows[: max(20, len(similarity_rows))]:
        answer = (
            f"Similarities in methods/topics: {', '.join(row['shared_methods'][:8] + row['shared_topics'][:8]) or 'No strong overlap extracted.'} "
            f"Shared result terms: {', '.join(row['shared_results_terms'][:8]) or 'none extracted'}. "
            f"Shared conclusion terms: {', '.join(row['shared_conclusion_terms'][:8]) or 'none extracted'}."
        )
        compare_rows.append({
            "messages": [
                {"role": "system", "content": "Compare papers only from the supplied structured comparison data."},
                {"role": "user", "content": f"Compare {row['title_a']} and {row['title_b']} in terms of methodology, results, and conclusions.\n\nStructured comparison data:\n{json.dumps(row, ensure_ascii=False)}"},
                {"role": "assistant", "content": answer},
            ],
            "paper_a": row["paper_a"],
            "paper_b": row["paper_b"],
            "task": "paper_comparison_qa",
        })

    profile_path = out_dir / "paper_profile_qa_seed.jsonl"
    compare_path = out_dir / "paper_comparison_qa_seed.jsonl"
    write_jsonl(profile_path, profile_rows)
    write_jsonl(compare_path, compare_rows)
    return {"paper_profile_qa_seed": str(profile_path), "paper_comparison_qa_seed": str(compare_path)}




def _corpus_vector_store_cache_is_usable(manifest_path: Path, default_chroma_dir: Path) -> bool:
    """Validate that an existing corpus-level Chroma store can actually be opened."""
    try:
        manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
        collection_name = manifest.get("collection_name", "corpus_vector")
        chroma_path = Path(manifest.get("chroma_dir") or default_chroma_dir)
        if not chroma_path.exists():
            return False
        return chroma_collection_is_readable(chroma_path, collection_name)
    except Exception as exc:
        print(f"Warning: could not validate corpus vector cache {manifest_path}: {exc}")
        return False

if BUILD_CORPUS_COMPARISON:
    corpus_chunks_manifest_path = CORPUS_VECTOR_DIR / "paper_profile_chunks.jsonl"
    write_jsonl(corpus_chunks_manifest_path, all_corpus_chunks)
    corpus_vector_manifest_path = CORPUS_VECTOR_DIR / "manifest.json"
    if (
        REUSE_EXISTING_VECTOR_STORES
        and not FORCE_REBUILD_CHROMA
        and corpus_vector_manifest_path.exists()
        and _vector_manifest_matches_current_embedding(corpus_vector_manifest_path)
        and _corpus_vector_store_cache_is_usable(corpus_vector_manifest_path, CORPUS_VECTOR_DIR / "chroma_db")
        and _artifacts_are_current(corpus_vector_manifest_path, [corpus_chunks_manifest_path])
    ):
        old_manifest = json.loads(corpus_vector_manifest_path.read_text(encoding="utf-8"))
        corpus_vector_info = {
            "collection_name": old_manifest.get("collection_name", "corpus_vector"),
            "chroma_dir": old_manifest.get("chroma_dir", str(CORPUS_VECTOR_DIR / "chroma_db")),
            "count": int(old_manifest.get("count", 0) or 0),
            "reused_existing_corpus_vector_store": True,
        }
    else:
        corpus_vector_info = build_corpus_vector_store(all_corpus_chunks, CORPUS_VECTOR_DIR)

    similarity_rows = build_pairwise_similarity(all_profiles, top_k=PAIRWISE_TOP_K)
    similarity_path = MANIFEST_DIR / "paper_similarity.json"
    similarity_path.write_text(json.dumps(similarity_rows, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame(similarity_rows).to_csv(MANIFEST_DIR / "paper_similarity.csv", index=False)

    corpus_graph = build_corpus_graph(paper_registry, all_profiles, similarity_rows, CORPUS_GRAPH_DIR)

    corpus_vector_manifest = {
        "collection_name": corpus_vector_info["collection_name"],
        "chroma_dir": corpus_vector_info["chroma_dir"],
        "count": corpus_vector_info["count"],
        "paper_profile_chunks": str(corpus_chunks_manifest_path),
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimensions": EMBEDDING_DIMENSIONS if EMBEDDING_DIMENSIONS is not None else "model_default",
        "provider": "openai",
        "note": "Corpus-level VectorRAG for cross-paper comparison and routing.",
    }
    (CORPUS_VECTOR_DIR / "manifest.json").write_text(json.dumps(corpus_vector_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    corpus_graph_manifest = {
        "graph_dir": str(CORPUS_GRAPH_DIR),
        "paper_similarity_json": str(similarity_path),
        "node_count": corpus_graph.number_of_nodes(),
        "edge_count": corpus_graph.number_of_edges(),
        "note": "Corpus-level GraphRAG for cross-paper comparison and shared-concept traversal.",
    }
    (CORPUS_GRAPH_DIR / "manifest.json").write_text(json.dumps(corpus_graph_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    if EXPORT_FINETUNE_FILES:
        finetune_paths = export_finetune_seed_files(all_profiles, similarity_rows, FINETUNE_EXPORT_DIR)
    else:
        finetune_paths = {}

    print("Corpus vector dir:", CORPUS_VECTOR_DIR)
    print("Corpus graph dir:", CORPUS_GRAPH_DIR)
    print("Similarity file:", similarity_path)
    if finetune_paths:
        print("Fine-tune seed files:", finetune_paths)
else:
    similarity_rows = []
    finetune_paths = {}
    print("Corpus comparison build skipped.")


GraphML export failed for corpus_graph: All strings must be XML compatible: Unicode or ASCII, no NULL bytes or control characters
Corpus vector dir: rag_output_openai/corpus_vector
Corpus graph dir: rag_output_openai/corpus_graph
Similarity file: rag_output_openai/manifests/paper_similarity.json
Fine-tune seed files: {'paper_profile_qa_seed': 'rag_output_openai/finetune_exports/paper_profile_qa_seed.jsonl', 'paper_comparison_qa_seed': 'rag_output_openai/finetune_exports/paper_comparison_qa_seed.jsonl'}


## 10. Production inference flow: per-paper retrieval first, corpus retrieval second, answer synthesis third

The helper functions below implement a production-style query flow:

1. **Route the question** to likely paper(s) using explicit paper-name matching plus corpus-level vector and graph routing.
2. **Retrieve detailed evidence from the selected paper stores first** (`<paper>_vector` and `<paper>_graph`).
3. **Retrieve corpus-level comparison context second** from `corpus_vector` and `corpus_graph`.
4. **Synthesize the answer** from retrieved evidence only, with automatic comparison mode when the question or routing indicates a comparison task.

Use `answer_with_production_flow(...)` for the default public-facing query path.


In [12]:

paper_registry_by_stem = {row["paper_stem"]: row for row in paper_registry}
paper_registry_by_file = {row["file_name"]: row for row in paper_registry}

_PER_PAPER_RESOURCE_CACHE: Dict[str, Tuple[Dict[str, Dict[str, Any]], nx.MultiDiGraph, Any]] = {}
_CORPUS_GRAPH_CACHE: Optional[nx.MultiDiGraph] = None
_CORPUS_PROFILE_CHUNKS_CACHE: Optional[List[Dict[str, Any]]] = None

COMPARISON_KEYWORDS = [
    "compare", "comparison", "versus", "vs", "similar", "similarity", "different", "difference",
    "differ", "across papers", "across the papers", "between papers", "between the papers",
    "methodologies", "methodology differences", "results differences", "conclusions differences",
    "similarities", "differences", "which paper", "how do the papers", "paper comparison",
]


def resolve_paper_entry(name_or_stem: str) -> Dict[str, Any]:
    key = safe_filename(str(name_or_stem).replace(".pdf", ""))
    if key in paper_registry_by_stem:
        return paper_registry_by_stem[key]
    if name_or_stem in paper_registry_by_file:
        return paper_registry_by_file[name_or_stem]
    raise KeyError(f"Paper not found: {name_or_stem}")


def normalize_lookup_text(text: str) -> str:
    text = clean_text(str(text or "")).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def paper_aliases(entry: Dict[str, Any]) -> List[str]:
    aliases = {
        str(entry.get("paper_stem", "")),
        str(entry.get("file_name", "")).replace(".pdf", ""),
        str(entry.get("title", "")),
    }
    out = []
    for alias in aliases:
        alias = clean_text(alias)
        if alias and len(alias) >= 3:
            out.append(alias)
    return sorted(set(out))


def alias_match_score(question: str, alias: str) -> float:
    qnorm = normalize_lookup_text(question)
    anorm = normalize_lookup_text(alias)
    if not anorm:
        return 0.0
    if anorm in qnorm and len(anorm) >= 5:
        return min(1.0, 0.85 + 0.01 * min(10, len(anorm)))
    if fuzz is not None and len(anorm) >= 8:
        return max(
            float(fuzz.token_set_ratio(qnorm, anorm)) / 100.0,
            float(fuzz.partial_ratio(qnorm, anorm)) / 100.0,
        )
    q_terms = set(qnorm.split())
    a_terms = set(anorm.split())
    if not a_terms:
        return 0.0
    return len(q_terms & a_terms) / max(1, len(a_terms))


def infer_explicit_papers(question: str, max_matches: int = 4) -> List[Dict[str, Any]]:
    rows = []
    for entry in paper_registry:
        best_alias, best_score = "", 0.0
        for alias in paper_aliases(entry):
            score = alias_match_score(question, alias)
            if score > best_score:
                best_alias, best_score = alias, score
        if best_score >= 0.90:
            rows.append({
                "paper_stem": entry["paper_stem"],
                "file_name": entry["file_name"],
                "title": entry.get("title", entry["file_name"]),
                "score": float(best_score),
                "signal": "explicit_match",
                "matched_alias": best_alias,
            })
    rows.sort(key=lambda x: x["score"], reverse=True)
    return rows[:max_matches]


def open_collection(chroma_dir: Path, collection_name: str):
    client, _actual_chroma_dir = create_chroma_persistent_client(Path(chroma_dir), repair_if_needed=False)
    return client.get_collection(collection_name)


def vector_search_collection(collection, query: str, top_k: int = VECTOR_TOP_K) -> List[Dict[str, Any]]:
    qemb = openai_embed([query], model=EMBEDDING_MODEL, batch_size=1)[0]
    res = collection.query(query_embeddings=[qemb], n_results=top_k, include=["documents", "metadatas", "distances"])
    hits = []
    for doc, meta, dist, cid in zip(
        res.get("documents", [[]])[0],
        res.get("metadatas", [[]])[0],
        res.get("distances", [[]])[0],
        res.get("ids", [[]])[0],
    ):
        hits.append({
            "chunk_id": cid,
            "text": doc,
            "metadata": meta,
            "distance": dist,
            "score": 1.0 / (1.0 + max(0.0, float(dist))),
            "source": "vector",
        })
    return hits


def graph_node_score(query: str, node_attrs: Dict[str, Any]) -> float:
    target = " ".join(str(node_attrs.get(k, "")) for k in ["label", "type", "description"])
    if not target.strip():
        return 0.0
    if fuzz is not None:
        return float(fuzz.token_set_ratio(query, target)) / 100.0
    q_terms = set(re.findall(r"[A-Za-z0-9_]+", query.lower()))
    t_terms = set(re.findall(r"[A-Za-z0-9_]+", target.lower()))
    return len(q_terms & t_terms) / max(1, len(q_terms))


def graph_search_graph(
    G: nx.MultiDiGraph,
    chunk_lookup: Dict[str, Dict[str, Any]],
    query: str,
    top_nodes: int = GRAPH_TOP_NODES,
    hops: int = GRAPH_HOPS,
) -> List[Dict[str, Any]]:
    scored = []
    for node_id, attrs in G.nodes(data=True):
        if str(node_id).startswith("Chunk:"):
            continue
        scored.append((graph_node_score(query, attrs), node_id, attrs))
    scored.sort(reverse=True, key=lambda x: x[0])
    seeds = [node_id for score, node_id, attrs in scored[:top_nodes] if score > 0]
    evidence_chunk_ids = set()
    for seed in seeds:
        neighborhood, frontier = {seed}, {seed}
        for _ in range(hops):
            new_frontier = set()
            for n in frontier:
                new_frontier.update(G.successors(n))
                new_frontier.update(G.predecessors(n))
            neighborhood.update(new_frontier)
            frontier = new_frontier
        for n in neighborhood:
            if str(n).startswith("Chunk:"):
                evidence_chunk_ids.add(str(n).split("Chunk:", 1)[1])

    hits = []
    for cid in evidence_chunk_ids:
        c = chunk_lookup.get(cid)
        if c:
            hits.append({
                "chunk_id": cid,
                "text": c["text"],
                "metadata": {
                    "citation": c["citation"],
                    "file_name": c["file_name"],
                    "page": c["page"],
                    "element_type": c["element_type"],
                    "paper_stem": c["paper_stem"],
                },
                "distance": None,
                "score": 0.75,
                "source": "graph",
            })
    return hits


def load_per_paper_resources(paper_entry: Dict[str, Any]) -> Tuple[Dict[str, Dict[str, Any]], nx.MultiDiGraph, Any]:
    paper_stem = paper_entry["paper_stem"]
    if paper_stem in _PER_PAPER_RESOURCE_CACHE:
        return _PER_PAPER_RESOURCE_CACHE[paper_stem]

    vector_dir = Path(paper_entry["vector_dir"])
    graph_dir = Path(paper_entry["graph_dir"])
    chunks = normalize_chunks_for_paper(read_jsonl(vector_dir / f"{paper_stem}_chunks.jsonl"), paper_stem=paper_stem)
    chunk_lookup = {c["chunk_id"]: c for c in chunks}
    G = load_graph_from_json(graph_dir / "knowledge_graph.json") if (graph_dir / "knowledge_graph.json").exists() else nx.MultiDiGraph()
    manifest_path = vector_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    chroma_path = Path(manifest.get("chroma_dir") or manifest.get("vector_chroma_dir") or (vector_dir / "chroma_db"))
    collection = open_collection(chroma_path, paper_entry["vector_collection_name"])
    _PER_PAPER_RESOURCE_CACHE[paper_stem] = (chunk_lookup, G, collection)
    return _PER_PAPER_RESOURCE_CACHE[paper_stem]


def hybrid_retrieve_single_paper(
    paper_name_or_stem: str,
    query: str,
    max_context_chunks: int = MAX_CONTEXT_CHUNKS,
) -> List[Dict[str, Any]]:
    entry = resolve_paper_entry(paper_name_or_stem)
    chunk_lookup, G, collection = load_per_paper_resources(entry)

    vector_hits = vector_search_collection(collection, query, VECTOR_TOP_K)
    graph_hits = graph_search_graph(G, chunk_lookup, query, GRAPH_TOP_NODES, GRAPH_HOPS)

    merged, seen = [], set()
    for hit in sorted(vector_hits + graph_hits, key=lambda x: float(x.get("score", 0.0)), reverse=True):
        cid = hit["chunk_id"]
        if cid in seen:
            continue
        seen.add(cid)
        hit["paper_stem"] = entry["paper_stem"]
        merged.append(hit)
        if len(merged) >= max_context_chunks:
            break
    return merged


def corpus_vector_search(query: str, top_k: int = CORPUS_ROUTER_TOP_K) -> List[Dict[str, Any]]:
    manifest_path = CORPUS_VECTOR_DIR / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        chroma_path = Path(manifest.get("chroma_dir") or (CORPUS_VECTOR_DIR / "chroma_db"))
        collection_name = manifest.get("collection_name", "corpus_vector")
    else:
        chroma_path = CORPUS_VECTOR_DIR / "chroma_db"
        collection_name = "corpus_vector"
    if not chroma_path.exists():
        return []
    collection = open_collection(chroma_path, collection_name)
    return vector_search_collection(collection, query, top_k=top_k)


def load_corpus_graph() -> nx.MultiDiGraph:
    global _CORPUS_GRAPH_CACHE
    if _CORPUS_GRAPH_CACHE is None:
        path = CORPUS_GRAPH_DIR / "knowledge_graph.json"
        _CORPUS_GRAPH_CACHE = load_graph_from_json(path) if path.exists() else nx.MultiDiGraph()
    return _CORPUS_GRAPH_CACHE


def corpus_graph_route_papers(query: str, top_nodes: int = 10, max_papers: int = 6) -> List[Dict[str, Any]]:
    G = load_corpus_graph()
    if G.number_of_nodes() == 0:
        return []

    node_rows = []
    for node_id, attrs in G.nodes(data=True):
        score = graph_node_score(query, attrs)
        if score <= 0:
            continue
        node_rows.append((score, node_id, attrs))
    node_rows.sort(reverse=True, key=lambda x: x[0])

    paper_scores: Dict[str, float] = defaultdict(float)
    paper_evidence: Dict[str, List[str]] = defaultdict(list)

    for score, node_id, attrs in node_rows[:top_nodes]:
        if str(node_id).startswith("Paper:"):
            paper_stem = str(attrs.get("paper_stem") or str(node_id).split("Paper:", 1)[1])
            paper_scores[paper_stem] += float(score) * 1.2
            paper_evidence[paper_stem].append(str(attrs.get("label", node_id)))
            continue

        neighbors = set(G.successors(node_id)) | set(G.predecessors(node_id))
        for nbr in neighbors:
            if str(nbr).startswith("Paper:"):
                nbr_attrs = G.nodes[nbr]
                paper_stem = str(nbr_attrs.get("paper_stem") or str(nbr).split("Paper:", 1)[1])
                paper_scores[paper_stem] += float(score)
                paper_evidence[paper_stem].append(str(attrs.get("label", node_id)))

    rows = []
    for paper_stem, score in paper_scores.items():
        entry = paper_registry_by_stem.get(paper_stem)
        if entry is None:
            continue
        rows.append({
            "paper_stem": paper_stem,
            "file_name": entry["file_name"],
            "title": entry.get("title", entry["file_name"]),
            "score": float(score),
            "signal": "corpus_graph",
            "matched_alias": ", ".join(sorted(set(paper_evidence.get(paper_stem, [])[:6]))),
        })
    rows.sort(key=lambda x: x["score"], reverse=True)
    return rows[:max_papers]


def rank_candidate_papers(
    question: str,
    preferred_papers: Optional[List[str]] = None,
    max_papers: int = ROUTER_TOP_PAPERS,
) -> Dict[str, Any]:
    preferred_papers = preferred_papers or []
    explicit_rows = infer_explicit_papers(question, max_matches=max(4, len(preferred_papers) + 2))
    corpus_vector_hits = corpus_vector_search(question, top_k=max(CORPUS_ROUTER_TOP_K, max_papers * 2))
    graph_rows = corpus_graph_route_papers(question, top_nodes=max(8, max_papers * 2), max_papers=max_papers)

    score_map: Dict[str, float] = defaultdict(float)
    signals: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    for row in explicit_rows:
        score_map[row["paper_stem"]] += 5.0 * float(row["score"])
        signals[row["paper_stem"]].append(row)

    for hit in corpus_vector_hits:
        paper_stem = hit["metadata"].get("paper_stem")
        if not paper_stem:
            continue
        score = float(hit.get("score", 0.0))
        score_map[paper_stem] += 2.0 * score
        signals[paper_stem].append({
            "paper_stem": paper_stem,
            "file_name": hit["metadata"].get("file_name", ""),
            "title": hit["metadata"].get("title", hit["metadata"].get("file_name", "")),
            "score": score,
            "signal": "corpus_vector",
            "matched_alias": hit["metadata"].get("section", ""),
            "citation": hit["metadata"].get("citation", ""),
        })

    for row in graph_rows:
        score_map[row["paper_stem"]] += 1.5 * float(row["score"])
        signals[row["paper_stem"]].append(row)

    for pref in preferred_papers:
        try:
            entry = resolve_paper_entry(pref)
        except Exception:
            continue
        score_map[entry["paper_stem"]] += 10.0
        signals[entry["paper_stem"]].append({
            "paper_stem": entry["paper_stem"],
            "file_name": entry["file_name"],
            "title": entry.get("title", entry["file_name"]),
            "score": 10.0,
            "signal": "preferred_paper",
            "matched_alias": pref,
        })

    ranked = []
    for paper_stem, score in score_map.items():
        entry = paper_registry_by_stem.get(paper_stem)
        if entry is None:
            continue
        ranked.append({
            "paper_stem": paper_stem,
            "file_name": entry["file_name"],
            "title": entry.get("title", entry["file_name"]),
            "score": float(score),
            "signals": signals.get(paper_stem, []),
        })
    ranked.sort(key=lambda x: x["score"], reverse=True)
    ranked = [row for row in ranked if row["score"] >= MIN_ROUTER_SCORE][:max_papers]

    return {
        "question": question,
        "explicit_matches": explicit_rows,
        "corpus_vector_hits": corpus_vector_hits,
        "corpus_graph_matches": graph_rows,
        "ranked_papers": ranked,
    }


def detect_comparison_mode(
    question: str,
    preferred_papers: Optional[List[str]] = None,
    explicit_matches: Optional[List[Dict[str, Any]]] = None,
    ranked_papers: Optional[List[Dict[str, Any]]] = None,
) -> bool:
    preferred_papers = preferred_papers or []
    explicit_matches = explicit_matches or []
    ranked_papers = ranked_papers or []

    q = normalize_lookup_text(question)
    if len(preferred_papers) >= 2:
        return True
    if len(explicit_matches) >= 2:
        return True
    if any(term in q for term in COMPARISON_KEYWORDS):
        return True
    if "papers" in q and len(ranked_papers) >= 2:
        return True
    return False


def dedupe_hits(hits: List[Dict[str, Any]], limit: Optional[int] = None) -> List[Dict[str, Any]]:
    out, seen = [], set()
    for hit in hits:
        chunk_id = hit.get("chunk_id")
        citation = hit.get("metadata", {}).get("citation", "")
        key = chunk_id or citation or stable_id(hit.get("text", "")[:120])
        if key in seen:
            continue
        seen.add(key)
        out.append(hit)
        if limit is not None and len(out) >= limit:
            break
    return out


def load_corpus_profile_chunks() -> List[Dict[str, Any]]:
    global _CORPUS_PROFILE_CHUNKS_CACHE
    if _CORPUS_PROFILE_CHUNKS_CACHE is None:
        manifest_path = MANIFEST_DIR / "paper_profile_chunks.jsonl"
        _CORPUS_PROFILE_CHUNKS_CACHE = read_jsonl(manifest_path)
    return _CORPUS_PROFILE_CHUNKS_CACHE


def build_context_block(hits: List[Dict[str, Any]]) -> str:
    blocks = []
    for hit in hits:
        citation = hit["metadata"].get("citation", "")
        paper_stem = hit["metadata"].get("paper_stem") or hit.get("paper_stem", "")
        source = hit.get("source", "")
        blocks.append(
            f"[{citation}] (paper={paper_stem}; source={source})\n{hit['text']}"
        )
    return "\n\n".join(blocks)


def build_routing_summary(routing: Dict[str, Any], selected_papers: List[str], mode: str) -> str:
    lines = [f"Mode: {mode}", f"Selected papers: {', '.join(selected_papers) if selected_papers else 'none'}"]

    explicit = routing.get("explicit_matches", [])
    if explicit:
        lines.append("Explicit matches:")
        for row in explicit[:6]:
            lines.append(f"- {row['paper_stem']} via '{row.get('matched_alias', '')}' (score={row['score']:.2f})")

    ranked = routing.get("ranked_papers", [])
    if ranked:
        lines.append("Ranked candidates:")
        for row in ranked[:8]:
            sigs = ", ".join(sorted(set(sig.get("signal", "") for sig in row.get("signals", []))))
            lines.append(f"- {row['paper_stem']} score={row['score']:.2f} signals={sigs}")

    return "\n".join(lines)


def retrieve_with_production_flow(
    question: str,
    preferred_papers: Optional[List[str]] = None,
    mode: str = "auto",
    answer_model: str = ANSWER_MODEL,
) -> Dict[str, Any]:
    preferred_papers = preferred_papers or []
    routing = rank_candidate_papers(question, preferred_papers=preferred_papers, max_papers=ROUTER_TOP_PAPERS)

    inferred_compare = detect_comparison_mode(
        question,
        preferred_papers=preferred_papers,
        explicit_matches=routing.get("explicit_matches", []),
        ranked_papers=routing.get("ranked_papers", []),
    )

    if mode == "auto":
        final_mode = "comparison" if (PRODUCTION_AUTO_COMPARISON and inferred_compare) else "single_paper"
    elif mode in {"single_paper", "comparison"}:
        final_mode = mode
    else:
        raise ValueError("mode must be 'auto', 'single_paper', or 'comparison'")

    if preferred_papers:
        selected_papers = []
        for pref in preferred_papers:
            try:
                selected_papers.append(resolve_paper_entry(pref)["paper_stem"])
            except Exception:
                continue
    else:
        ranked = routing.get("ranked_papers", [])
        if final_mode == "single_paper":
            selected_papers = [ranked[0]["paper_stem"]] if ranked else []
        else:
            selected_papers = [row["paper_stem"] for row in ranked[: max(2, CROSS_PAPER_TOP_PAPERS)]]

    selected_papers = list(dict.fromkeys(selected_papers))

    per_paper_hits: List[Dict[str, Any]] = []
    if PRODUCTION_USE_PER_PAPER_FIRST:
        if final_mode == "single_paper":
            per_paper_limit = MAX_CONTEXT_CHUNKS
        else:
            per_paper_limit = CROSS_PAPER_PER_PAPER_HITS
        for paper_stem in selected_papers:
            per_paper_hits.extend(hybrid_retrieve_single_paper(paper_stem, question, max_context_chunks=per_paper_limit))

    corpus_hits: List[Dict[str, Any]] = []
    if PRODUCTION_USE_CORPUS_SECOND:
        raw_corpus_hits = corpus_vector_search(question, top_k=CORPUS_ROUTER_TOP_K)
        if final_mode == "single_paper" and selected_papers:
            selected = set(selected_papers)
            corpus_hits = [hit for hit in raw_corpus_hits if hit["metadata"].get("paper_stem") in selected]
            if not corpus_hits:
                corpus_hits = raw_corpus_hits[: max(4, CORPUS_ROUTER_TOP_K // 2)]
        else:
            candidate_set = set(selected_papers)
            ranked_set = set(row["paper_stem"] for row in routing.get("ranked_papers", [])[:ROUTER_TOP_PAPERS])
            corpus_hits = [hit for hit in raw_corpus_hits if hit["metadata"].get("paper_stem") in candidate_set | ranked_set]
            if not corpus_hits:
                corpus_hits = raw_corpus_hits[:CORPUS_ROUTER_TOP_K]

    final_hits = dedupe_hits(per_paper_hits + corpus_hits, limit=MAX_SYNTHESIS_CONTEXT_HITS)

    return {
        "question": question,
        "mode": final_mode,
        "selected_papers": selected_papers,
        "routing": routing,
        "per_paper_hits": dedupe_hits(per_paper_hits),
        "corpus_hits": dedupe_hits(corpus_hits),
        "final_hits": final_hits,
        "routing_summary": build_routing_summary(routing, selected_papers, final_mode),
        "answer_model": answer_model,
    }


def production_answer_prompt(question: str, package: Dict[str, Any]) -> str:
    mode = package["mode"]
    selected_papers = package.get("selected_papers", [])
    routing_summary = package.get("routing_summary", "")
    per_paper_context = build_context_block(package.get("per_paper_hits", []))
    corpus_context = build_context_block(package.get("corpus_hits", []))

    if mode == "single_paper":
        instruction = """
You are answering a scientific question in production mode.
Use the detailed per-paper evidence as the primary source of truth.
Use corpus-level evidence only to supplement or disambiguate.
Every substantive claim must cite one or more supporting chunk citations in square brackets.
If the evidence is insufficient, say exactly what is missing.
Do not invent methods, results, or conclusions.
When the question asks about equations, prefer `equation_latex` evidence and render equations as LaTeX.
""".strip()
    else:
        instruction = """
You are answering a cross-paper scientific comparison question in production mode.
Use the detailed per-paper evidence as the primary source of truth.
Use corpus-level evidence only as secondary comparison context.
Explicitly call out similarities and differences in methodology, materials/datasets, results, conclusions, and limitations when the evidence supports it.
Every substantive claim must cite one or more supporting chunk citations in square brackets.
If the evidence does not support a comparison category, say so explicitly.
Do not invent consensus, disagreement, or trends.
""".strip()

    return f"""
{instruction}

Question:
{question}

Selected papers:
{", ".join(selected_papers) if selected_papers else "No specific paper confidently selected."}

Routing summary:
{routing_summary}

Primary per-paper evidence:
{per_paper_context or "No per-paper evidence retrieved."}

Secondary corpus evidence:
{corpus_context or "No corpus evidence retrieved."}
""".strip()


def answer_with_production_flow(
    question: str,
    preferred_papers: Optional[List[str]] = None,
    mode: str = "auto",
    answer_model: str = ANSWER_MODEL,
) -> Tuple[str, Dict[str, Any]]:
    package = retrieve_with_production_flow(question, preferred_papers=preferred_papers, mode=mode, answer_model=answer_model)
    prompt = production_answer_prompt(question, package)
    answer = openai_generate(prompt=prompt, model=answer_model, temperature=0.0, timeout=300)
    package["answer"] = answer
    return answer, package


# Backward-compatible helpers
def answer_single_pdf(
    paper_name_or_stem: str,
    question: str,
    answer_model: str = ANSWER_MODEL,
) -> Tuple[str, Dict[str, Any]]:
    return answer_with_production_flow(
        question=question,
        preferred_papers=[paper_name_or_stem],
        mode="single_paper",
        answer_model=answer_model,
    )


def answer_across_papers(
    question: str,
    top_papers: int = CROSS_PAPER_TOP_PAPERS,
    per_paper_hits: int = CROSS_PAPER_PER_PAPER_HITS,
    answer_model: str = ANSWER_MODEL,
) -> Tuple[str, Dict[str, Any]]:
    # top_papers and per_paper_hits remain configurable globally; this wrapper keeps the old API shape.
    return answer_with_production_flow(
        question=question,
        preferred_papers=None,
        mode="comparison",
        answer_model=answer_model,
    )


def compare_specific_papers(
    paper_a: str,
    paper_b: str,
    question: str = "Compare the methodologies, results, conclusions, and limitations.",
    answer_model: str = ANSWER_MODEL,
) -> Tuple[str, Dict[str, Any]]:
    return answer_with_production_flow(
        question=question,
        preferred_papers=[paper_a, paper_b],
        mode="comparison",
        answer_model=answer_model,
    )


def routing_debug_dataframe(question: str, preferred_papers: Optional[List[str]] = None) -> pd.DataFrame:
    package = retrieve_with_production_flow(question, preferred_papers=preferred_papers, mode="auto")
    rows = []
    for row in package["routing"].get("ranked_papers", []):
        rows.append({
            "paper_stem": row["paper_stem"],
            "file_name": row["file_name"],
            "title": row["title"],
            "score": row["score"],
            "signals": ", ".join(sorted(set(sig.get("signal", "") for sig in row.get("signals", [])))),
        })
    return pd.DataFrame(rows)


## 11. Validation and release-readiness checks

In [13]:

def validate_paper_artifacts(entry: Dict[str, Any]) -> Dict[str, Any]:
    vector_dir = Path(entry["vector_dir"])
    graph_dir = Path(entry["graph_dir"])
    paper_stem = entry["paper_stem"]

    vector_manifest_path = vector_dir / "manifest.json"
    vector_chroma_path = vector_dir / "chroma_db"
    if vector_manifest_path.exists():
        try:
            _vm = json.loads(vector_manifest_path.read_text(encoding="utf-8"))
            vector_chroma_path = Path(_vm.get("chroma_dir") or _vm.get("vector_chroma_dir") or vector_chroma_path)
        except Exception:
            pass

    required = {
        "vector_chunks": vector_dir / f"{paper_stem}_chunks.jsonl",
        "vector_units": vector_dir / f"{paper_stem}_units.jsonl",
        "vector_profile": vector_dir / "paper_profile.json",
        "vector_manifest": vector_manifest_path,
        "vector_chroma_dir": vector_chroma_path,
        "graph_json": graph_dir / "knowledge_graph.json",
        "graph_graphml": graph_dir / "knowledge_graph.graphml",
        "graph_nodes_csv": graph_dir / "kg_nodes.csv",
        "graph_edges_csv": graph_dir / "kg_edges.csv",
        "graph_manifest": graph_dir / "manifest.json",
    }

    status = {"paper_stem": paper_stem, "file_name": entry["file_name"]}
    for key, path in required.items():
        status[key] = path.exists()

    try:
        chunks = read_jsonl(vector_dir / f"{paper_stem}_chunks.jsonl")
        status["chunk_count"] = len(chunks)
    except Exception:
        status["chunk_count"] = 0

    try:
        G = load_graph_from_json(graph_dir / "knowledge_graph.json")
        status["graph_nodes"] = G.number_of_nodes()
        status["graph_edges"] = G.number_of_edges()
    except Exception:
        status["graph_nodes"] = 0
        status["graph_edges"] = 0

    return status


validation_rows = [validate_paper_artifacts(entry) for entry in paper_registry]
validation_df = pd.DataFrame(validation_rows)
validation_path = MANIFEST_DIR / "validation_report.csv"
validation_df.to_csv(validation_path, index=False)

master_manifest = {
    "pdf_dir": str(PDF_DIR),
    "out_dir": str(OUT_DIR),
    "generation_model_used_for_extraction": GENERATION_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "answer_model": ANSWER_MODEL,
    "papers": paper_registry,
    "corpus_vector_dir": str(CORPUS_VECTOR_DIR) if BUILD_CORPUS_COMPARISON else None,
    "corpus_graph_dir": str(CORPUS_GRAPH_DIR) if BUILD_CORPUS_COMPARISON else None,
    "finetune_export_dir": str(FINETUNE_EXPORT_DIR) if EXPORT_FINETUNE_FILES else None,
    "note": "For public-facing answers, do not rely on fine-tuning alone. Use the retrieval bundles at inference time for citations, freshness, and debuggability.",
    "production_inference_flow": "per-paper retrieval first, corpus retrieval second, answer synthesis third, with automatic comparison mode",
}
master_manifest_path = MANIFEST_DIR / "master_manifest.json"
master_manifest_path.write_text(json.dumps(master_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("Validation report:", validation_path)
print("Master manifest:", master_manifest_path)
validation_df


Validation report: rag_output_openai/manifests/validation_report.csv
Master manifest: rag_output_openai/manifests/master_manifest.json


,paper_stem,file_name,vector_chunks,vector_units,vector_profile,vector_manifest,vector_chroma_dir,graph_json,graph_graphml,graph_nodes_csv,graph_edges_csv,graph_manifest,chunk_count,graph_nodes,graph_edges
0,mitra,mitra.pdf,True,True,True,True,True,True,True,True,True,True,229,164,501
1,regolith,regolith.pdf,True,True,True,True,True,True,True,True,True,True,199,216,671


## 12. Example queries

In [15]:
# Example 1: the default production path with automatic routing
answer, package = answer_with_production_flow("What equations are used in mitra.pdf?")
print(package["routing_summary"])
print(answer)

Mode: single_paper
Selected papers: mitra
Explicit matches:
- mitra via 'mitra' (score=0.90)
Ranked candidates:
- mitra score=19.40 signals=corpus_graph, corpus_vector, explicit_match
- regolith score=7.21 signals=corpus_graph, corpus_vector
The following equations are explicitly shown/used in **mitra.pdf** (as captured in the available extracted chunks):

- **Likelihood function (Eq. 3)**  
  \[
  L(\theta_1,\ldots,\theta_n)=\prod_{i=1}^{n}\ \prod_{x\in A_i} f_{\theta_i}(x)
  \]
  [mitra.pdf p.10 equation_candidate chunk_5bfcd0f06527f92f]

- **Maximum-likelihood estimate via likelihood maximization (Eq. 4)**  
  \[
  (\theta_1^\ast,\ldots,\theta_n^\ast)=\arg\max_{(\theta_1,\ldots,\theta_n)\in\Theta^n} L(\theta_1,\ldots,\theta_n)
  \]
  [mitra.pdf p.10 equation_candidate chunk_5bfcd0f06527f92f]

- **Linear regression mapping process parameter to distribution parameters (Eq. 5)**  
  \[
  g(z)=a\cdot z+b
  \]
  [mitra.pdf p.10 equation_candidate chunk_645c5b1e341e2ad3]

- **Least-square